# 01 · 데이터 확인 및 EDA

시안 1~2단계의 기존 분석을 한 노트북으로 통합했습니다.
CN7 분석 → RG3 구조 분석 → 공정 기반 분석 → 추가 군집/근접 패턴 분석 → 비라벨 컬럼 그래프 순서입니다.
분석용 전체 데이터 통계·3-fold 누수 비교는 모델 학습용 분할이 아닙니다.
학습 데이터 저장은 `02_preprocess.ipynb`, 고정 분할은 `03_split.ipynb`에서만 수행합니다.
기존 해석 문장은 당시 결과 기록이며 현재 실행 표와 함께 확인하세요. 행 번호를 생산 시간으로 해석하지 않습니다.


In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/origin').is_dir())
os.environ['MPLCONFIGDIR'] = str(PROJECT_ROOT/'tmp/matplotlib_cache')
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


# CN7 라벨 분포·충돌·변수 조합

원본 분석: `EDA/cn7eda.ipynb`. 모듈화 전 원본은 복구 ZIP에 보관되어 있습니다.


# 0. Libraries

In [ ]:
# ?? ?????
import hashlib
import json
import math
import os
from itertools import combinations
from pathlib import Path

# ??? ?? ? ?? ???
import numpy as np
import openpyxl
import pandas as pd

# ??? ? ??? ??
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

# ?? ? ?? ??? ??
from scipy.stats import ks_2samp
from sklearn.model_selection import StratifiedKFold


# 1. Load Dataset

## 1.1. 경로 설정

In [ ]:
project_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'data/origin/moldset_labeled_cn7.csv').is_file()
)


## 1.2. 데이터 불러오기 및 구조 확인

In [ ]:
cn7_l = pd.read_csv(PROJECT_ROOT / 'data/origin/moldset_labeled_cn7.csv', encoding='cp949')

In [ ]:
cn7_l.shape

In [ ]:
cn7_l.head(10)

## 1.3. 컬럼 확인

In [ ]:
cn7_l.columns

#### 가이드북 p.20 참고


* 'Unnamed: 0': 인덱스 번호

* 'PassOrFail': 사출되는 각 물품마다 검수자가 양품 선별을 하여 붙이는 레이블 값

* 'Injection_Time'(Sec (초)): 고압+사출시간(고압(사출압): 재료를 금형에 유입시킬때의 압력, 사출시간: 재료를 금형에 유입시키는데 소요되는 시간)

* 'Filling_Time'(Sec (초)): 충진시간으로 사출기에서 금형으로 내용물이 주입되는 시간

* 'Plasticizing_Time'(Sec (초)): 계량시간으로 재료를 스크류에 1번 생산할 만큼 용융되어 저장되는 시간

* 'Cycle_Time'(Sec (초)): 1번의 제품생산에 소요되는 생산시간

* 'Clamp_Close_Time'(Sec (초)): 제품이 생산되고 난후 열려있는 금형을 사출기가 닫아주고 빈틈이 없이 고정축과 이동축을 꽉 잡아주는데 걸리는 시간

* 'Cushion_Position'(mm): 보압(사출압의 다음으로 가해지는 압력(금형내부압력을 조절하여 과충전을방지))을 하기위한 스크류의 위치

* 'Plasticizing_Position'(mm): 계량완료위치(계량을 마친 스크류의 위치)

* 'Clamp_Open_Position'(mm): 제품이 생산되어 추출하기위해 금형이 열리고 난 위치

* 'Max_Injection_Speed'(mm/s): 배럴에 계량되어 있는 용융수지가 금형으로 흘러들어가는데 측정되는 최대 속도

* 'Max_Screw_RPM'(mm/s): 사출을 위한 스크류의 최대속도

* 'Average_Screw_RPM'(mm/s): 사출을 위한 스크류의 평균속도

* 'Max_Injection_Pressure'(MPa): 배럴에 계량되어 있는 용융수지가 금형으로 흘러들어가는데 가해지는 최대 압력

* 'Max_Switch_Over_Pressure'(MPa): 사출에서 보압(충진된 수지가 밀리지않게 압력을준다)으로 변환되는 압력

* 'Max_Back_Pressure'(MPa): 수지가 계량이 되는중에 스크류가 밀려나는 현상을 저지하기위한 최대압력

* 'Average_Back_Pressure'(MPa): 수지가 계량이 되는중에 스크류가 밀려나는 현상을 저지하기위한 평균압력

* 'Barrel_Temperature_1~6'(섭씨 (°C)): 계량 및 사출시 수지가 일정하게 용융(녹임)을 유지하기위해 온도가 일정해야한다

* 'Hopper_Temperature'(섭씨 (°C)): 재료주입구의 온도(충분히 건조시켜주며 재료가 용융되는시간을 절약시켜 주기위해 온도가 높아야한다)

* 'Mold_Temperature_3, 4'(섭씨 (°C)): 재료주입구의 온도(충분히 건조시켜주며 재료가 용융되는시간을 절약시켜 주기위해 온도가 높아야한다)

# 2. EDA

## 2.1. 통계량 확인

In [ ]:
cn7_l.describe()

In [ ]:
cn7_l = cn7_l.drop(columns = ['Unnamed: 0'])

In [ ]:
cn7_l.describe()

**통계량 해석:** 이 표는 양품과 불량을 합친 전체 분포이므로 평균·범위만으로 불량 조건을 정할 수 없다. 특히 반복된 공정조건이 통계량에 중복 반영될 수 있다. 다음 절에서는 동일 공정값의 반복과 라벨 충돌을 먼저 확인한다.

In [ ]:
# 종속변수 & 독립변수 분리

cn7_l_y = cn7_l['PassOrFail']
cn7_l_x = cn7_l.drop(columns = ['PassOrFail'])

## 2.2. 데이터 중복 확인

* 중복 삭제 근거: 보수적으로 접근 -> 동일 레코드에 대해 한 경우에만 불량이 발생한 경우 잠재적으로 불량이 발생할 가능성이 있다고 간주하여 정상 데이터를 삭제

In [ ]:
temp = cn7_l_x.copy()
temp["PassOrFail"] = cn7_l_y


# 불량(1)이 먼저 오도록 정렬
temp = temp.sort_values(
    by="PassOrFail",
    ascending=False
)

# 독립변수가 동일한 경우
# 불량을 우선적으로 하나만 남김
temp_dedup = temp.drop_duplicates(
    subset=cn7_l_x.columns.tolist(),
    keep="first"
)

# 원래 index 기준으로 정렬
temp_dedup = temp_dedup.sort_index()

cn7_l_x = temp_dedup.drop(columns="PassOrFail")
cn7_l_y = temp_dedup["PassOrFail"]

**중복 처리의 의미:** 이 처리는 같은 공정조건에서 불량이 한 번이라도 관측되면 그 조건을 불량으로 대표시키는 선택이다. 뒤의 3.1에서 확인하는 원본 1,211행(양품 1,194·불량 17)은 처리 후 606개 조건(양품 592·불량 14)이 된다. 이때 양품과 불량이 공존하는 11개 조건의 양품 기록도 제거된다. 따라서 이후 분포는 원본 생산 빈도가 아닌 이 대표 조건 집합의 분포이며, 원본 불량률이나 완전 분리 가능성으로 해석하지 않는다.

## 2.3. 컬럼별 데이터 확인

In [ ]:



plot_df = cn7_l_x.copy()
plot_df["PassOrFail"] = cn7_l_y.values

# 숫자형 변수만 선택
numeric_cols = cn7_l_x.select_dtypes(include="number").columns

# 그래프 설정
n_cols = 4
n_rows = math.ceil(len(numeric_cols) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(36, 4.5 * n_rows)
)

axes = axes.flatten()

for i, col in enumerate(numeric_cols):

    ax = axes[i]

    # -------------------------
    # 1. Histogram
    # -------------------------
    sns.histplot(
        data=plot_df,
        x=col,
        hue="PassOrFail",
        stat="density",
        common_norm=False,
        element="step",
        alpha=0.3,
        kde=False,          # KDE는 따로 처리
        ax=ax
    )

    # -------------------------
    # 2. 클래스별 KDE
    # -------------------------
    for label in plot_df["PassOrFail"].dropna().unique():

        values = (
            plot_df.loc[
                plot_df["PassOrFail"] == label,
                col
            ]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        # KDE가 가능한 경우만 수행
        if (
            len(values) >= 3
            and values.nunique() >= 2
            and values.std() > 1e-10
        ):
            try:
                sns.kdeplot(
                    x=values,
                    ax=ax,
                    label=f"{label} KDE"
                )
            except Exception:
                pass

    ax.set_title(f"{col} Distribution")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.grid(alpha=0.2)

# 남는 subplot 제거
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# KS 통계량 확인
# KS 통계량: 두 누적 분포 함수의 최대 거리 차이를 측정하는 비모수적 통계량

ks_results = []

numeric_cols = cn7_l_x.select_dtypes(include="number").columns

for col in numeric_cols:
    normal = cn7_l_x.loc[cn7_l_y == 0, col].dropna()
    abnormal = cn7_l_x.loc[cn7_l_y == 1, col].dropna()

    # 두 그룹 중 하나라도 데이터가 없으면 제외
    if len(normal) == 0 or len(abnormal) == 0:
        continue

    ks_stat, p_value = ks_2samp(
        normal,
        abnormal
    )

    ks_results.append({
        "Feature": col,
        "Normal_Median": normal.median(),
        "Abnormal_Median": abnormal.median(),
        "Normal_IQR": normal.quantile(0.75) - normal.quantile(0.25),
        "Abnormal_IQR": abnormal.quantile(0.75) - abnormal.quantile(0.25),
        "KS_Statistic": ks_stat,
        "P_Value": p_value,
        "Normal_Count": len(normal),
        "Abnormal_Count": len(abnormal)
    })

ks_df = (
    pd.DataFrame(ks_results)
    .sort_values("KS_Statistic", ascending=False)
    .reset_index(drop=True)
)

ks_df["Rank"] = ks_df.index + 1

ks_df = ks_df[
    [
        "Rank",
        "Feature",
        "KS_Statistic",
        "P_Value",
        "Normal_Median",
        "Abnormal_Median",
        "Normal_IQR",
        "Abnormal_IQR",
        "Normal_Count",
        "Abnormal_Count"
    ]
]

ks_df

In [ ]:
plt.figure(figsize=(10, 10))

sns.barplot(
    data=ks_df,
    x="KS_Statistic",
    y="Feature"
)

plt.title("KS Statistic by Feature")
plt.xlabel("KS Statistic")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

**분포·KS 결과 해석:** 불량 우선 중복 처리 데이터에서 KS는 계량 위치 0.758, 금형 온도 4 0.721, 금형 온도 3 0.711, 최대 배압 0.706, 계량 시간 0.701 순이다. 이는 이 데이터 표현에서 양품·불량의 분포 차이가 큰 변수라는 뜻이며 원인 또는 예측 중요도 순위는 아니다. 원본을 다시 확인하면 불량은 계량 위치·두 온도가 양품 중앙값보다 높은 쪽, 배압·계량 시간이 낮은 쪽에 관측된다(3.2). 다만 동일 입력의 양품이 제거된 효과를 구분해야 하므로, 3.1에서 라벨 공존을 복원하고 3.3에서 원본·중복 처리별 KS를 비교한다.

# 3. 양품·불량 공정조건 심화 EDA
분석 순서: **① 불량 유형 구분 → 기존 KS 상위 5개 변수의 10개 쌍 조합 → ② 개별 변수 분포와 사례 의존성 → ④ 공정 단계별 비교**.

기존 `cn7_l_x`, `cn7_l_y`, `ks_df`는 변경하지 않는다. 기존 KS 순위는 **동일 입력에서 불량 우선으로 한 행만 남긴 데이터**의 결과이므로, 원본 기록과 구분해서 해석한다. 아래에서는 CSV를 별도로 읽어 원본의 양품·불량 공존을 확인한다. 같은 24개 공정값을 하나의 패턴으로 정의하며 저장 인덱스는 시간으로 해석하지 않는다.

모든 공정값은 파일별 표준화 값이다. 분포·상관·관측 불량 비율은 탐색 결과이며 물리적 규격, 원인 또는 검증된 예측 성능을 뜻하지 않는다. 중복과 이산값이 많고 불량이 적으므로 KS는 분포 차이의 크기로 사용하고 p값으로 유의성을 판정하지 않는다.

In [ ]:
eda3_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'data/origin/moldset_labeled_cn7.csv').is_file())
eda3_raw = pd.read_csv(eda3_root / 'data/origin/moldset_labeled_cn7.csv')
eda3_features = [c for c in eda3_raw if c not in ['Unnamed: 0', 'PassOrFail']]
assert len(eda3_features) == 24
assert np.isfinite(eda3_raw[eda3_features].to_numpy()).all()
assert set(eda3_raw.PassOrFail.unique()) == {0, 1}
eda3_raw = eda3_raw.rename(columns={'Unnamed: 0': 'source_id'})
eda3_raw['pattern_id'] = pd.MultiIndex.from_frame(eda3_raw[eda3_features]).factorize(sort=False)[0]
eda3_patterns = eda3_raw.groupby('pattern_id').PassOrFail.agg(rows='size', defects='sum')
eda3_patterns['normal'] = eda3_patterns.rows - eda3_patterns.defects
eda3_patterns['type'] = np.select(
    [eda3_patterns.defects.eq(0), eda3_patterns.normal.eq(0)],
    ['normal_only', 'defect_only'], default='conflicting')
eda3_raw['pattern_type'] = eda3_raw.pattern_id.map(eda3_patterns['type'])
eda3_normal = eda3_raw[eda3_raw.PassOrFail.eq(0)].copy()
eda3_bad = eda3_raw[eda3_raw.PassOrFail.eq(1)].copy()
eda3_normal_unique = eda3_normal.drop_duplicates(eda3_features)
eda3_bad_unique = eda3_bad.drop_duplicates(eda3_features)
eda3_current = cn7_l_x.copy()
eda3_current['PassOrFail'] = cn7_l_y
eda3_top5 = ks_df.sort_values('Rank').head(5).Feature.tolist()
assert len(set(eda3_top5)) == 5

def eda3_say(text):
    display(Markdown(text))

eda3_colors = {'normal':'#3274A1', 'conflicting':'#E69F00', 'defect_only':'#CC3311'}
def eda3_scatter(ax, x, y):
    ax.scatter(eda3_normal_unique[x], eda3_normal_unique[y], s=20, alpha=.3,
               color=eda3_colors['normal'], label=f'Normal patterns (n={len(eda3_normal_unique)})')
    for kind, marker in [('conflicting','o'), ('defect_only','x')]:
        g = eda3_bad_unique[eda3_bad_unique.pattern_type.eq(kind)]
        kwargs = {'facecolors':'none','edgecolors':eda3_colors[kind]} if marker == 'o' else {'color':eda3_colors[kind]}
        ax.scatter(g[x], g[y], s=65, marker=marker, linewidths=1.5,
                   label=f'{kind} defects (n={len(g)})', **kwargs)
    ax.set(xlabel=x, ylabel=y)
    ax.grid(alpha=.15)
    ax.legend(fontsize=7)

display(ks_df.sort_values('Rank').head(5)[['Rank','Feature','KS_Statistic','Normal_Count','Abnormal_Count']])
eda3_say('**이번 조합 분석에 사용할 기존 KS 상위 5개:** ' + ', '.join(eda3_top5))

## 3.1 불량 유형 구분 ①
**질문:** 불량의 24개 공정값이 양품에서도 관측되는가?

`conflicting`은 같은 입력에 양품·불량이 모두 있는 패턴, `defect_only`는 현재 데이터에서 동일 입력의 양품이 없는 패턴이다. 후자가 제조상 불가능하거나 반드시 불량인 조건이라는 의미는 아니다. 행 수와 고유 패턴 수를 함께 센다.

In [ ]:
eda3_type_summary = eda3_patterns.groupby('type').agg(
    patterns=('rows','size'), rows=('rows','sum'), normal_rows=('normal','sum'), defect_rows=('defects','sum'))
display(eda3_type_summary)
eda3_bad_cases = eda3_bad[['source_id','pattern_id','pattern_type']].copy()
eda3_bad_cases['same_input_normal_rows'] = eda3_bad.pattern_id.map(eda3_patterns.normal)
eda3_bad_cases['same_input_defect_rows'] = eda3_bad.pattern_id.map(eda3_patterns.defects)
display(eda3_bad_cases)
display(pd.DataFrame([
    ['Original rows', len(eda3_raw), len(eda3_normal), len(eda3_bad)],
    ['Unique X + label', len(eda3_normal_unique)+len(eda3_bad_unique), len(eda3_normal_unique), len(eda3_bad_unique)],
    ['Existing defect-priority X', len(eda3_current), int(eda3_current.PassOrFail.eq(0).sum()), int(eda3_current.PassOrFail.eq(1).sum())]
], columns=['view','total','normal','defect']))
fig, ax = plt.subplots(figsize=(7,4))
eda3_bad.groupby('pattern_type').agg(rows=('pattern_id','size'), patterns=('pattern_id','nunique')).plot.bar(ax=ax, rot=0)
ax.set(title='CN7 defect rows and unique input patterns', ylabel='Count', xlabel='Defect type')
for container in ax.containers: ax.bar_label(container)
plt.tight_layout(); plt.show()
eda3_shared_n = int(eda3_bad.pattern_type.eq('conflicting').sum())
eda3_only_ids = eda3_bad.loc[eda3_bad.pattern_type.eq('defect_only'),'source_id'].tolist()
eda3_say(f'**결과:** 불량 {len(eda3_bad)}행 중 {eda3_shared_n}행은 동일 입력의 양품이 존재한다. '
         f'나머지 {len(eda3_bad)-eda3_shared_n}행의 ID는 {eda3_only_ids}이며 '
         f'{eda3_bad.loc[eda3_bad.pattern_type.eq("defect_only"),"pattern_id"].nunique()}개 고유 패턴이다. '
         '기존 불량 우선 처리는 충돌 조건의 양품을 제거하므로 원본의 관측 불량률이나 분리 가능성과 동일하게 해석하지 않는다.')

**해석과 다음 단계:** 원본 불량 17행 중 11행은 양품과 24개 값이 모두 같고, 나머지 6행은 비충돌 3개 패턴이 각각 2회 기록된 것이다. 따라서 불량 17행을 서로 다른 17개 조건으로 볼 수 없다. 같은 24개 값만 입력받는 결정적 판정 규칙은 충돌 11개 패턴 안의 양품과 불량에 서로 다른 답을 줄 수 없다. 3.2의 새로운 정보는 이 겹침을 재확인하는 데 있지 않고, 나머지 3개 패턴의 특이성이 단일 값인지 조합인지 구분하고 불량 집중 구간을 정량화하는 데 있다.

## 3.2 상위 변수 조합의 추가 정보와 한계
**앞 절에서 이어지는 질문:** 3.1에서 24개 공정값이 모두 같은 양품·불량을 확인했다. 이 충돌은 두 변수를 골라도 해소할 수 없다. 따라서 여기서는 비충돌 불량 3개 패턴에 대해 **단일 변수로 보이지 않던 특이 조합이 추가로 드러나는가**, 그리고 **불량이 집중되는 범위를 더 좁혀 설명할 수 있는가**를 확인한다.

KS는 변수 하나의 분포 차이를 측정하므로 두 변수의 동시 조건을 직접 설명하지 않는다. 이것이 조합을 확인하는 근거다. 다만 KS 상위라는 사실이 유용한 상호작용을 보장하지는 않는다. 상위 5개는 앞선 결과를 점검하기 위한 탐색 범위이며, 24개 변수 전체에서 최적 조합을 찾은 결과가 아니다. 10개 쌍을 모두 확인해 유리한 그림만 고르는 것을 피한다.

**판단 기준:** 각 변수 값은 양품에 각각 존재하지만 두 값의 조합은 양품에 없는 경우만, 해당 두 단일 변수 대비 추가 정보로 센다. 단일 값부터 양품에 없는 사례는 조합의 성과로 세지 않는다. 아래 산점도·빈도표 다음에 이 비교를 제시한다. 이는 정확한 값의 일치 여부에 한정한 점검이며 분류 성능이나 통계적 상호작용 검정은 아니다.

산점도는 라벨별 고유 전체입력 패턴을 사용한다. 파란 점은 양품, 주황색 빈 원은 전체입력이 같은 양품도 있는 불량, 빨간 ×는 전체입력이 같은 양품이 없는 불량이다. 서로 다른 전체입력 패턴도 두 변수 좌표에서는 겹칠 수 있다. Spearman 상관은 양품 고유 패턴에서 계산한다.


In [ ]:
eda3_pair_rows = []
eda3_pair_details = []
fig, axes = plt.subplots(5, 2, figsize=(16,23))
for ax, (x,y) in zip(axes.flat, combinations(eda3_top5, 2)):
    eda3_scatter(ax, x, y)
    freq = eda3_normal.groupby([x,y]).size()
    counts = np.array([freq.get(tuple(v),0) for v in eda3_bad_unique[[x,y]].to_numpy()])
    eda3_pair_rows.append([x,y,eda3_normal_unique[[x,y]].corr(method='spearman').iloc[0,1],
                           len(counts),int((counts>0).sum()),int((counts==0).sum())])
    for (_,r), count in zip(eda3_bad_unique.iterrows(), counts):
        eda3_pair_details.append([x,y,int(r.source_id),int(r.pattern_id),r.pattern_type,int(count),
                                  int(eda3_patterns.loc[r.pattern_id,'defects'])])
    ax.set_title(f'Exact pair shared with normal: {(counts>0).sum()}/{len(counts)} defect patterns', fontsize=10)
plt.tight_layout(); plt.show()
eda3_pair_summary = pd.DataFrame(eda3_pair_rows, columns=[
    'feature_1','feature_2','normal_pattern_spearman','defect_patterns','shared_with_normal','no_exact_normal_pair'])
eda3_pair_detail = pd.DataFrame(eda3_pair_details, columns=[
    'feature_1','feature_2','representative_id','pattern_id','pattern_type','normal_rows_matching_pair','defect_rows_in_pattern'])
display(eda3_pair_summary.round(3))
display(eda3_pair_detail.pivot(index=['representative_id','pattern_type'], columns=['feature_1','feature_2'], values='normal_rows_matching_pair'))
eda3_strong = eda3_pair_summary.loc[eda3_pair_summary.normal_pattern_spearman.abs().ge(.8), ['feature_1','feature_2','normal_pattern_spearman']]
eda3_say('**겹침과 중복 정보:** 아래 표는 양품 고유 패턴에서 |Spearman|≥0.8인 쌍이다. '
         '0.8은 관계를 요약하기 위한 표시 기준이다. 높은 상관은 두 변수가 비슷한 순서 정보를 가질 수 있음을 뜻하며, 독립적인 불량 요인 두 개를 발견했다는 의미는 아니다.')
display(eda3_strong.round(3))
eda3_say(f'10개 쌍에서 동일 조합의 양품이 존재하는 불량 패턴 수는 '
         f'{eda3_pair_summary.shared_with_normal.min()}–{eda3_pair_summary.shared_with_normal.max()}/{len(eda3_bad_unique)}개다. '
         '전체입력이 같은 충돌 패턴은 어느 변수 쌍에서도 분리할 수 없다. 쌍별 차이는 비충돌 불량을 중심으로 해석한다.')

In [ ]:
# 조합의 추가 정보: 두 단일 값은 양품에 있지만 두 값의 조합은 없는가?
eda3_increment_rows = []
for x, y in combinations(eda3_top5, 2):
    x_absent = ~eda3_bad_unique[x].isin(eda3_normal[x])
    y_absent = ~eda3_bad_unique[y].isin(eda3_normal[y])
    freq = eda3_normal.groupby([x, y]).size()
    pair_absent = pd.Series(
        [freq.get(tuple(v), 0) == 0 for v in eda3_bad_unique[[x, y]].to_numpy()],
        index=eda3_bad_unique.index)
    additional = pair_absent & ~x_absent & ~y_absent
    eda3_increment_rows.append([
        x, y, int(x_absent.sum()), int(y_absent.sum()), int(pair_absent.sum()),
        int(additional.sum()), eda3_bad_unique.loc[additional, 'source_id'].tolist()])
eda3_pair_increment = pd.DataFrame(eda3_increment_rows, columns=[
    'feature_1', 'feature_2', 'x_alone_absent', 'y_alone_absent',
    'pair_absent', 'additional_over_two_singles', 'additional_representative_IDs'])
display(eda3_pair_increment)


**조합 분석의 실제 결론:** 불량 고유 패턴 14개 중 12~14개가 각 쌍에서 양품과 좌표를 공유한다. 이 수치만으로는 단일 변수보다 좋아졌다고 할 수 없다. 위 비교표에서 최대 배압이 포함된 네 쌍의 양품 미관측 패턴은 모두 ID 117·119이며, 이 둘은 **배압 값 자체가 양품에 없다**. 따라서 조합이 새로 찾아낸 사례가 아니다.

유일한 추가 조합 사례는 **계량 위치–계량 시간의 ID 119**이다. 두 값은 각각 양품에 있지만 동일한 조합은 없다. 다만 ID 119는 배압 하나로도 이미 양품 미관측 사례이므로, 상위 5개 단일 변수 전체에 비해 새로 포착한 불량 패턴은 없다. ID 115는 10개 쌍 모두 동일 좌표의 양품이 있어 이 조합들로 구별되지 않는다. 금형 온도 3·4의 상관 0.988과 두 온도 쌍의 추가 사례 0개도 이 데이터에서 두 온도가 상당히 중복된 정보를 준다는 해석을 뒷받침한다.

따라서 이 단계는 강한 상호작용을 발견한 결과가 아니라 **정확한 값의 겹침 기준에서 상위 변수 조합의 추가 정보가 매우 제한적임을 확인한 결과**다. 다음 구간 분석에서는 완전 분리와 별개로 불량이 집중되는 범위를 얼마나 좁힐 수 있는지 본다.

### 두 변수 구간별 관측 건수: 불량이 집중되는 범위는 좁아지는가?
정확한 값의 겹침만으로는 주변 구간의 관측 빈도를 알 수 없다. 이번에는 양품 원본 중앙값으로 각 쌍을 네 구간으로 나누고, 불량이 포함된 구간의 크기와 양품 수를 확인한다. 단일 변수 구간과 비교해야 조합으로 범위가 얼마나 좁아졌는지 알 수 있으므로 뒤에 비교표를 추가한다.

경계와 같은 값은 `≤ median`에 포함한다. 이산값·동률 때문에 구간 크기는 같지 않을 수 있다. 각 칸은 **불량 행/전체 행, 전체입력 패턴 수**를 표시한다. 중앙값은 탐색용 구분이며 최적 불량 임계값이 아니다. 원본 중복을 유지한 관측 비율로, 독립 표본 기반 확률 추정은 아니다.


In [ ]:
eda3_quadrants = []
fig, axes = plt.subplots(5,2,figsize=(14,21))
for ax,(x,y) in zip(axes.flat, combinations(eda3_top5,2)):
    mx,my = eda3_normal[x].median(),eda3_normal[y].median()
    rates = np.full((2,2),np.nan); labels = np.empty((2,2),dtype=object)
    for iy in range(2):
        for ix in range(2):
            mask = (eda3_raw[x].gt(mx) if ix else eda3_raw[x].le(mx)) & (eda3_raw[y].gt(my) if iy else eda3_raw[y].le(my))
            g=eda3_raw.loc[mask]; total=len(g); bad=int(g.PassOrFail.sum()); patterns=g.pattern_id.nunique()
            rate=bad/total if total else np.nan
            rates[iy,ix]=100*rate
            labels[iy,ix]=f'{bad}/{total} rows\n{patterns} patterns\n{100*rate:.1f}%' if total else 'No observations'
            eda3_quadrants.append([x,y,mx,my,ix,iy,total,bad,patterns,rate])
    sns.heatmap(rates,annot=labels,fmt='',cmap='YlOrRd',vmin=0,vmax=100,cbar=False,ax=ax,
                xticklabels=['<= median','> median'],yticklabels=['<= median','> median'])
    for iy,ix in zip(*np.where(np.isnan(rates))):ax.text(ix+.5,iy+.5,'No observations',ha='center',va='center',fontsize=9)
    ax.set(xlabel=x,ylabel=y,title=f'Normal medians: x={mx:.3f}, y={my:.3f}')
plt.tight_layout();plt.show()
eda3_quadrant_table=pd.DataFrame(eda3_quadrants,columns=['feature_1','feature_2','x_cut','y_cut','x_high','y_high','rows','defects','patterns','observed_defect_fraction'])
display(eda3_quadrant_table)
eda3_say('**해석:** 같은 색의 차이도 분모와 고유 패턴 수를 먼저 확인한다. '
         '불량이 포함된 구간에 양품도 많다면 그 조합은 점검할 조건을 설명하지만 단독 판정 규칙으로는 부족하다. '
         'KS 상위 변수 선정과 이 구간 비교는 같은 데이터를 사용했으므로 독립 검증이 아니다.')

In [ ]:
# 단일 변수 구간과 두 변수 구간을 같은 원본 행 기준으로 비교한다.
eda3_single_regions = []
for c in eda3_top5:
    cut = eda3_normal[c].median()
    for high in [False, True]:
        g = eda3_raw[eda3_raw[c].gt(cut) if high else eda3_raw[c].le(cut)]
        if g.PassOrFail.sum() > 0:
            eda3_single_regions.append([c, '> median' if high else '<= median',
                len(g), int(g.PassOrFail.sum()), g.pattern_id.nunique(),
                g.PassOrFail.mean()])
eda3_single_region_table = pd.DataFrame(eda3_single_regions, columns=[
    'feature', 'region', 'rows', 'defects', 'patterns', 'observed_defect_fraction'])
display(eda3_single_region_table)
display(eda3_quadrant_table.loc[eda3_quadrant_table.defects.gt(0), [
    'feature_1', 'feature_2', 'x_high', 'y_high', 'rows', 'defects', 'patterns',
    'observed_defect_fraction']])


**구간 분석의 실제 결론:** 전체 원본의 불량은 17/1,211행(1.40%)이다. 불량 17행은 모두 양품 중앙값 기준으로 계량 위치·온도 3·4가 높은 쪽, 배압·계량 시간이 낮은 쪽에 있다. 이는 불량이 넓게 흩어지기보다 특정 방향의 공정조건에 집중되었다는 관측이다.

단일 변수로 보면 온도 4가 높은 구간은 17/589행(2.89%), 계량 시간이 낮은 구간은 17/629행(2.70%)이다. 두 조건을 결합하면 **17/375행(4.53%), 188개 전체입력 패턴**으로 좁아진다. 온도 3가 높고 계량 시간이 낮은 조합도 같은 집계값이다. 이 10개 쌍의 중앙값 구간 중 불량을 모두 포함하는 가장 작은 구간이지만, 그 안에는 **양품 358행**이 있다. 이 구간을 모두 불량으로 판정하면 관측상 17건을 포함하는 대신 양품 358건도 함께 걸린다.

반면 두 온도가 모두 높은 구간은 17/581행(2.93%)으로, 온도 4 하나의 589행에 비해 8행만 줄어든다. 높은 상관과 일치하게, 두 온도를 결합하는 추가 이득은 작다. **조합의 유용성은 점검 범위를 좁히는 데 있으며 단독 불량 판정에는 부족하다.** 같은 자료로 변수를 고르고 구간을 평가했으므로 이 비율을 새 생산 데이터의 성능으로 보지 않는다. 다음 3.3에서는 이 집중 경향이 비충돌 3개 패턴을 제외해도 유지되는지 확인한다.

## 3.3 상위 변수의 개별 분포와 사례 의존성 ②
**질문:** 기존 KS 상위 변수의 차이는 원본에서도 나타나며, 특이 불량 3개 패턴을 제외해도 남는가?

같은 5개 변수를 유지하고 다음 KS를 비교한다: 원본 행, 기존 불량 우선 중복 처리, 라벨별 고유 패턴, 원본 양품 대비 충돌 불량만, 원본 양품 대비 비충돌 불량만. 마지막 두 비교는 불량 유형의 특성을 보기 위한 것이며 성능 평가가 아니다. 모든 유형 비교에서 양품 참조집합을 동일하게 유지한다.

In [ ]:
eda3_ks_rows=[]
eda3_views={
    'Original rows':(eda3_normal,eda3_bad),
    'Existing defect-priority':(eda3_current[eda3_current.PassOrFail.eq(0)],eda3_current[eda3_current.PassOrFail.eq(1)]),
    'Unique within label':(eda3_normal_unique,eda3_bad_unique),
    'Conflicting defects only':(eda3_normal,eda3_bad[eda3_bad.pattern_type.eq('conflicting')]),
    'Defect-only patterns only':(eda3_normal,eda3_bad[eda3_bad.pattern_type.eq('defect_only')])}
for view,(normal,bad) in eda3_views.items():
    for c in eda3_top5:
        eda3_ks_rows.append([view,c,len(normal),len(bad),ks_2samp(normal[c],bad[c]).statistic,
                             normal[c].median(),bad[c].median()])
eda3_ks_sensitivity=pd.DataFrame(eda3_ks_rows,columns=['view','feature','normal_n','defect_n','KS','normal_median','defect_median'])
display(eda3_ks_sensitivity)
eda3_ks_wide=eda3_ks_sensitivity.pivot(index='feature',columns='view',values='KS').reindex(eda3_top5)
display(eda3_ks_wide.round(3))
assert np.allclose(eda3_ks_wide['Existing defect-priority'],ks_df.set_index('Feature').loc[eda3_top5,'KS_Statistic'])
fig,axes=plt.subplots(3,2,figsize=(15,13))
for ax,c in zip(axes.flat,eda3_top5):
    for label,g,color in [('Normal rows',eda3_normal,'#3274A1'),('All defect rows',eda3_bad,'#555555')]:
        v=np.sort(g[c].to_numpy());ax.step(v,np.arange(1,len(v)+1)/len(v),where='post',label=f'{label} n={len(v)}',color=color)
    for kind,level,marker in [('conflicting',-.07,'o'),('defect_only',-.15,'x')]:
        g=eda3_bad[eda3_bad.pattern_type.eq(kind)]
        ax.scatter(g[c],np.full(len(g),level),color=eda3_colors[kind],marker=marker,label=f'{kind} rows n={len(g)}')
    ax.set(title=c,xlabel='File-standardized value (not physical units)',ylabel='ECDF / defect rug',ylim=(-.22,1.05))
    ax.legend(fontsize=8);ax.grid(alpha=.15)
axes.flat[-1].axis('off');plt.tight_layout();plt.show()

# 불량에서 관측된 실제 값별로 양품 빈도와 고유 전체입력 패턴 수를 표시한다.
eda3_value_rows=[]
for c in eda3_top5:
    for value,b in eda3_bad.groupby(c):
        a=eda3_normal[eda3_normal[c].eq(value)]
        eda3_value_rows.append([c,value,len(a),len(b),a.pattern_id.nunique(),b.pattern_id.nunique(),b.source_id.tolist(),len(b)/(len(a)+len(b))])
eda3_value_table=pd.DataFrame(eda3_value_rows,columns=['feature','value','normal_rows','defect_rows','normal_patterns','defect_patterns','defect_IDs','observed_defect_fraction_at_value'])
display(eda3_value_table)
eda3_say('**특이 3개 패턴 제외 후의 결과:**')
for c in eda3_top5:
    r=eda3_ks_wide.loc[c]
    eda3_say(f'- {c}: 원본 KS **{r["Original rows"]:.3f}**, 충돌 불량만 **{r["Conflicting defects only"]:.3f}**, '
             f'비충돌 불량만 **{r["Defect-only patterns only"]:.3f}**. '
             '크기 차이는 유형별 분포 차이이며 독립적인 예측력 차이가 아니다.')
eda3_say('양품과 같은 입력을 가진 불량만 남겨도 KS가 0일 필요는 없다. '
         '충돌 불량은 전체 양품 중 특정 조건에 집중된 일부이므로 빈도 분포는 다를 수 있다. '
         '따라서 큰 KS와 동일 입력의 라벨 충돌은 동시에 존재할 수 있다.')

**데이터가 말하는 것:** 원본 KS는 계량 위치 0.752, 온도 4 0.715, 온도 3 0.707, 배압 0.700, 계량 시간 0.695이다. 비충돌 3개 패턴을 제외하고 충돌 불량만 남겨도 각각 0.752, 0.715, 0.705, 0.700, 0.695로 거의 같다. 따라서 상위 변수의 분포 차이가 특이한 3개 패턴만으로 만들어졌다고 보기는 어렵다. 불량이 양품에서도 가능한 특정 조건에 집중되어 있어, 큰 KS와 라벨 충돌이 동시에 나타나는 것이다.

반면 비충돌 불량만 비교한 배압 KS는 0.853으로 더 크다. 3.2의 배압 특이값(ID 117·119)과 연결되는 결과지만, 이 집단은 6행·3개 패턴에 불과하다. 값별 관측 비율이 100%인 배압도 반복된 소수 사례에 기반하므로 확정적인 불량 기준으로 채택하지 않는다. 다음 절에서는 KS 순위 밖의 충전·전환 관련 변수에 이 3개 패턴의 추가 특징이 있는지 확인한다.

## 3.4 공정 단계별 조합 비교 ④
**질문:** KS 상위 변수의 관찰은 충전·사출·보압 관련 공정값과 어떻게 연결되는가?

충전시간–최대사출속도, 사출시간–최대사출압력, 전환압력–쿠션위치에 더해, 상위 5개에 포함된 계량위치–배압·계량시간을 비교한다. `Injection_Time`과 최대사출압력은 사출 공통 요약값이며 가압 단계만의 측정값이 아니다. 전환압력과 쿠션 위치만으로 보압 유지 전체를 관측할 수 없다. `Mold_Temperature_3/4`는 가이드북의 이름과 설명이 불일치하므로 열적 조건 참고로만 남긴다.

In [ ]:
eda3_process_pairs=[
    ('Filling','Filling_Time','Max_Injection_Speed'),
    ('Injection summary','Injection_Time','Max_Injection_Pressure'),
    ('Switch / cushion','Max_Switch_Over_Pressure','Cushion_Position'),
    ('Plasticizing / back pressure','Plasticizing_Position','Max_Back_Pressure'),
    ('Plasticizing time / position','Plasticizing_Time','Plasticizing_Position'),
    ('Thermal context (definition uncertain)','Mold_Temperature_3','Mold_Temperature_4')]
eda3_process_rows=[]
fig,axes=plt.subplots(3,2,figsize=(16,14))
for ax,(stage,x,y) in zip(axes.flat,eda3_process_pairs):
    eda3_scatter(ax,x,y);ax.set_title(stage)
    freq=eda3_normal.groupby([x,y]).size()
    for _,r in eda3_bad_unique.iterrows():
        eda3_process_rows.append([stage,int(r.source_id),int(r.pattern_id),r.pattern_type,
            int(freq.get((r[x],r[y]),0)),
            bool(r[x]<eda3_normal[x].min() or r[x]>eda3_normal[x].max()),
            bool(r[y]<eda3_normal[y].min() or r[y]>eda3_normal[y].max())])
plt.tight_layout();plt.show()
eda3_process_detail=pd.DataFrame(eda3_process_rows,columns=['stage','representative_id','pattern_id','type','normal_rows_matching_pair','x_outside_observed_normal_range','y_outside_observed_normal_range'])
eda3_process_summary=eda3_process_detail.groupby(['stage','type']).agg(
    defect_patterns=('pattern_id','size'),pair_shared_with_normal=('normal_rows_matching_pair',lambda s:int(s.gt(0).sum())),
    x_outside=('x_outside_observed_normal_range','sum'),y_outside=('y_outside_observed_normal_range','sum'))
display(eda3_process_summary)
display(eda3_process_detail[eda3_process_detail['type'].eq('defect_only')])
eda3_say('**공정별 결과 — 비충돌 불량 고유 패턴 기준:**')
for stage,_,_ in eda3_process_pairs:
    g=eda3_process_detail[(eda3_process_detail.stage.eq(stage)) & eda3_process_detail['type'].eq('defect_only')]
    eda3_say(f'- {stage}: {len(g)}개 중 {g.normal_rows_matching_pair.gt(0).sum()}개는 '
             f'같은 2변수 조합의 양품이 있다. x 관측 양품 범위 밖 {g.x_outside_observed_normal_range.sum()}개, '
             f'y 범위 밖 {g.y_outside_observed_normal_range.sum()}개.')
eda3_say('**종합:** KS 상위 변수는 불량이 집중된 조건의 분포를 설명하고, 공정별 조합은 일부 불량의 특이 조건을 설명한다. '
         '충돌 불량은 현재 24개 값으로 양품과 구별할 수 없다는 한계가 남는다. '
         '관측 범위 밖이라는 표현은 제조 규격 초과가 아니며, 변수 간 동반 변화로 공정 원인이나 이상 전파를 확정하지 않는다.')

**공정 비교의 해석:** 비충돌 3개 패턴 모두 최대 사출 속도와 쿠션 위치가 관측 양품 범위 밖에 있다. 반면 상위 변수의 계량 위치–배압에서는 배압 범위 밖이 1개 패턴이고, 계량 시간–위치 및 두 온도에서는 각 축의 양품 범위를 벗어난 패턴이 없다. 즉, 전체 불량의 분포 차이가 큰 변수와 소수 특이 패턴을 설명하는 변수는 같지 않다. KS 상위 5개만 점검했다면 충전·전환 관련 관측을 놓칠 수 있다.

이 분석의 결론은 두 갈래다. 충돌 11개 패턴에는 현재 공정값 외의 정보가 필요하고, 비충돌 3개 패턴에는 사출 속도·쿠션 위치·배압의 원기록과 측정 조건을 우선 확인할 근거가 있다. 관측 양품 범위 이탈은 제조 규격 초과나 인과관계의 증거가 아니다. 추가 정보의 후보는 로트·시간·검수·측정 해상도 등이지만, 어떤 정보가 충돌을 설명하는지는 현재 데이터만으로 정해지지 않는다.

In [ ]:
plt.close("all")


# RG3 구조·군집·변수 조합

원본 분석: `EDA/rg3eda.ipynb`. 모듈화 전 원본은 복구 ZIP에 보관되어 있습니다.


# 0. Libraries

In [ ]:
# 필요한 경우 설치: pandas numpy matplotlib seaborn scipy scikit-learn openpyxl

In [ ]:
# 표준 라이브러리
import hashlib
import json
import math
import os
from itertools import combinations
from pathlib import Path

# 데이터 처리 및 파일 입출력
import numpy as np
import openpyxl
import pandas as pd

# 시각화 및 노트북 출력
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

# 통계 및 검증 데이터 분할
from scipy.stats import ks_2samp
from sklearn.model_selection import StratifiedKFold
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix

# EDA용 비지도 군집화와 군집 비교
from scipy.cluster.hierarchy import linkage, cut_tree, dendrogram
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from threadpoolctl import threadpool_limits


# 1. Load Dataset

## 1.1. 경로 설정

In [ ]:
project_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'data/origin/moldset_labeled_rg3.csv').is_file()
)


## 1.2. 데이터 불러오기 및 구조 확인

In [ ]:
rg3_l = pd.read_csv(PROJECT_ROOT / 'data/origin/moldset_labeled_rg3.csv', encoding='cp949')

In [ ]:
rg3_l.shape

In [ ]:
rg3_l.head(10)

## 1.3. 컬럼 확인

In [ ]:
rg3_l.columns

#### 가이드북 p.20 참고


* 'Unnamed: 0': 인덱스 번호

* 'PassOrFail': 사출되는 각 물품마다 검수자가 양품 선별을 하여 붙이는 레이블 값

* 'Injection_Time'(Sec (초)): 고압+사출시간(고압(사출압): 재료를 금형에 유입시킬때의 압력, 사출시간: 재료를 금형에 유입시키는데 소요되는 시간)

* 'Filling_Time'(Sec (초)): 충진시간으로 사출기에서 금형으로 내용물이 주입되는 시간

* 'Plasticizing_Time'(Sec (초)): 계량시간으로 재료를 스크류에 1번 생산할 만큼 용융되어 저장되는 시간

* 'Cycle_Time'(Sec (초)): 1번의 제품생산에 소요되는 생산시간

* 'Clamp_Close_Time'(Sec (초)): 제품이 생산되고 난후 열려있는 금형을 사출기가 닫아주고 빈틈이 없이 고정축과 이동축을 꽉 잡아주는데 걸리는 시간

* 'Cushion_Position'(mm): 보압(사출압의 다음으로 가해지는 압력(금형내부압력을 조절하여 과충전을방지))을 하기위한 스크류의 위치

* 'Plasticizing_Position'(mm): 계량완료위치(계량을 마친 스크류의 위치)

* 'Clamp_Open_Position'(mm): 제품이 생산되어 추출하기위해 금형이 열리고 난 위치

* 'Max_Injection_Speed'(mm/s): 배럴에 계량되어 있는 용융수지가 금형으로 흘러들어가는데 측정되는 최대 속도

* 'Max_Screw_RPM'(mm/s): 사출을 위한 스크류의 최대속도

* 'Average_Screw_RPM'(mm/s): 사출을 위한 스크류의 평균속도

* 'Max_Injection_Pressure'(MPa): 배럴에 계량되어 있는 용융수지가 금형으로 흘러들어가는데 가해지는 최대 압력

* 'Max_Switch_Over_Pressure'(MPa): 사출에서 보압(충진된 수지가 밀리지않게 압력을준다)으로 변환되는 압력

* 'Max_Back_Pressure'(MPa): 수지가 계량이 되는중에 스크류가 밀려나는 현상을 저지하기위한 최대압력

* 'Average_Back_Pressure'(MPa): 수지가 계량이 되는중에 스크류가 밀려나는 현상을 저지하기위한 평균압력

* 'Barrel_Temperature_1~6'(섭씨 (°C)): 계량 및 사출시 수지가 일정하게 용융(녹임)을 유지하기위해 온도가 일정해야한다

* 'Hopper_Temperature'(섭씨 (°C)): 재료주입구의 온도(충분히 건조시켜주며 재료가 용융되는시간을 절약시켜 주기위해 온도가 높아야한다)

* 'Mold_Temperature_3, 4'(섭씨 (°C)): 재료주입구의 온도(충분히 건조시켜주며 재료가 용융되는시간을 절약시켜 주기위해 온도가 높아야한다)

# 2. EDA

## 2.1. 통계량 확인

In [ ]:
rg3_l.describe()

In [ ]:
rg3_l = rg3_l.drop(columns = ['Unnamed: 0'])

In [ ]:
rg3_l.describe()

**통계량 해석:** 이 표는 양품과 불량을 합친 전체 분포이므로 평균·범위만으로 불량 조건을 정할 수 없다. 특히 반복된 공정조건이 통계량에 중복 반영될 수 있다. 다음 절에서는 동일 공정값의 반복과 라벨 충돌을 먼저 확인한다.

In [ ]:
# 종속변수 & 독립변수 분리

rg3_l_y = rg3_l['PassOrFail']
rg3_l_x = rg3_l.drop(columns = ['PassOrFail'])

## 2.2. 데이터 중복 확인

**중복 처리 비교의 목적:** CN7과 같은 순서로, 불량을 우선하여 동일 공정조건을 하나로 축약한 데이터의 분포를 먼저 확인한다. 이 처리는 충돌 양품을 제거하므로 최종 학습 데이터 정책은 아니다. 3절에서는 원본을 복원해 라벨 공존을 확인하고, 4절에서 최종 처리 기준을 결정한다.

In [ ]:
temp = rg3_l_x.copy()
temp["PassOrFail"] = rg3_l_y


# 불량(1)이 먼저 오도록 정렬
temp = temp.sort_values(
    by="PassOrFail",
    ascending=False
)

# 독립변수가 동일한 경우
# 불량을 우선적으로 하나만 남김
temp_dedup = temp.drop_duplicates(
    subset=rg3_l_x.columns.tolist(),
    keep="first"
)

# 원래 index 기준으로 정렬
temp_dedup = temp_dedup.sort_index()

rg3_l_x = temp_dedup.drop(columns="PassOrFail")
rg3_l_y = temp_dedup["PassOrFail"]

**중복 처리의 의미:** 원본 1,182행(양품 1,157·불량 25)은 불량 우선 처리 후 591개 조건(양품 566·불량 25)이 된다. 각 전체입력 패턴은 정확히 2회씩 기록되어 있으며, 25개 패턴은 양품 1행·불량 1행으로 구성된다. 불량 우선 처리는 이 25개 양품을 제거한다. 따라서 이 처리 후의 분포와 불량 비율은 원본과 다르며, 분리 가능성이나 원본 발생률로 해석하지 않는다.

## 2.3. 컬럼별 데이터 확인

In [ ]:



plot_df = rg3_l_x.copy()
plot_df["PassOrFail"] = rg3_l_y.values

# 숫자형 변수만 선택
numeric_cols = rg3_l_x.select_dtypes(include="number").columns

# 그래프 설정
n_cols = 4
n_rows = math.ceil(len(numeric_cols) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(36, 4.5 * n_rows)
)

axes = axes.flatten()

for i, col in enumerate(numeric_cols):

    ax = axes[i]

    # -------------------------
    # 1. Histogram
    # -------------------------
    sns.histplot(
        data=plot_df,
        x=col,
        hue="PassOrFail",
        stat="density",
        common_norm=False,
        element="step",
        alpha=0.3,
        kde=False,          # KDE는 따로 처리
        ax=ax
    )

    # -------------------------
    # 2. 클래스별 KDE
    # -------------------------
    for label in plot_df["PassOrFail"].dropna().unique():

        values = (
            plot_df.loc[
                plot_df["PassOrFail"] == label,
                col
            ]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        # KDE가 가능한 경우만 수행
        if (
            len(values) >= 3
            and values.nunique() >= 2
            and values.std() > 1e-10
        ):
            try:
                sns.kdeplot(
                    x=values,
                    ax=ax,
                    label=f"{label} KDE"
                )
            except Exception:
                pass

    ax.set_title(f"{col} Distribution")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.grid(alpha=0.2)

# 남는 subplot 제거
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# KS 통계량 확인
# KS 통계량: 두 누적 분포 함수의 최대 거리 차이를 측정하는 비모수적 통계량

ks_results = []

numeric_cols = rg3_l_x.select_dtypes(include="number").columns

for col in numeric_cols:
    normal = rg3_l_x.loc[rg3_l_y == 0, col].dropna()
    abnormal = rg3_l_x.loc[rg3_l_y == 1, col].dropna()

    # 두 그룹 중 하나라도 데이터가 없으면 제외
    if len(normal) == 0 or len(abnormal) == 0:
        continue

    ks_stat, p_value = ks_2samp(
        normal,
        abnormal
    )

    ks_results.append({
        "Feature": col,
        "Normal_Median": normal.median(),
        "Abnormal_Median": abnormal.median(),
        "Normal_IQR": normal.quantile(0.75) - normal.quantile(0.25),
        "Abnormal_IQR": abnormal.quantile(0.75) - abnormal.quantile(0.25),
        "KS_Statistic": ks_stat,
        "P_Value": p_value,
        "Normal_Count": len(normal),
        "Abnormal_Count": len(abnormal)
    })

ks_df = (
    pd.DataFrame(ks_results)
    .sort_values("KS_Statistic", ascending=False)
    .reset_index(drop=True)
)

ks_df["Rank"] = ks_df.index + 1

ks_df = ks_df[
    [
        "Rank",
        "Feature",
        "KS_Statistic",
        "P_Value",
        "Normal_Median",
        "Abnormal_Median",
        "Normal_IQR",
        "Abnormal_IQR",
        "Normal_Count",
        "Abnormal_Count"
    ]
]

ks_df

In [ ]:
plt.figure(figsize=(10, 10))

sns.barplot(
    data=ks_df,
    x="KS_Statistic",
    y="Feature"
)

plt.title("KS Statistic by Feature")
plt.xlabel("KS Statistic")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

**분포·KS 결과 해석:** 기존 처리에서 상위 KS는 배럴 온도 5 0.189, 금형 온도 4 0.169, 호퍼 온도 0.151, 금형 온도 3 0.146, 최대 사출 속도 0.134이다. 개별 분포 차이는 크지 않으며, 상대적 순위를 더 탐색하는 것만으로 전처리·개선 결정을 내리기 어렵다. KS 계산 자체가 부적합하다는 뜻이 아니라, 분석의 중심 질문을 바꾼다는 뜻이다.

다음 3절은 **반복·라벨 구조 → 라벨·비라벨 범위 차이 → 분할 그룹 점검 → 전처리 결정** 순서로 진행한다. 기존 KS 상위 조합과 공정별 분포는 부록 A에 보존한다.

# 3. 데이터 구조와 전처리 기준을 확인하는 EDA

**목표:** 반복·라벨 충돌, 라벨·비라벨의 값 종류와 범위, 분할 간 동일 패턴의 겹침을 확인하고 전처리 기준을 정한다.

분석 순서는 **반복 구조 → 값 종류·관측 범위 → 분할 그룹 점검 → 처리 결정**이다. 관측된 데이터 구조와 제조 불량의 인과적 원인은 구분하며, 비라벨 데이터에 불량 정답을 부여하지 않는다.

추가 분석은 **이산 조합 조건군 → 조건군별 충돌 → 기존 집중 구간의 조건군 내 비교 → 주변 조건 구성 → 비라벨 범위 이탈 분해** 순서로 진행한다.

## 3.1 반복 구조와 관측 라벨의 식별 한계

동일 조건의 반복이 실제 생산 반복인지 복제·가공의 결과인지 현재 CSV만으로 확정하지 않는다. 먼저 관측된 구조를 기록하고, 입력으로 사용하면 안 되는 정보와 보존할 라벨을 구분한다.

In [ ]:
audit_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / 'data/origin/moldset_labeled_rg3.csv').is_file())
audit_labeled = pd.read_csv(audit_root / 'data/origin/moldset_labeled_rg3.csv')
audit_unlabeled = pd.read_csv(audit_root / 'data/origin/moldset_unlabeled_rg3.csv')
audit_features = [c for c in audit_labeled if c not in ['Unnamed: 0', 'PassOrFail']]
audit_X = audit_labeled[audit_features].copy()
audit_y = audit_labeled.PassOrFail.copy()
audit_groups = pd.Series(pd.MultiIndex.from_frame(audit_X).factorize(sort=False)[0], name='pattern_id')
audit_work = audit_labeled.assign(pattern_id=audit_groups, row_position=np.arange(len(audit_X)))
audit_patterns = audit_work.groupby('pattern_id').agg(
    rows=('PassOrFail','size'), defects=('PassOrFail','sum'),
    source_ids=('Unnamed: 0',list), positions=('row_position',list), labels=('PassOrFail',list))
audit_patterns['normal'] = audit_patterns.rows - audit_patterns.defects
audit_patterns['type'] = np.select([audit_patterns.defects.eq(0),audit_patterns.normal.eq(0)],
                                     ['normal_only','defect_only'],default='conflicting')
audit_pattern_summary = audit_patterns.groupby('type').agg(
    patterns=('rows','size'),rows=('rows','sum'),normal=('normal','sum'),defects=('defects','sum'))
display(audit_pattern_summary)
assert audit_patterns.rows.eq(2).all()
audit_pair_structure = pd.DataFrame({
    'row_gap': audit_patterns.positions.map(lambda a:a[1]-a[0]),
    'label_order': audit_patterns.labels.map(lambda a:','.join(map(str,a))),
}).value_counts().rename('patterns').reset_index().sort_values(['label_order','row_gap'])
display(audit_pair_structure)
fig,ax=plt.subplots(figsize=(7,3))
audit_pattern_summary.patterns.plot.bar(ax=ax,rot=0,color=['#E69F00','#3274A1'])
ax.set(title='RG3 unique input patterns',ylabel='Patterns',xlabel='Pattern type')
for container in ax.containers: ax.bar_label(container)
plt.tight_layout();plt.show()

**결과와 판단:** 591개 패턴 모두 2행이며, 566개는 양품·양품, 25개는 양품·불량이다. 574개 패턴은 원본 행 위치가 인접한다. 충돌 25개 중 불량이 먼저 기록된 패턴은 23개, 나중인 패턴은 2개다. 가까운 행과 기록 순서에 구조가 있지만, ID가 생산 시간이라는 근거나 데이터 생성 과정의 설명은 없다. 행 번호·홀짝·중복 내 순서를 특징으로 넣지 않는다.

동일한 24개 공정값에 서로 다른 라벨이 있으므로, 현재 기록만으로 두 행의 차이를 설명할 수 없다. 추가적인 기록·검수 정보의 확인이 필요하다.

**처리 결정:** 원본 라벨을 보존하고 전체입력 패턴을 검증 그룹으로 사용한다. 불량 우선 중복 제거는 '한 번이라도 불량이 나온 조건'으로 목표를 바꾸므로 기본 전처리로 확정하지 않는다. 원인 규명을 위해서는 동일 패턴 두 행의 생성·검수·반복 기록 기준을 우선 확인해야 한다.

## 3.2 학습 데이터와 적용 데이터의 값 종류·범위 차이

평균과 표준편차가 비슷하다는 것만으로 같은 분포나 같은 물리 조건이라고 볼 수 없다. 각 파일의 고유값 수와 표준화 좌표상 관측 범위를 비교한다. 이 단계는 적용 범위 점검이며, 비라벨 행의 이상·불량 판정이 아니다.

In [ ]:
audit_U = audit_unlabeled[audit_features]
audit_outside = audit_U.lt(audit_X.min()) | audit_U.gt(audit_X.max())
audit_coverage = pd.DataFrame({
    'feature':audit_features,
    'labeled_unique':audit_X.nunique().values,'unlabeled_unique':audit_U.nunique().values,
    'labeled_mean':audit_X.mean().values,'unlabeled_mean':audit_U.mean().values,
    'labeled_std_ddof0':audit_X.std(ddof=0).values,'unlabeled_std_ddof0':audit_U.std(ddof=0).values,
    'unlabeled_outside_labeled_range':audit_outside.mean().values,
})
display(audit_coverage.round(4))
audit_active = audit_X.columns[audit_X.nunique().gt(1)]
audit_outside_rows = audit_outside[audit_active].any(axis=1)
display(pd.DataFrame([
    ['labeled',len(audit_X),len(audit_X.drop_duplicates())],
    ['unlabeled',len(audit_U),len(audit_U.drop_duplicates())],
],columns=['dataset','rows','unique_input_patterns']))
display(Markdown('**학습 상수열을 제외한 범위 점검:** ' +
    f'{int(audit_outside_rows.sum()):,}/{len(audit_U):,}행 '
    f'({audit_outside_rows.mean():.2%})이 적어도 한 변수에서 라벨 데이터의 관측 범위 밖이다.'))
fig,axes=plt.subplots(1,2,figsize=(16,7))
audit_coverage.set_index('feature')[['labeled_unique','unlabeled_unique']].plot.barh(ax=axes[0],logx=True)
axes[0].set(title='Observed distinct values (log scale)',xlabel='Distinct values',ylabel='')
(100*audit_coverage.set_index('feature').unlabeled_outside_labeled_range).sort_values().plot.barh(ax=axes[1])
axes[1].set(title='Outside labeled range in supplied coordinates',xlabel='Unlabeled rows (%)',ylabel='')
plt.tight_layout();plt.show()

**결과와 판단:** 충전시간은 라벨 데이터에서 고유값 2개, 비라벨에서 97개이고, 사출시간은 3개 대 91개다. 라벨 데이터의 591개 조건에 비해 비라벨은 29,707개 고유 조건이다. 라벨 데이터가 실제 공정조건의 변화를 충분히 대표하는지 확인할 필요가 있다. 더 적은 행 수가 고유값 수 차이의 일부를 설명할 수 있으므로, 이 수치만으로 측정 정밀도나 생성 방식을 확정하지 않는다.

두 파일은 상수열을 제외한 평균이 거의 0, 모집단 표준편차가 거의 1이다. 파일별 표준화와 일치하는 수치지만, 원래 물리 단위의 정규화 계수는 이 분석으로 복원할 수 없다. 같은 숫자 임계값을 물리 규격으로 해석하지 않는다.

`Clamp_Open_Position`은 학습에서 상수인 반면 비라벨에서는 53개 값으로 변하며, 비라벨의 모든 값이 학습의 단일 값과 다르다. 이 열을 제외해도 **18,416/35,941행(51.24%)**이 하나 이상의 학습 관측 범위를 벗어난다. 이는 제공 좌표계의 적용 범위 차이이며, 실제 공정 이상이 51.24%라는 의미가 아니다.

**처리 결정:** 공통 원본 24열을 보존하고 라벨 데이터의 상수열을 명세에 기록한다. 비라벨을 별도로 다시 표준화하거나 라벨 관측 범위로 강제 클리핑하지 않는다. 범위 밖 비율은 데이터 적용 범위 점검 지표로 남긴다. 원본 단위·정규화 계수·데이터 생성 기준을 확인해야 한다.

## 3.3 분할 방식이 만드는 검증 실패

같은 데이터에 대해 행 단위 층화 분할과 패턴 단위 층화 분할을 비교한다. 모델 점수의 유불리가 아니라, 검증이 '처음 보는 조건'을 평가하는지부터 확인한다. 랜덤 시드는 42, 폴드는 3개로 고정한다.

In [ ]:
audit_splitter=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
audit_pattern_fold=pd.Series(-1,index=audit_patterns.index,dtype='int64')
for k,(_,va) in enumerate(audit_splitter.split(np.zeros((len(audit_patterns),1)),audit_patterns.type)):
    audit_pattern_fold.iloc[va]=k
audit_folds=audit_groups.map(audit_pattern_fold)
audit_split_rows=[]
for scheme in ['row_stratified','pattern_grouped']:
    splits=(audit_splitter.split(audit_X,audit_y) if scheme=='row_stratified' else
            ((np.flatnonzero(audit_folds.ne(k)),np.flatnonzero(audit_folds.eq(k))) for k in range(3)))
    for k,(tr,va) in enumerate(splits):
        seen=audit_groups.iloc[va].isin(audit_groups.iloc[tr]).to_numpy()
        bad=audit_y.iloc[va].eq(1).to_numpy()
        audit_split_rows.append([scheme,k,len(va),int(bad.sum()),int(seen.sum()),int((seen & bad).sum())])
        if scheme=='pattern_grouped': assert not seen.any()
audit_split_table=pd.DataFrame(audit_split_rows,columns=[
    'scheme','fold','valid_rows','valid_defects','rows_with_seen_pattern','defects_with_seen_pattern'])
display(audit_split_table)
fig,ax=plt.subplots(figsize=(8,3))
audit_split_table.groupby('scheme').rows_with_seen_pattern.sum().plot.bar(ax=ax,rot=0)
ax.set(title='Validation rows whose exact input is also in training',ylabel='Rows',xlabel='')
for container in ax.containers:ax.bar_label(container)
plt.tight_layout();plt.show()

**결과와 판단:** 행 분할에서는 검증 전체 1,182행 중 **794행(67.17%)**, 불량 25행 중 **22행**이 학습에 같은 입력을 가진다. 그룹 분할에서는 두 수치 모두 0이다. 특히 불량과 다른 라벨의 양품 짝이 학습에 들어가는 경우도 있으므로, 중복 분할이 반드시 점수를 올린다고 단정하지 않는다. 확실한 문제는 새 공정조건에 대한 일반화와 이미 본 조건의 재등장이 섞인다는 점이다.

**처리 결정:** 이후 실험은 동일한 패턴 그룹 폴드를 사용한다. ID에 시간 의미가 확인되기 전에는 시간 검증이라고 부르지 않는다. 그룹 검증도 비라벨 파일의 조건 다양성·정규화 차이를 재현하는 것은 아니므로, 적용 데이터 성능을 보장하지 않는다.

## 3.4 이산 조합으로 관측 조건군 정의

**질문:** 충전시간 2개 값과 사출시간 3개 값이 실제로 어떤 조합으로 나타나는가? 최대 6개 조합을 확인하고, 관측 조합별 다른 공정값의 중앙값을 비교한다. 분석 단위는 원본 행이 아니라 591개 고유 전체입력 패턴이다.

조건군은 두 변수의 관측 조합에 붙인 이름이다. 이를 실제 생산 모드·설비 상태로 단정하거나 비라벨 데이터에 같은 물리 조건군으로 적용하지 않는다.

In [ ]:
structure_keys=['Filling_Time','Injection_Time']
structure_patterns=audit_work.drop_duplicates(audit_features).set_index('pattern_id')[audit_features].copy()
structure_patterns['conflicting']=audit_patterns.type.eq('conflicting').astype(int)
structure_combos=structure_patterns[structure_keys].drop_duplicates().sort_values(structure_keys)
structure_lookup={tuple(values):f'G{i+1}' for i,values in enumerate(structure_combos.to_numpy())}
structure_patterns['condition_group']=[structure_lookup[tuple(v)] for v in structure_patterns[structure_keys].to_numpy()]
structure_groups=structure_patterns.groupby('condition_group').agg(
    Filling_Time=('Filling_Time','first'),Injection_Time=('Injection_Time','first'),
    patterns=('conflicting','size'))
structure_groups['original_rows']=2*structure_groups.patterns
display(structure_groups)
structure_count_grid=structure_patterns.groupby(structure_keys).size().unstack('Injection_Time').reindex(
    index=sorted(structure_patterns.Filling_Time.unique()),columns=sorted(structure_patterns.Injection_Time.unique())).fillna(0).astype(int)
structure_context=['Max_Injection_Speed','Max_Back_Pressure','Plasticizing_Time',
                   'Cushion_Position','Barrel_Temperature_5','Mold_Temperature_3']
structure_medians=structure_patterns.groupby('condition_group')[structure_context].median()
display(structure_medians.round(3))
fig,axes=plt.subplots(1,2,figsize=(15,4))
sns.heatmap(structure_count_grid,annot=True,fmt='d',cmap='Blues',ax=axes[0],
    xticklabels=[f'{v:.3f}' for v in structure_count_grid.columns],
    yticklabels=[f'{v:.3f}' for v in structure_count_grid.index])
axes[0].set(title='Unique patterns in observed time combinations')
sns.heatmap(structure_medians,annot=True,fmt='.2f',cmap='vlag',center=0,ax=axes[1])
axes[1].set(title='Other process medians by observed combination',ylabel='')
plt.tight_layout();plt.show()

**결과와 해석:** 6개 가능한 조합 중 4개만 관측된다. G1·G4는 사출시간의 양 끝 값에 해당하며 각각 3개·5개 패턴뿐이다. 같은 중앙 사출시간을 갖는 G2·G3가 각각 241개·342개 패턴으로 전체의 대부분을 차지한다. 이 두 조건군은 충전시간 값으로 구분된다.

G2의 최대 사출 속도 중앙값은 0.618, 배압은 -0.708이고, G3는 각각 -0.952, 0.536이다. 즉 충전시간의 두 값은 다른 공정값의 변화와 함께 관측된다. 반면 배럴 온도 5의 중앙값은 두 조건군 모두 -0.031이다. 모든 변수가 같은 방향으로 달라지는 것은 아니다.

**판단:** 전체를 한 분포로만 보는 대신 두 주요 관측 조건군을 나누어 해석할 근거가 있다. 그러나 이 조합을 독립적인 불량 원인이나 물리 운전 모드로 확정하지 않는다. G1·G4는 표본이 적어 별도 정상 조건이나 삭제할 이상값으로 정하지 않는다.

## 3.5 조건군별 충돌 패턴의 위치와 빈도

**질문:** 다른 공정값을 갖는 조건군 중 어느 쪽에 충돌이 집중되는가? 패턴 기준 충돌 비율과 원본 행 기준 불량 비율을 구분한다. RG3에서 충돌 패턴은 양품 1행·불량 1행으로 구성되어 두 비율이 다르다. 이 유형은 분석용이며 입력에 추가하지 않는다.

In [ ]:
structure_conflicts=structure_patterns.groupby('condition_group').agg(
    patterns=('conflicting','size'),conflicting_patterns=('conflicting','sum'))
structure_conflicts['normal_only_patterns']=structure_conflicts.patterns-structure_conflicts.conflicting_patterns
structure_conflicts['conflicting_pattern_fraction']=structure_conflicts.conflicting_patterns/structure_conflicts.patterns
structure_conflicts['original_rows']=2*structure_conflicts.patterns
structure_conflicts['defect_rows']=structure_conflicts.conflicting_patterns
structure_conflicts['observed_defect_fraction']=structure_conflicts.defect_rows/structure_conflicts.original_rows
display(structure_conflicts.round(4))
fig,ax=plt.subplots(figsize=(8,3))
(100*structure_conflicts.conflicting_pattern_fraction).plot.bar(ax=ax,rot=0)
ax.set(title='Conflicting patterns by observed condition group',ylabel='Conflicting patterns (%)',xlabel='Condition group')
for i,(_,r) in enumerate(structure_conflicts.iterrows()):
    ax.text(i,100*r.conflicting_pattern_fraction+.1,f'{int(r.conflicting_patterns)}/{int(r.patterns)}',ha='center')
ax.set_ylim(0,6);plt.tight_layout();plt.show()

**결과와 해석:** G2는 10/241개 패턴(4.15%), G3는 15/342개 패턴(4.39%)이 충돌한다. 원본 불량 비율로는 각각 10/482행(2.07%), 15/684행(2.19%)이다. 공정값 중앙값의 차이와 달리 두 주요 조건군의 충돌 비율은 비슷하다. 이 조건군 구분 자체로 불량이 집중되는 집단을 설명하기는 어렵다.

G1·G4의 충돌은 0개지만 분모가 3개·5개 패턴이므로 안전한 조건이라는 근거는 아니다. **판단:** 조건군은 해석용으로 유지하되 조건군별 라벨 변경이나 정상 확정 규칙을 만들지 않는다. 다음에는 기존 집중 구간의 경향이 조건군 안에서도 나타나는지 확인한다.

## 3.6 기존 집중 구간은 조건군 안에서도 유지되는가?

기존에 관측한 `Barrel_Temperature_5 > 양품 중앙값` 및 `Max_Injection_Speed > 양품 중앙값`을 그대로 사용한다. 조건군마다 경계를 다시 최적화하지 않는다. 동일한 구간을 G2·G3 안팎으로 나눠 조건군의 구성 비율만으로 기존 집중이 설명되는지 살펴본다.

In [ ]:
structure_normal=audit_labeled.loc[audit_y.eq(0)]
structure_barrel_cut=structure_normal.Barrel_Temperature_5.median()
structure_speed_cut=structure_normal.Max_Injection_Speed.median()
structure_patterns['focus_region']=(structure_patterns.Barrel_Temperature_5.gt(structure_barrel_cut)&
                                     structure_patterns.Max_Injection_Speed.gt(structure_speed_cut))
structure_full_index=pd.MultiIndex.from_product([structure_groups.index,[False,True]],names=['condition_group','focus_region'])
structure_region=structure_patterns.groupby(['condition_group','focus_region']).conflicting.agg(
    patterns='size',conflicting_patterns='sum').reindex(structure_full_index,fill_value=0)
structure_region['original_rows']=2*structure_region.patterns
structure_region['defect_rows']=structure_region.conflicting_patterns
structure_region['normal_rows']=structure_region.original_rows-structure_region.defect_rows
structure_region['observed_defect_fraction']=structure_region.defect_rows/structure_region.original_rows.replace(0,np.nan)
display(structure_region)
structure_inside=structure_region.xs(True,level='focus_region')
assert structure_inside.original_rows.sum()==218 and structure_inside.defect_rows.sum()==9
fig,ax=plt.subplots(figsize=(8,4))
(100*structure_region.observed_defect_fraction.unstack()).rename(columns={False:'Outside fixed region',True:'Inside fixed region'}).plot.bar(ax=ax,rot=0)
ax.set(title='Same fixed region, compared within condition groups',ylabel='Observed defect rows (%)',xlabel='Condition group')
plt.tight_layout();plt.show()

**결과와 해석:** G2에서는 구간 안이 7/190행(3.68%), 밖이 3/292행(1.03%)이다. G3에서도 안이 2/28행(7.14%), 밖이 13/656행(1.98%)이다. 두 주요 조건군 모두 안쪽의 관측 불량 비율이 높으므로, 전체 9/218행(4.13%)이라는 관측을 조건군 혼합만으로 설명할 수는 없다.

그러나 G3의 안쪽은 14개 패턴·불량 2행뿐이다. G2는 95/241개 패턴이 구간 안에 있지만 G3는 14/342개로, 구간이 포함하는 조건군의 비중도 크게 다르다. G1·G4에는 해당 구간 관측이 없어 빈 칸을 0%로 해석하지 않는다.

**판단:** 배럴 온도 5·최대 사출 속도의 동시 조건은 계속 확인할 관측 가설로 남긴다. 동일 데이터에서 발견한 구간을 다시 나눠 본 사후 탐색이므로 독립 재현이나 인과 검증은 아니다. 구간 밖 불량 16행이 있고 안에 양품 209행이 있으므로 라벨 교정·행 삭제·확정 판정 기준으로 사용하지 않는다.

## 3.7 동일 입력의 짝을 제외한 주변 조건의 구성

**질문:** 충돌 패턴 주변에도 충돌 패턴이 모이는가? 591개 고유 조건만 사용해 동일 입력의 양품 짝을 제거한 상태로 비교한다. 거리 비교는 각 조건군 안에서 수행한다.

공정값을 전체 고유 패턴에서 평균 순위 백분율로 바꿔 큰 수치 간격의 영향을 줄이고, 순위 좌표의 유클리드 거리를 계산한다. 상수열을 제외한 23개 변수와, 절대 Spearman 상관 0.9 이상일 때 원본 열 순서상 앞선 변수 하나만 남긴 거리용 21개 변수 결과를 비교한다. 이는 거리 정의의 민감도 확인이며 원본 입력 삭제가 아니다.

이웃 수 5·10·20과 동률인 경계 이웃을 모두 포함한다. 자기 자신은 제외하고 작은 조건군은 가능한 이웃만 사용한다. 비교 기준으로 조건군별 충돌 개수를 보존한 채 위치를 2,000회 무작위 재배치한다(시드 42). 재배치는 관측 위치의 집중 정도를 설명하는 참고 분포이며, 결과로 원본 라벨을 변경하지 않는다. 분류기나 예측 모델은 사용하지 않는다.

In [ ]:
structure_distance_columns=[c for c in audit_features if structure_patterns[c].nunique()>1]
structure_abs_corr=structure_patterns[structure_distance_columns].corr(method='spearman').abs()
structure_reduced=[];structure_redundant=[]
for c in structure_distance_columns:
    similar=[other for other in structure_reduced if structure_abs_corr.loc[c,other]>=.9]
    if similar:structure_redundant.append([c,similar[0],structure_abs_corr.loc[c,similar[0]]])
    else:structure_reduced.append(c)
structure_distance_redundancy=pd.DataFrame(structure_redundant,columns=['excluded_from_alternative_distance','retained','abs_spearman'])
display(structure_distance_redundancy)
structure_flags=structure_patterns.conflicting.to_numpy()
structure_group_array=structure_patterns.condition_group.to_numpy()
structure_rng=np.random.default_rng(42)
structure_permutations=np.tile(structure_flags,(2000,1)).astype(float)
for label in structure_groups.index:
    ids=np.flatnonzero(structure_group_array==label)
    for b in range(len(structure_permutations)):
        structure_permutations[b,ids]=structure_rng.permutation(structure_flags[ids])
structure_neighbor_rows=[];structure_neighbor_cases=[]
for distance_name,columns in [('all_nonconstant',structure_distance_columns),('correlation_reduced',structure_reduced)]:
    ranked=structure_patterns[columns].rank(method='average',pct=True).to_numpy()
    distances=cdist(ranked,ranked)/np.sqrt(len(columns))
    np.fill_diagonal(distances,np.inf)
    distances[structure_group_array[:,None]!=structure_group_array[None,:]]=np.inf
    for k in [5,10,20]:
        effective_k=np.minimum(k,np.isfinite(distances).sum(axis=1))
        assert (effective_k>0).all()
        boundary=np.array([np.partition(row,int(kk)-1)[int(kk)-1] for row,kk in zip(distances,effective_k)])
        adjacency=(distances<=boundary[:,None]+1e-12)&np.isfinite(distances)
        assert not np.diag(adjacency).any()
        weights=csr_matrix(adjacency/adjacency.sum(axis=1)[:,None])
        neighbor_fraction=weights@structure_flags
        observed=float(neighbor_fraction[structure_flags==1].mean())
        randomized=np.sum((weights@structure_permutations.T)*structure_permutations.T,axis=0)/structure_flags.sum()
        structure_neighbor_rows.append([distance_name,k,len(columns),observed,float(randomized.mean()),
            float(np.quantile(randomized,.025)),float(np.quantile(randomized,.975))])
        for pos in np.flatnonzero(structure_flags==1):
            structure_neighbor_cases.append([distance_name,k,int(structure_patterns.index[pos]),
                structure_group_array[pos],int(adjacency[pos].sum()),float(neighbor_fraction[pos])])
structure_neighbor_summary=pd.DataFrame(structure_neighbor_rows,columns=[
    'distance','requested_neighbors','distance_features','observed_conflict_neighbor_fraction',
    'randomized_mean','randomized_q025','randomized_q975'])
structure_neighbor_detail=pd.DataFrame(structure_neighbor_cases,columns=[
    'distance','requested_neighbors','pattern_id','condition_group','actual_neighbors','conflict_neighbor_fraction'])
display(structure_neighbor_summary.round(4))
fig,ax=plt.subplots(figsize=(10,4))
for i,r in structure_neighbor_summary.iterrows():
    ax.plot([i,i],[100*r.randomized_q025,100*r.randomized_q975],color='gray',linewidth=4)
    ax.scatter(i,100*r.randomized_mean,color='#3274A1',label='Randomized mean' if i==0 else None)
    ax.scatter(i,100*r.observed_conflict_neighbor_fraction,color='#CC3311',marker='x',s=70,label='Observed' if i==0 else None)
ax.set_xticks(range(len(structure_neighbor_summary)),[
    f'{r.distance}\nk={r.requested_neighbors}' for r in structure_neighbor_summary.itertuples()])
ax.set(ylabel='Conflicting neighbors (%)',title='Within-group neighborhood: observed vs randomized reference')
ax.legend();plt.tight_layout();plt.show()

**결과와 해석:** 대체 거리에서 제외한 변수는 평균 배압(최대 배압과 상관 0.944)과 금형 온도 4(온도 3과 0.987)이다. 원본 저장 데이터의 열은 그대로 유지한다.

충돌 조건의 이웃 10개를 볼 때 주변 충돌 비율은 23개 변수 거리에서 5.6%, 중복 정보를 줄인 21개 변수 거리에서 6.4%다. 같은 조건군 안에서 충돌 개수를 유지한 무작위 재배치 평균은 약 3.9%, 중앙 95% 참고 범위는 각각 1.2~7.2%다. 5·10·20개 이웃과 두 거리 정의의 관측값 모두 각 참고 범위 안에 있다.

**판단:** 일부 가까운 충돌이 있지만, 이번 거리 정의와 조건군 내 비교에서 뚜렷한 별도 충돌 군집을 확인했다고 보기 어렵다. 이는 군집이 없다는 증명이 아니다. 충돌 25개가 적고 거리의 정의에도 의존하므로, 주변 양품을 불량으로 바꾸거나 국소 이상값으로 삭제하는 근거로 쓰지 않는다. 참고 범위는 관측값의 신뢰구간이나 여러 탐색을 보정한 검정 결과가 아니다.

## 3.8 충전시간을 중심으로 비라벨 적용 범위 분해

**질문:** 비라벨의 51.24% 범위 이탈 중 충전시간이 얼마나 기여하는가? 라벨의 두 충전시간 값에 대한 위치를 하한 미만·관측값 일치·두 값 사이·상한 초과로 나누고, 다른 비상수 변수의 범위 이탈과 교차한다. 일치 허용오차는 표준화 좌표에서 절대 1e-10으로 고정한다.

이 구분을 생산 모드로 해석하거나 비라벨을 가까운 라벨 값으로 강제 대응시키지 않는다. 서로 다른 파일의 제공 좌표계 비교이며 실제 물리 단위의 범위 차이는 별도 확인이 필요하다.

In [ ]:
structure_fill_low=audit_X.Filling_Time.min();structure_fill_high=audit_X.Filling_Time.max()
structure_unlabeled_fill=audit_U.Filling_Time
structure_on_observed=(np.isclose(structure_unlabeled_fill,structure_fill_low,atol=1e-10,rtol=0)|
                       np.isclose(structure_unlabeled_fill,structure_fill_high,atol=1e-10,rtol=0))
structure_fill_category=np.select([structure_on_observed,structure_unlabeled_fill.lt(structure_fill_low),
                                  structure_unlabeled_fill.gt(structure_fill_high)],
                                 ['on_observed','below','above'],default='between')
structure_other_columns=[c for c in audit_active if c!='Filling_Time']
structure_other_outside=(audit_U[structure_other_columns].lt(audit_X[structure_other_columns].min())|
                         audit_U[structure_other_columns].gt(audit_X[structure_other_columns].max())).any(axis=1)
structure_coverage_cross=pd.crosstab(pd.Series(structure_fill_category,name='filling_location'),
    structure_other_outside.rename('other_variable_outside')).reindex(
        index=['below','on_observed','between','above'],columns=[False,True],fill_value=0)
structure_coverage_cross.columns=['other_variables_within_range','other_variable_outside']
display(structure_coverage_cross)
structure_fill_outside=structure_unlabeled_fill.lt(structure_fill_low)|structure_unlabeled_fill.gt(structure_fill_high)
structure_coverage_parts=pd.DataFrame([
    ['Filling only',int((structure_fill_outside & ~structure_other_outside).sum())],
    ['Filling and other',int((structure_fill_outside & structure_other_outside).sum())],
    ['Other only',int((~structure_fill_outside & structure_other_outside).sum())],
    ['Within all nonconstant ranges',int((~structure_fill_outside & ~structure_other_outside).sum())],
],columns=['range_category','rows'])
structure_coverage_parts['fraction']=structure_coverage_parts.rows/len(audit_U)
display(structure_coverage_parts)
assert structure_coverage_parts.rows.sum()==len(audit_U)
assert structure_coverage_parts.iloc[:3].rows.sum()==int(audit_outside_rows.sum())
fig,ax=plt.subplots(figsize=(10,4))
structure_coverage_parts.set_index('range_category').rows.plot.bar(ax=ax,rot=15)
for container in ax.containers:ax.bar_label(container)
ax.set(title='Unlabeled support in supplied coordinates (constant column excluded)',ylabel='Rows',xlabel='')
plt.tight_layout();plt.show()

**결과와 판단:** 충전시간이 두 관측값 사이에 있는 비라벨 행은 **22,792행(63.42%)**, 상한 초과는 **13,147행**, 하한 미만은 **2행**이다. 허용오차 내 두 관측값과 일치하는 행은 0개다. 충전시간만 보면 범위 안에 있는 행도 라벨 데이터에서 직접 관측된 값은 아니다.

비상수 변수의 범위 밖 18,416행은 **충전시간만 이탈 8,986행**, **충전시간과 다른 변수 동시 이탈 4,163행**, **다른 변수만 이탈 5,267행**으로 나뉜다. 따라서 기존 51.24%를 여러 변수의 전반적 이탈로만 설명하면 안 된다. 반대로 충전시간 하나의 문제로 모두 설명할 수도 없다.

**처리 결정:** 비라벨 값을 두 라벨 값으로 반올림하거나 조건군에 강제 배정하지 않는다. 라벨의 희소한 값과 비라벨의 중간값·외부값 구조를 범위 점검표에 남긴다. 각 파일의 정규화 기준이 확인되기 전까지 중간값을 물리적 보간, 범위 밖을 실제 공정 이상이라고 부르지 않는다.

In [ ]:
structure_output_dir=audit_root/'output/modular/eda/rg3_structure_eda'
structure_output_dir.mkdir(parents=True,exist_ok=True)
structure_tables={
    'condition_groups.csv':structure_groups.reset_index(),
    'condition_medians.csv':structure_medians.reset_index(),
    'conflicts_by_condition.csv':structure_conflicts.reset_index(),
    'fixed_region_by_condition.csv':structure_region.reset_index(),
    'distance_redundancy.csv':structure_distance_redundancy,
    'neighborhood_summary.csv':structure_neighbor_summary,
    'neighborhood_cases.csv':structure_neighbor_detail,
    'filling_coverage_cross.csv':structure_coverage_cross.reset_index(),
    'coverage_components.csv':structure_coverage_parts,
}
for name,table in structure_tables.items():table.to_csv(structure_output_dir/name,index=False,encoding='utf-8')
structure_config={
    'scope':'Descriptive EDA only; no predictive model or source-label modification.',
    'condition_features':structure_keys,'unit':'Unique full-input pattern except row coverage tables',
    'fixed_region':{'Barrel_Temperature_5_gt':float(structure_barrel_cut),'Max_Injection_Speed_gt':float(structure_speed_cut)},
    'neighbors':[5,10,20],'neighbor_scope':'Within condition group, self excluded, boundary ties included',
    'distance':'Euclidean on global average percentile ranks, divided by sqrt(feature count)',
    'alternative_distance_features':structure_reduced,
    'randomization':{'seed':42,'repetitions':2000,'preserve':'Conflict counts within each condition group'},
    'coordinate_equality_atol':1e-10,
}
(structure_output_dir/'eda_config.json').write_text(json.dumps(structure_config,indent=2)+'\n',encoding='utf-8')
print('추가 EDA 표와 설정 저장:',structure_output_dir)

## 3.9 세 가지 비지도 군집화 비교

**질문:** 공정값만으로 만든 군집에서 양품 전용·충돌 패턴이 다르게 분포하는가? **계층적 군집화(Ward) → K-Medoids → K-Means** 순서로 적용한다. 분류기 학습이나 예측 성능 평가가 아니라 공정조건의 구조를 확인하는 EDA다.

원본 1,182행을 591개 고유 전체입력 패턴으로 표현한 뒤 군집 번호를 원본 행에 다시 연결한다. 라벨·충돌 여부·ID·행 순서·기존 조건군은 군집 입력에서 제외한다. 군집화 후에만 라벨과 기존 G1~G4의 구성을 비교한다.

**공통 기준:** 주 분석은 상수열을 제외한 제공 표준화 값 23개와 유클리드 거리다. 군집 수 후보는 2~6개이며 세 방법의 실루엣 평균이 가장 높은 공통 군집 수를 선택한다. 불량 집중도를 이용해 군집 수를 고르지 않는다. 추가로 23개 변수의 순위 좌표, 상관 중복을 줄인 21개 변수의 순위 좌표를 비교한다. 원본 저장 열과 라벨은 변경하지 않는다.

K-Medoids는 전용 외부 패키지 대신 군집 내 거리합을 최소화하는 관측점으로 대표점을 반복 갱신하는 **alternate 방식**을 명시적으로 구현한다. 5회 초기화 중 거리합이 작은 결과를 사용하며, 전역 최적해나 PAM의 전체 교환 최적화를 보장하지 않는다. K-Means는 20회 초기화, 시드는 42다.

In [ ]:
def medoids(D,k,seed=42,n_init=5,max_iter=100):
    best=None
    for restart in range(n_init):
        rng=np.random.default_rng(seed+restart)
        med=[int(rng.integers(len(D)))]
        while len(med)<k:
            nearest=D[:,med].min(axis=1)**2
            med.append(int(rng.choice(len(D),p=nearest/nearest.sum())))
        for step in range(max_iter):
            labels=D[:,med].argmin(axis=1)
            updated=[]
            for j in range(k):
                members=np.flatnonzero(labels==j)
                updated.append(int(members[np.argmin(D[np.ix_(members,members)].sum(axis=1))]))
            if updated==med:break
            med=updated
        labels=D[:,med].argmin(axis=1);cost=D[np.arange(len(D)),np.asarray(med)[labels]].sum()
        if best is None or cost<best[0]:best=(cost,labels,np.asarray(med),step+1)
    return best

# 작은 두 집단에서 대표점·목적함수가 정확히 계산되는지 확인한다.
cluster_toy=cdist(np.array([[0.],[1.],[2.],[10.],[11.],[12.]]),np.array([[0.],[1.],[2.],[10.],[11.],[12.]]))
cluster_toy_fit=medoids(cluster_toy,2)
assert np.isclose(cluster_toy_fit[0],4.)
assert set(cluster_toy_fit[2])=={1,4}

cluster_methods=['Ward','KMedoids','KMeans']
cluster_forms={
    'supplied_23':structure_patterns[structure_distance_columns].to_numpy(),
    'rank_23':structure_patterns[structure_distance_columns].rank(method='average',pct=True).to_numpy(),
    'rank_21':structure_patterns[structure_reduced].rank(method='average',pct=True).to_numpy(),
}
cluster_distances={};cluster_linkages={};cluster_assignments={};cluster_medoids={};cluster_candidate_rows=[]
with threadpool_limits(limits=1):
    for representation,values in cluster_forms.items():
        distances=cdist(values,values);np.fill_diagonal(distances,0)
        cluster_distances[representation]=distances
        cluster_linkages[representation]=linkage(values,method='ward')
        for k in range(2,7):
            ward_labels=cut_tree(cluster_linkages[representation],n_clusters=k).ravel()
            medoid_fit=medoids(distances,k,seed=42,n_init=5)
            km=KMeans(n_clusters=k,n_init=20,random_state=42).fit(values)
            assignments={'Ward':ward_labels,'KMedoids':medoid_fit[1],'KMeans':km.labels_}
            cluster_medoids[(representation,k)]=medoid_fit[2]
            for method,labels in assignments.items():
                assert len(np.unique(labels))==k
                cluster_assignments[(representation,k,method)]=labels
                counts=np.bincount(labels)
                cluster_candidate_rows.append([representation,k,method,
                    silhouette_score(distances,labels,metric='precomputed'),int(counts.min()),int(counts.max())])
cluster_candidates=pd.DataFrame(cluster_candidate_rows,columns=[
    'representation','k','method','silhouette','smallest_cluster','largest_cluster'])
cluster_selected_k={name:int(g.groupby('k').silhouette.mean().idxmax())
                     for name,g in cluster_candidates.groupby('representation')}
display(cluster_candidates.round(4))
display(pd.DataFrame(cluster_selected_k.items(),columns=['representation','selected_common_k']))
fig,axes=plt.subplots(1,3,figsize=(15,4),sharey=True)
for ax,(name,g) in zip(axes,cluster_candidates.groupby('representation',sort=False)):
    g.pivot(index='k',columns='method',values='silhouette')[cluster_methods].plot(marker='o',ax=ax)
    ax.set(title=name,ylabel='Silhouette',xlabel='Number of clusters')
plt.tight_layout();plt.show()

**군집 수 비교 해석:** 주 분석에서 k=2의 실루엣은 Ward 0.188, K-Medoids 0.197, K-Means 0.199다. 세 방법 각각에서도 후보 2~6 중 2가 가장 높고, 두 순위 표현에서도 공통 선택은 2다. 여기서 2는 정상·불량 두 라벨을 맞추기 위해 정한 값이 아니다.

점수가 약 0.2이므로 군집 수를 더 늘리는 것보다 상대적으로 낫다는 결과와 군집 경계가 뚜렷하다는 주장은 구분한다. 원본 표현의 k=5에서는 3개 또는 5개짜리 작은 군집도 나타나므로 작은 군집을 불량 집단이라고 해석하지 않는다. 아래에서 같은 k=2를 기준으로 구성을 비교한다.

### 3.9.1 계층적 군집화: 큰 공정조건의 분기 확인

Ward는 병합에 따른 군집 내 제곱거리 증가를 기준으로 계층을 만든다. 덴드로그램은 마지막 12개 병합 단위만 표시하고, 세 방법의 비교는 공통 k를 사용한다. 군집 번호 자체는 임의의 이름이다.

In [ ]:
cluster_primary='supplied_23'
cluster_k=cluster_selected_k[cluster_primary]
fig,ax=plt.subplots(figsize=(11,4))
dendrogram(cluster_linkages[cluster_primary],truncate_mode='lastp',p=12,show_leaf_counts=True,ax=ax)
ax.set(title='Ward hierarchy: last 12 merged groups',ylabel='Linkage distance',xlabel='Merged groups / leaf counts')
plt.tight_layout();plt.show()
cluster_row_summaries=[];cluster_group_tables=[];cluster_profile_tables=[];cluster_pattern_tables=[]
for representation in cluster_forms:
    k=cluster_selected_k[representation]
    for method in cluster_methods:
        labels=cluster_assignments[(representation,k,method)]
        table=structure_patterns.copy()
        table['cluster']=labels
        summary=table.groupby('cluster').conflicting.agg(patterns='size',conflicting_patterns='sum')
        summary['normal_only_patterns']=summary.patterns-summary.conflicting_patterns
        summary['conflicting_pattern_fraction']=summary.conflicting_patterns/summary.patterns
        summary['original_rows']=2*summary.patterns
        summary['defect_rows']=summary.conflicting_patterns
        summary['normal_rows']=summary.original_rows-summary.defect_rows
        summary['original_defect_fraction']=summary.defect_rows/summary.original_rows
        summary['representation']=representation;summary['method']=method
        cluster_row_summaries.append(summary.reset_index())
        cross=pd.crosstab(table.cluster,table.condition_group).reset_index()
        cross['representation']=representation;cross['method']=method;cluster_group_tables.append(cross)
        profiles=table.groupby('cluster')[audit_features].median().reset_index()
        profiles['representation']=representation;profiles['method']=method;cluster_profile_tables.append(profiles)
        patterns=pd.DataFrame({'pattern_id':structure_patterns.index,'cluster':labels,
            'representation':representation,'method':method})
        cluster_pattern_tables.append(patterns)
cluster_summary=pd.concat(cluster_row_summaries,ignore_index=True)
cluster_condition_cross=pd.concat(cluster_group_tables,ignore_index=True).fillna(0)
cluster_profiles=pd.concat(cluster_profile_tables,ignore_index=True)
cluster_pattern_assignments=pd.concat(cluster_pattern_tables,ignore_index=True)
display(cluster_summary.query("representation == 'supplied_23' and method == 'Ward'"))
display(cluster_condition_cross.query("representation == 'supplied_23' and method == 'Ward'"))

**Ward 해석:** 두 군집은 221개·370개 패턴이고 충돌 패턴은 11개·14개다. 패턴 기준 충돌 비율은 4.98%·3.78%이며 두 군집 모두 양품 전용과 충돌 조건을 포함한다. 원본 행의 불량 비율은 이 비율의 절반이므로 혼동하지 않는다. 계층이 존재한다는 것만으로 정상·불량의 경계를 찾았다고 볼 수 없다.

### 3.9.2 K-Medoids: 실제 관측 대표조건으로 비교

대표점은 실제 관측된 전체입력 패턴이다. 대표 ID와 공정값은 군집을 설명하는 자료이며, 그 ID나 라벨을 거리 계산에 사용하지 않는다.

In [ ]:
cluster_medoid_positions=cluster_medoids[(cluster_primary,cluster_k)]
cluster_representatives=structure_patterns.iloc[cluster_medoid_positions][audit_features].copy()
cluster_representatives.insert(0,'cluster',np.arange(cluster_k))
cluster_representatives.insert(1,'representative_source_id',[
    audit_patterns.loc[pid,'source_ids'][0] for pid in cluster_representatives.index])
cluster_representatives=cluster_representatives.reset_index()
display(cluster_representatives)
display(cluster_summary.query("representation == 'supplied_23' and method == 'KMedoids'"))
display(cluster_condition_cross.query("representation == 'supplied_23' and method == 'KMedoids'"))

**K-Medoids 해석:** 두 군집은 229개·362개 패턴이고 충돌은 10개·15개로, 충돌 비율이 4.37%·4.14%다. 공정조건을 두 대표점으로 요약할 수 있지만 라벨 구성의 차이는 작다. 실제 대표조건을 보여줄 수 있다는 장점과 불량 집중을 설명하는 능력은 별개다. 대표점 교대 갱신의 국소해에 의존하므로 이후 표본 변화에 대한 민감도도 확인한다.

### 3.9.3 K-Means: 평균 중심과 공통 시각화

세 방법을 같은 PCA 2차원 좌표에 표시한다. PCA는 그림의 축을 만들기 위한 용도이며 군집은 전체 23개 좌표에서 계산했다. 주황색 빈 원은 충돌 패턴 위치로, 그림을 만들 때 사후에 표시한다.

In [ ]:
display(cluster_summary.query("representation == 'supplied_23' and method == 'KMeans'"))
display(cluster_condition_cross.query("representation == 'supplied_23' and method == 'KMeans'"))
cluster_pca=PCA(n_components=2)
cluster_coordinates=cluster_pca.fit_transform(cluster_forms[cluster_primary])
fig,axes=plt.subplots(1,3,figsize=(15,4),sharex=True,sharey=True)
for ax,method in zip(axes,cluster_methods):
    labels=cluster_assignments[(cluster_primary,cluster_k,method)]
    ax.scatter(cluster_coordinates[:,0],cluster_coordinates[:,1],c=labels,cmap='coolwarm',s=14,alpha=.5)
    mask=structure_patterns.conflicting.eq(1).to_numpy()
    ax.scatter(cluster_coordinates[mask,0],cluster_coordinates[mask,1],facecolors='none',edgecolors='#E69F00',s=65,label='Conflicting pattern')
    ax.set(title=method,xlabel='PC1',ylabel='PC2');ax.legend(fontsize=8)
fig.suptitle(f'Same projection for all methods: {cluster_pca.explained_variance_ratio_.sum():.1%} variance in two PCs')
plt.tight_layout();plt.show()

**K-Means 해석:** 두 군집은 273개·318개 패턴, 충돌은 14개·11개로 비율은 5.13%·3.46%다. 세 방법 중 주 분석에서 군집 간 충돌 비율 차이는 더 크지만, 라벨을 보고 이 방법을 우수하다고 선택하지 않는다. PCA 그림에서 보이는 겹침은 전체 공간의 군집 품질과 동일하지 않으며, 2차원에서 떨어져 보이는 점을 불량으로 단정하지 않는다.

### 3.9.4 방법·입력 표현·부분표본에 따른 일관성

방법 간 군집 일치는 ARI로 비교한다. 여기서 ARI는 **군집 번호끼리**의 일치도이며 정상·불량에 대한 정확도가 아니다. 값이 1이면 같은 분할이고, 0 부근이면 우연 일치 수준이다.

주 분석에서 무작위 80% 패턴을 비복원 추출해 10회 재군집화하고, 전체 자료의 군집 중 해당 부분과 얼마나 일치하는지 확인한다. 군집 수와 입력 좌표를 고정하며, 이 수치는 표본 변화 민감도이지 독립 검증 성능이 아니다.

In [ ]:
cluster_agreement=[];cluster_representation_agreement=[]
for representation in cluster_forms:
    k=cluster_selected_k[representation]
    for left,right in combinations(cluster_methods,2):
        cluster_agreement.append([representation,left,right,adjusted_rand_score(
            cluster_assignments[(representation,k,left)],cluster_assignments[(representation,k,right)])])
for method in cluster_methods:
    for alternative in ['rank_23','rank_21']:
        cluster_representation_agreement.append([method,alternative,adjusted_rand_score(
            cluster_assignments[(cluster_primary,cluster_k,method)],
            cluster_assignments[(alternative,cluster_selected_k[alternative],method)])])
cluster_agreement=pd.DataFrame(cluster_agreement,columns=['representation','method_1','method_2','ARI'])
cluster_representation_agreement=pd.DataFrame(cluster_representation_agreement,columns=['method','alternative_representation','ARI_vs_supplied'])
cluster_stability_rows=[]
with threadpool_limits(limits=1):
    for trial in range(10):
        rng=np.random.default_rng(100+trial)
        ids=np.sort(rng.choice(len(structure_patterns),size=int(.8*len(structure_patterns)),replace=False))
        values=cluster_forms[cluster_primary][ids]
        D=cluster_distances[cluster_primary][np.ix_(ids,ids)]
        trial_labels={
            'Ward':cut_tree(linkage(values,method='ward'),n_clusters=cluster_k).ravel(),
            'KMedoids':medoids(D,cluster_k,seed=42,n_init=5)[1],
            'KMeans':KMeans(n_clusters=cluster_k,n_init=20,random_state=42).fit_predict(values),
        }
        for method,labels in trial_labels.items():
            cluster_stability_rows.append([method,trial,adjusted_rand_score(
                cluster_assignments[(cluster_primary,cluster_k,method)][ids],labels)])
cluster_stability=pd.DataFrame(cluster_stability_rows,columns=['method','trial','ARI_vs_full_subset'])
display(cluster_agreement.round(3));display(cluster_representation_agreement.round(3))
display(cluster_stability.groupby('method').ARI_vs_full_subset.agg(['mean','min','max']).round(3))
display(cluster_summary[['representation','method','cluster','patterns','conflicting_patterns','conflicting_pattern_fraction']].round(4))
display(cluster_profiles.query("representation == 'supplied_23'")[['method','cluster']+structure_context].round(3))

**방법 간 비교:** 주 분석에서 Ward–K-Means의 ARI는 0.678, Ward–K-Medoids는 0.270, K-Medoids–K-Means는 0.487이다. 모두 k=2를 택해도 같은 경계로 나뉘지는 않는다. 군집 번호 0·1은 방법 간 같은 공정조건을 뜻하지 않으므로 교차표와 대표 공정값을 함께 읽는다.

원본 좌표에서 순위 좌표로 바꾸거나 상관 중복을 줄이면 배정도 달라진다. 따라서 한 번 얻은 군집 번호를 확정적인 생산 모드로 간주하지 않는다. 부분표본 ARI가 낮거나 변동이 큰 방법은 표본 구성에도 더 민감하다. 세 방법 모두 양쪽 군집에 충돌이 남으며, 21개 순위 좌표에서도 충돌 비율은 약 3.17~5.43% 범위로 나타난다.

**부분표본 비교 결과:** 80% 패턴을 10회 다시 추출했을 때 평균 ARI는 K-Means 0.984(최소 0.917), Ward 0.912(최소 0.760), K-Medoids 0.660(최소 0.464)이다. 이번 고정 설정에서는 K-Means의 분할이 가장 일관되었고, K-Medoids는 부분표본 구성과 대표점 선택에 더 민감했다. 이 순위는 불량 예측 성능의 순위가 아니다.

**기존 조건군과의 연결:** Ward의 한 군집에는 G2의 241개 중 213개, 다른 군집에는 G3의 342개 중 337개가 들어간다. K-Means도 각각 G2의 225개와 G3의 297개를 같은 쪽에 묶는다. 따라서 두 방법은 기존에 본 충전시간 중심의 큰 공정조건 차이를 상당 부분 다시 표현한다. 군집을 얻었다는 사실이 새로운 불량 원인을 찾았다는 뜻은 아니다.

### 3.9.5 군집별 라벨 집중의 참고 범위와 처리 결정

선택된 주 분석 군집은 고정하고, 기존 G1~G4 안에서 충돌 개수를 보존해 위치만 2,000회 바꾼다. 각 군집의 충돌 비율이 기존 조건군 구성을 고려한 참고 범위를 벗어나는지 확인한다. 군집 선택에 사용하지 않는 사후 설명이며, 여러 방법을 비교한 뒤 유리한 결과를 골라 유의성을 주장하지 않는다.

In [ ]:
cluster_reference_rows=[]
for method in cluster_methods:
    labels=cluster_assignments[(cluster_primary,cluster_k,method)]
    for cluster in range(cluster_k):
        mask=labels==cluster
        simulated=structure_permutations[:,mask].mean(axis=1)
        cluster_reference_rows.append([method,cluster,int(mask.sum()),
            float(structure_flags[mask].mean()),float(simulated.mean()),
            float(np.quantile(simulated,.025)),float(np.quantile(simulated,.975))])
cluster_reference=pd.DataFrame(cluster_reference_rows,columns=[
    'method','cluster','patterns','observed_fraction','randomized_mean','randomized_q025','randomized_q975'])
cluster_reference['outside_reference_interval']=(cluster_reference.observed_fraction.lt(cluster_reference.randomized_q025)|
                                                cluster_reference.observed_fraction.gt(cluster_reference.randomized_q975))
display(cluster_reference.round(4))
display(Markdown(f'**참고 비교 결과:** 주 분석의 6개 군집 중 '
    f'{int(cluster_reference.outside_reference_interval.sum())}개가 각 재배치 참고 구간 밖이다. '
    '이 범위는 관측 비율의 신뢰구간이나 다중비교를 보정한 검정 결과가 아니다. '
    '충돌 25개가 적다는 점을 포함해 탐색적으로 해석한다.'))
cluster_output_dir=audit_root/'output/modular/eda/rg3_clustering_eda'
cluster_output_dir.mkdir(parents=True,exist_ok=True)
cluster_exports={
    'candidate_comparison.csv':cluster_candidates,'cluster_composition.csv':cluster_summary,
    'condition_group_cross.csv':cluster_condition_cross,'cluster_medians.csv':cluster_profiles,
    'pattern_assignments.csv':cluster_pattern_assignments,'medoid_representatives.csv':cluster_representatives,
    'method_agreement.csv':cluster_agreement,'representation_agreement.csv':cluster_representation_agreement,
    'subsample_stability.csv':cluster_stability,'label_reference.csv':cluster_reference,
}
cluster_source_assignments=audit_work[['Unnamed: 0','pattern_id','PassOrFail']].merge(
    cluster_pattern_assignments.query("representation == 'supplied_23'"),on='pattern_id',how='left',validate='many_to_many')
assert len(cluster_source_assignments)==len(audit_work)*3
assert cluster_source_assignments.groupby(['method','pattern_id']).cluster.nunique().eq(1).all()
cluster_exports['source_row_assignments.csv']=cluster_source_assignments
for filename,table in cluster_exports.items():table.to_csv(cluster_output_dir/filename,index=False,encoding='utf-8')
cluster_config={
    'scope':'Unsupervised descriptive EDA; labels attached only after clustering.',
    'primary_features':structure_distance_columns,'reduced_rank_features':structure_reduced,
    'k_candidates':[2,3,4,5,6],'selected_common_k':cluster_selected_k,
    'selection':'Maximum mean silhouette across three methods within each representation, no labels.',
    'Ward':'Euclidean Ward linkage',
    'KMedoids':{'algorithm':'alternate medoid update','n_init':5,'max_iter':100,'seed':42},
    'KMeans':{'n_init':20,'seed':42},
    'stability':{'sample_fraction':.8,'trials':10,'sampling_seeds':list(range(100,110))},
    'label_reference':'2000 within-condition-group randomizations reused from section 3.7',
    'source_sha256':hashlib.sha256((audit_root/'data/origin/moldset_labeled_rg3.csv').read_bytes()).hexdigest(),
}
(cluster_output_dir/'eda_config.json').write_text(json.dumps(cluster_config,indent=2)+'\n',encoding='utf-8')
print('군집 EDA 결과 저장:',cluster_output_dir)

**종합 해석:** 이번 군집화는 두 개의 큰 공정조건 집단을 요약하는 데 유용하지만, 양품 전용·충돌 패턴을 나누는 뚜렷한 군집을 확인한 결과는 아니다. K-Medoids는 실제 대표조건 설명에, Ward는 병합 구조 설명에, K-Means는 비교 기준에 각각 장점이 있다. 실루엣의 작은 차이나 군집별 불량 비율만으로 한 방법을 최종 선택하지 않는다.

**전처리 반영:** 군집 번호는 별도 EDA 산출물로만 저장한다. 원본 라벨, 24개 공정변수, 기존 그룹 폴드는 유지하고 군집별 라벨 교정·행 삭제·자동 특징 추가를 하지 않는다. 동일 입력의 양품·불량 짝에는 동일 군집 번호가 연결되므로 그 두 행을 군집화로 구분할 수 없다는 구조적 한계도 유지된다.

후속 EDA에서 한 가지 공통 구분이 필요하다면 이번 표현·부분표본 점검에서는 K-Means를 기준으로 삼고 Ward로 구조를 교차 확인하는 것이 합리적이다. K-Medoids는 실제 대표조건 설명에 활용하되 분할의 민감도를 함께 밝힌다. 세 방법의 6개 주 분석 군집 모두 사후 재배치 참고 범위 안에 있어, 군집에 정상·불량 의미를 부여하지 않는다.

## 3.10 EDA에 근거한 전처리 결정

| 관측 결과 | 해석과 처리 결정 |
|---|---|
| 591개 패턴이 모두 2행이며 불량 25행은 양품과 동일 입력을 공유 | 원본 라벨 보존. 반복·검수·데이터 생성 기준 확인 필요 |
| ID와 인접 기록에 반복 구조가 있음 | ID·행 순서는 추적 정보로만 보관하며 시간으로 해석하지 않음 |
| 라벨·비라벨 데이터의 고유값 수와 관측 범위가 다름 | 공통 24열 보존. 물리 단위·정규화 과정을 확인하기 전 클리핑·별도 재표준화하지 않음 |
| `Clamp_Open_Position`은 라벨에서만 상수 | 상수 여부를 처리 명세에 기록. 비라벨에서 변하므로 원본 스키마에는 유지 |
| 행 단위 분할에서 794행의 동일 조건이 다른 분할에도 존재 | 같은 전체입력 패턴을 하나의 그룹으로 묶어 분할 정보 준비 |

현재 데이터만으로 제조 불량의 원인을 확정할 수는 없다. EDA의 결론은 라벨 충돌, 반복 구조, 적용 범위의 차이와 데이터 분할 시 주의점을 확인했다는 것이다. 아래 4절에서 이 근거에 따라 입력·타깃·메타데이터를 분리하고 저장한다.

**추가 EDA 반영:** 주요 조건군 G2·G3의 충돌 비율은 비슷하다. 기존 집중 구간의 경향은 두 조건군 안에도 남지만 사례가 적고 사후 탐색이다. 이웃 분석에서도 뚜렷한 별도 충돌 군집의 근거가 충분하지 않다. 조건군·집중 구간·이웃 정보로 라벨을 변경하지 않는다. 비라벨 충전시간은 두 관측값 사이 또는 밖에 있어 두 값으로 강제 반올림하지 않는다.

In [ ]:
audit_output_dir=audit_root / 'output/modular/eda/rg3_pipeline_audit'
audit_output_dir.mkdir(parents=True,exist_ok=True)
audit_tables={
    'pattern_summary.csv':audit_pattern_summary.reset_index(),
    'pair_structure.csv':audit_pair_structure,
    'feature_coverage.csv':audit_coverage,
    'split_overlap.csv':audit_split_table,
}
for filename,frame in audit_tables.items():
    frame.to_csv(audit_output_dir/filename,index=False,encoding='utf-8')
print('EDA 점검표 저장:',audit_output_dir)

# A. 참고 분석: KS 상위 변수와 공정별 분포

분석 순서: **불량 유형 구분 → 상위 5개 변수의 10개 쌍 → 개별 분포와 처리 방식별 민감도 → 공정 단계별 비교**.

2절의 `rg3_l_x`, `rg3_l_y`, `ks_df`는 불량 우선 중복 처리 결과다. 아래에서는 원본을 별도로 읽어 동일한 24개 공정값을 하나의 패턴으로 정의하고, 양품·불량의 공존을 확인한다. 저장 인덱스는 시간으로 해석하지 않는다.

파일에 제공된 표준화 값을 사용하며, 분포·상관·관측 비율은 탐색 결과다. KS는 분포 차이의 크기로 비교하고 중복·이산값·소수 불량이 있는 조건에서 p값만으로 유의성을 판정하지 않는다. CN7에서 발견한 비충돌 패턴의 존재나 집중 방향을 RG3에 가정하지 않는다.

이 절은 기존 탐색 결과를 보존한 참고 자료다. 전처리·검증·개선 판단의 근거는 본문의 3절이며, KS 순위를 입력 선택 기준으로 사용하지 않는다.

In [ ]:
eda3_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'data/origin/moldset_labeled_rg3.csv').is_file())
eda3_raw = pd.read_csv(eda3_root / 'data/origin/moldset_labeled_rg3.csv')
eda3_features = [c for c in eda3_raw if c not in ['Unnamed: 0', 'PassOrFail']]
assert len(eda3_features) == 24
assert np.isfinite(eda3_raw[eda3_features].to_numpy()).all()
assert set(eda3_raw.PassOrFail.unique()) == {0, 1}
eda3_raw = eda3_raw.rename(columns={'Unnamed: 0': 'source_id'})
eda3_raw['pattern_id'] = pd.MultiIndex.from_frame(eda3_raw[eda3_features]).factorize(sort=False)[0]
eda3_patterns = eda3_raw.groupby('pattern_id').PassOrFail.agg(rows='size', defects='sum')
eda3_patterns['normal'] = eda3_patterns.rows - eda3_patterns.defects
eda3_patterns['type'] = np.select(
    [eda3_patterns.defects.eq(0), eda3_patterns.normal.eq(0)],
    ['normal_only', 'defect_only'], default='conflicting')
eda3_raw['pattern_type'] = eda3_raw.pattern_id.map(eda3_patterns['type'])
eda3_normal = eda3_raw[eda3_raw.PassOrFail.eq(0)].copy()
eda3_bad = eda3_raw[eda3_raw.PassOrFail.eq(1)].copy()
eda3_normal_unique = eda3_normal.drop_duplicates(eda3_features)
eda3_bad_unique = eda3_bad.drop_duplicates(eda3_features)
eda3_current = rg3_l_x.copy()
eda3_current['PassOrFail'] = rg3_l_y
eda3_top5 = ks_df.sort_values('Rank').head(5).Feature.tolist()
assert len(set(eda3_top5)) == 5

def eda3_say(text):
    display(Markdown(text))

eda3_colors = {'normal':'#3274A1', 'conflicting':'#E69F00', 'defect_only':'#CC3311'}
def eda3_scatter(ax, x, y):
    ax.scatter(eda3_normal_unique[x], eda3_normal_unique[y], s=20, alpha=.3,
               color=eda3_colors['normal'], label=f'Normal patterns (n={len(eda3_normal_unique)})')
    for kind, marker in [('conflicting','o'), ('defect_only','x')]:
        g = eda3_bad_unique[eda3_bad_unique.pattern_type.eq(kind)]
        kwargs = {'facecolors':'none','edgecolors':eda3_colors[kind]} if marker == 'o' else {'color':eda3_colors[kind]}
        ax.scatter(g[x], g[y], s=65, marker=marker, linewidths=1.5,
                   label=f'{kind} defects (n={len(g)})', **kwargs)
    ax.set(xlabel=x, ylabel=y)
    ax.grid(alpha=.15)
    ax.legend(fontsize=7)

display(ks_df.sort_values('Rank').head(5)[['Rank','Feature','KS_Statistic','Normal_Count','Abnormal_Count']])
eda3_say('**이번 조합 분석에 사용할 기존 KS 상위 5개:** ' + ', '.join(eda3_top5))

## A.1 불량 유형 구분
**질문:** 불량의 24개 공정값이 양품에서도 관측되는가?

`conflicting`은 동일 입력에 두 라벨이 있는 패턴, `defect_only`는 현재 데이터에서 같은 입력의 양품이 없는 패턴이다. 행 수와 패턴 수를 구분한다.

In [ ]:
eda3_type_summary = eda3_patterns.groupby('type').agg(
    patterns=('rows','size'), rows=('rows','sum'), normal_rows=('normal','sum'), defect_rows=('defects','sum'))
display(eda3_type_summary)
eda3_bad_cases = eda3_bad[['source_id','pattern_id','pattern_type']].copy()
eda3_bad_cases['same_input_normal_rows'] = eda3_bad.pattern_id.map(eda3_patterns.normal)
eda3_bad_cases['same_input_defect_rows'] = eda3_bad.pattern_id.map(eda3_patterns.defects)
display(eda3_bad_cases)
display(pd.DataFrame([
    ['Original rows', len(eda3_raw), len(eda3_normal), len(eda3_bad)],
    ['Unique X + label', len(eda3_normal_unique)+len(eda3_bad_unique), len(eda3_normal_unique), len(eda3_bad_unique)],
    ['Existing defect-priority X', len(eda3_current), int(eda3_current.PassOrFail.eq(0).sum()), int(eda3_current.PassOrFail.eq(1).sum())]
], columns=['view','total','normal','defect']))
fig, ax = plt.subplots(figsize=(7,4))
eda3_bad.groupby('pattern_type').agg(rows=('pattern_id','size'), patterns=('pattern_id','nunique')).plot.bar(ax=ax, rot=0)
ax.set(title='RG3 defect rows and unique input patterns', ylabel='Count', xlabel='Defect type')
for container in ax.containers: ax.bar_label(container)
plt.tight_layout(); plt.show()
eda3_shared_n = int(eda3_bad.pattern_type.eq('conflicting').sum())
eda3_only_ids = eda3_bad.loc[eda3_bad.pattern_type.eq('defect_only'),'source_id'].tolist()
eda3_say(f'**결과:** 불량 {len(eda3_bad)}행 중 {eda3_shared_n}행은 동일 입력의 양품이 존재한다. '
         f'나머지 {len(eda3_bad)-eda3_shared_n}행의 ID는 {eda3_only_ids}이며 '
         f'{eda3_bad.loc[eda3_bad.pattern_type.eq("defect_only"),"pattern_id"].nunique()}개 고유 패턴이다. '
         '기존 불량 우선 처리는 충돌 조건의 양품을 제거하므로 원본의 관측 불량률이나 분리 가능성과 동일하게 해석하지 않는다.')

**해석과 다음 단계:** RG3의 불량 25행은 모두 서로 다른 충돌 패턴 25개에 속하며, 각 패턴에 동일 입력의 양품이 1행씩 있다. 비충돌 불량은 0개다. 나머지 566개 패턴은 양품이 각각 2행이다. 따라서 현재 24개 입력을 사용하는 결정적 규칙으로 충돌 패턴 안의 두 행에 서로 다른 답을 줄 수 없다. 이 결과만으로 검수 오류인지, 미측정 조건인지, 데이터 구성 과정의 영향인지 구분할 수는 없다.

A.2의 정확한 좌표 겹침은 이 구조상 모든 쌍에서 남을 것이므로 새로운 분리 근거가 될 수 없다. 조합 분석은 이를 확인하고, 넓은 구간에서 불량의 관측 빈도가 상대적으로 높은 조건이 있는지 점검하는 용도로 제한한다.

## A.2 상위 변수 조합의 추가 정보와 한계

**질문:** 단일 변수보다 두 변수의 조합이 더 유용한 정보를 주는가?

KS는 개별 변수의 분포 차이만 보여준다. 따라서 상위 5개의 10개 쌍에서 각 단일 값은 양품에 있지만 두 값의 조합은 없는 사례가 있는지 비교한다. 다만 RG3는 전체입력이 같은 양품이 모든 불량에 존재하므로, 정확한 값의 조합으로 새로 분리할 수 있는 사례는 없을 것으로 예상된다. 이를 10개 쌍에서 확인한 뒤 구간별 집중 정도를 별도로 해석한다.

파란 점은 양품 고유 전체입력 패턴, 주황색 빈 원은 충돌 불량, 빨간 ×는 비충돌 불량이다. RG3에는 빨간 ×에 해당하는 사례가 없다. Spearman 상관은 양품 고유 패턴에서 계산하며, 높은 상관을 독립적인 두 불량 원인으로 세지 않는다.

In [ ]:
eda3_pair_rows = []
eda3_pair_details = []
fig, axes = plt.subplots(5, 2, figsize=(16,23))
for ax, (x,y) in zip(axes.flat, combinations(eda3_top5, 2)):
    eda3_scatter(ax, x, y)
    freq = eda3_normal.groupby([x,y]).size()
    counts = np.array([freq.get(tuple(v),0) for v in eda3_bad_unique[[x,y]].to_numpy()])
    eda3_pair_rows.append([x,y,eda3_normal_unique[[x,y]].corr(method='spearman').iloc[0,1],
                           len(counts),int((counts>0).sum()),int((counts==0).sum())])
    for (_,r), count in zip(eda3_bad_unique.iterrows(), counts):
        eda3_pair_details.append([x,y,int(r.source_id),int(r.pattern_id),r.pattern_type,int(count),
                                  int(eda3_patterns.loc[r.pattern_id,'defects'])])
    ax.set_title(f'Exact pair shared with normal: {(counts>0).sum()}/{len(counts)} defect patterns', fontsize=10)
plt.tight_layout(); plt.show()
eda3_pair_summary = pd.DataFrame(eda3_pair_rows, columns=[
    'feature_1','feature_2','normal_pattern_spearman','defect_patterns','shared_with_normal','no_exact_normal_pair'])
eda3_pair_detail = pd.DataFrame(eda3_pair_details, columns=[
    'feature_1','feature_2','representative_id','pattern_id','pattern_type','normal_rows_matching_pair','defect_rows_in_pattern'])
display(eda3_pair_summary.round(3))
display(eda3_pair_detail.pivot(index=['representative_id','pattern_type'], columns=['feature_1','feature_2'], values='normal_rows_matching_pair'))
eda3_strong = eda3_pair_summary.loc[eda3_pair_summary.normal_pattern_spearman.abs().ge(.8), ['feature_1','feature_2','normal_pattern_spearman']]
eda3_say('**겹침과 중복 정보:** 아래 표는 양품 고유 패턴에서 |Spearman|≥0.8인 쌍이다. '
         '0.8은 관계를 요약하기 위한 표시 기준이다. 높은 상관은 두 변수가 비슷한 순서 정보를 가질 수 있음을 뜻하며, 독립적인 불량 요인 두 개를 발견했다는 의미는 아니다.')
display(eda3_strong.round(3))
eda3_say(f'10개 쌍에서 동일 조합의 양품이 존재하는 불량 패턴 수는 '
         f'{eda3_pair_summary.shared_with_normal.min()}–{eda3_pair_summary.shared_with_normal.max()}/{len(eda3_bad_unique)}개다. '
         'RG3는 불량 전체가 충돌 패턴이므로 정확한 좌표 겹침으로는 분리할 수 없다. 구간별 빈도 차이는 다음 표에서 별도로 확인한다.')

In [ ]:
# 조합의 추가 정보: 두 단일 값은 양품에 있지만 두 값의 조합은 없는가?
eda3_increment_rows = []
for x, y in combinations(eda3_top5, 2):
    x_absent = ~eda3_bad_unique[x].isin(eda3_normal[x])
    y_absent = ~eda3_bad_unique[y].isin(eda3_normal[y])
    freq = eda3_normal.groupby([x, y]).size()
    pair_absent = pd.Series(
        [freq.get(tuple(v), 0) == 0 for v in eda3_bad_unique[[x, y]].to_numpy()],
        index=eda3_bad_unique.index)
    additional = pair_absent & ~x_absent & ~y_absent
    eda3_increment_rows.append([
        x, y, int(x_absent.sum()), int(y_absent.sum()), int(pair_absent.sum()),
        int(additional.sum()), eda3_bad_unique.loc[additional, 'source_id'].tolist()])
eda3_pair_increment = pd.DataFrame(eda3_increment_rows, columns=[
    'feature_1', 'feature_2', 'x_alone_absent', 'y_alone_absent',
    'pair_absent', 'additional_over_two_singles', 'additional_representative_IDs'])
display(eda3_pair_increment)


**조합 분석의 실제 결론:** 10개 쌍 모두 불량 25개 패턴 전체가 동일한 두 변수 값을 가진 양품과 겹친다. 단일 값이 양품에 없는 불량도 0개, 두 값을 조합해 추가로 드러나는 불량도 0개다. 이는 A.1의 전체입력 충돌로부터 예상한 결과이며, 조합 탐색으로 새로운 불량 판정 조건을 찾았다고 볼 수 없다.

`Mold_Temperature_3`와 `Mold_Temperature_4`의 Spearman 상관은 0.987이다. 두 온도는 양품 패턴에서 비슷한 순서 정보를 주므로, 두 변수가 상위에 있다는 사실을 서로 독립적인 두 원인의 증거로 보지 않는다. 다만 높은 상관만으로 어느 하나를 삭제하지 않고 모델 검증에 맡긴다.

### 두 변수 구간별 관측 건수: 불량이 집중되는 범위는 좁아지는가?
정확한 값의 겹침만으로는 주변 구간의 관측 빈도를 알 수 없다. 이번에는 양품 원본 중앙값으로 각 쌍을 네 구간으로 나누고, 불량이 포함된 구간의 크기와 양품 수를 확인한다. 단일 변수 구간과 비교해야 조합으로 범위가 얼마나 좁아졌는지 알 수 있으므로 뒤에 비교표를 추가한다.

경계와 같은 값은 `≤ median`에 포함한다. 이산값·동률 때문에 구간 크기는 같지 않을 수 있다. 각 칸은 **불량 행/전체 행, 전체입력 패턴 수**를 표시한다. 중앙값은 탐색용 구분이며 최적 불량 임계값이 아니다. 원본 중복을 유지한 관측 비율로, 독립 표본 기반 확률 추정은 아니다.


In [ ]:
eda3_quadrants = []
fig, axes = plt.subplots(5,2,figsize=(14,21))
for ax,(x,y) in zip(axes.flat, combinations(eda3_top5,2)):
    mx,my = eda3_normal[x].median(),eda3_normal[y].median()
    rates = np.full((2,2),np.nan); labels = np.empty((2,2),dtype=object)
    for iy in range(2):
        for ix in range(2):
            mask = (eda3_raw[x].gt(mx) if ix else eda3_raw[x].le(mx)) & (eda3_raw[y].gt(my) if iy else eda3_raw[y].le(my))
            g=eda3_raw.loc[mask]; total=len(g); bad=int(g.PassOrFail.sum()); patterns=g.pattern_id.nunique()
            rate=bad/total if total else np.nan
            rates[iy,ix]=100*rate
            labels[iy,ix]=f'{bad}/{total} rows\n{patterns} patterns\n{100*rate:.1f}%' if total else 'No observations'
            eda3_quadrants.append([x,y,mx,my,ix,iy,total,bad,patterns,rate])
    sns.heatmap(rates,annot=labels,fmt='',cmap='YlOrRd',vmin=0,vmax=100,cbar=False,ax=ax,
                xticklabels=['<= median','> median'],yticklabels=['<= median','> median'])
    for iy,ix in zip(*np.where(np.isnan(rates))):ax.text(ix+.5,iy+.5,'No observations',ha='center',va='center',fontsize=9)
    ax.set(xlabel=x,ylabel=y,title=f'Normal medians: x={mx:.3f}, y={my:.3f}')
plt.tight_layout();plt.show()
eda3_quadrant_table=pd.DataFrame(eda3_quadrants,columns=['feature_1','feature_2','x_cut','y_cut','x_high','y_high','rows','defects','patterns','observed_defect_fraction'])
display(eda3_quadrant_table)
eda3_say('**해석:** 같은 색의 차이도 분모와 고유 패턴 수를 먼저 확인한다. '
         '불량이 포함된 구간에 양품도 많다면 그 조합은 점검할 조건을 설명하지만 단독 판정 규칙으로는 부족하다. '
         'KS 상위 변수 선정과 이 구간 비교는 같은 데이터를 사용했으므로 독립 검증이 아니다.')

In [ ]:
# 단일 변수 구간과 두 변수 구간을 같은 원본 행 기준으로 비교한다.
eda3_single_regions = []
for c in eda3_top5:
    cut = eda3_normal[c].median()
    for high in [False, True]:
        g = eda3_raw[eda3_raw[c].gt(cut) if high else eda3_raw[c].le(cut)]
        if g.PassOrFail.sum() > 0:
            eda3_single_regions.append([c, '> median' if high else '<= median',
                len(g), int(g.PassOrFail.sum()), g.pattern_id.nunique(),
                g.PassOrFail.mean()])
eda3_single_region_table = pd.DataFrame(eda3_single_regions, columns=[
    'feature', 'region', 'rows', 'defects', 'patterns', 'observed_defect_fraction'])
display(eda3_single_region_table)
display(eda3_quadrant_table.loc[eda3_quadrant_table.defects.gt(0), [
    'feature_1', 'feature_2', 'x_high', 'y_high', 'rows', 'defects', 'patterns',
    'observed_defect_fraction']])


**구간 분석의 실제 결론:** 전체 불량 비율은 25/1,182행(2.12%)이다. 10개 쌍의 중앙값 네 구간 중 가장 높은 관측 비율은 `Barrel_Temperature_5 > median` 및 `Max_Injection_Speed > median`의 **9/218행(4.13%), 109개 패턴**이다. 배럴 온도 5 하나의 높은 구간은 14/448행(3.13%), 최대 사출 속도 하나의 높은 구간은 14/510행(2.75%)다.

두 조건을 결합하면 관측 비율은 높아지지만 포함되는 불량은 14건에서 9건으로 줄어든다. 이 구간을 불량으로 판정하면 양품 209건도 포함하며, 전체 불량 25건 중 16건을 놓친다. 따라서 높은 비율만으로 좋은 판정 규칙이라고 할 수 없다. RG3는 각 상위 변수의 중앙값 양쪽 모두에 불량이 있어, CN7처럼 모든 불량이 한 방향의 구간에 모이는 구조도 아니다.

이 구간은 같은 자료에서 40개 칸을 살펴본 탐색 결과다. 전체 자료 대비 약 1.95배라는 관측 비율 차이는 독립 검증된 위험도나 공정 원인으로 해석하지 않는다. 다음 절에서는 중복 처리 방식에 따라 개별 변수의 분포 차이가 얼마나 달라지는지 확인한다.

## A.3 상위 변수의 개별 분포와 처리 방식별 민감도

**질문:** 기존 상위 변수의 분포 차이는 원본과 라벨별 고유 패턴에서도 유지되는가?

같은 5개 변수에 대해 원본 행, 불량 우선 처리, 라벨별 고유 패턴, 충돌 불량만을 비교한다. 비충돌 불량은 0행이므로 그 집단의 KS는 계산하지 않는다. 충돌 불량만의 비교는 원본 불량 전체와 같은 집단이라는 점을 명시한다.

In [ ]:
eda3_ks_rows=[]
eda3_views={
    'Original rows':(eda3_normal,eda3_bad),
    'Existing defect-priority':(eda3_current[eda3_current.PassOrFail.eq(0)],eda3_current[eda3_current.PassOrFail.eq(1)]),
    'Unique within label':(eda3_normal_unique,eda3_bad_unique),
    'Conflicting defects only':(eda3_normal,eda3_bad[eda3_bad.pattern_type.eq('conflicting')])}
for view,(normal,bad) in eda3_views.items():
    for c in eda3_top5:
        eda3_ks_rows.append([view,c,len(normal),len(bad),ks_2samp(normal[c],bad[c]).statistic,
                             normal[c].median(),bad[c].median()])
eda3_ks_sensitivity=pd.DataFrame(eda3_ks_rows,columns=['view','feature','normal_n','defect_n','KS','normal_median','defect_median'])
display(eda3_ks_sensitivity)
eda3_ks_wide=eda3_ks_sensitivity.pivot(index='feature',columns='view',values='KS').reindex(eda3_top5)
display(eda3_ks_wide.round(3))
assert np.allclose(eda3_ks_wide['Existing defect-priority'],ks_df.set_index('Feature').loc[eda3_top5,'KS_Statistic'])
fig,axes=plt.subplots(3,2,figsize=(15,13))
for ax,c in zip(axes.flat,eda3_top5):
    for label,g,color in [('Normal rows',eda3_normal,'#3274A1'),('All defect rows',eda3_bad,'#555555')]:
        v=np.sort(g[c].to_numpy());ax.step(v,np.arange(1,len(v)+1)/len(v),where='post',label=f'{label} n={len(v)}',color=color)
    for kind,level,marker in [('conflicting',-.07,'o'),('defect_only',-.15,'x')]:
        g=eda3_bad[eda3_bad.pattern_type.eq(kind)]
        ax.scatter(g[c],np.full(len(g),level),color=eda3_colors[kind],marker=marker,label=f'{kind} rows n={len(g)}')
    ax.set(title=c,xlabel='File-standardized value (not physical units)',ylabel='ECDF / defect rug',ylim=(-.22,1.05))
    ax.legend(fontsize=8);ax.grid(alpha=.15)
axes.flat[-1].axis('off');plt.tight_layout();plt.show()

# 불량에서 관측된 실제 값별로 양품 빈도와 고유 전체입력 패턴 수를 표시한다.
eda3_value_rows=[]
for c in eda3_top5:
    for value,b in eda3_bad.groupby(c):
        a=eda3_normal[eda3_normal[c].eq(value)]
        eda3_value_rows.append([c,value,len(a),len(b),a.pattern_id.nunique(),b.pattern_id.nunique(),b.source_id.tolist(),len(b)/(len(a)+len(b))])
eda3_value_table=pd.DataFrame(eda3_value_rows,columns=['feature','value','normal_rows','defect_rows','normal_patterns','defect_patterns','defect_IDs','observed_defect_fraction_at_value'])
display(eda3_value_table)
eda3_say('**원본과 충돌 불량 비교:** RG3는 비충돌 불량이 없어 두 집단이 동일하다. 비충돌 집단의 KS는 계산하지 않는다.')
for c in eda3_top5:
    r = eda3_ks_wide.loc[c]
    eda3_say(f'- {c}: 원본 KS **{r["Original rows"]:.3f}**, '
             f'불량 우선 **{r["Existing defect-priority"]:.3f}**, '
             f'라벨별 고유 패턴 **{r["Unique within label"]:.3f}**.')


**데이터가 말하는 것:** 원본 KS는 배럴 온도 5 0.185, 금형 온도 4 0.165, 호퍼 온도 0.148, 금형 온도 3 0.143, 최대 사출 속도 0.131이다. 불량 우선 처리의 0.189·0.169·0.151·0.146·0.134보다 각각 조금 작다. 해당 처리에서 충돌 양품이 제거되며 분포 차이가 약간 커졌지만, 그것이 입력이 같은 양품·불량을 분리할 수 있게 해주는 것은 아니다.

충돌 불량만의 KS는 원본과 정확히 같다. 이는 특이 사례를 제외해도 경향이 유지되었다는 별도 증거가 아니라, 제외할 비충돌 불량이 없어서 같은 비교를 한 결과다. 불량에서 관측된 모든 개별 값은 양품에도 있고, 각 전체입력 패턴은 양품과 공존한다. 따라서 단일 값별 불량 비율이나 큰 순위 차이를 확정 임계값으로 바꾸지 않는다. 다음 절에서는 공정 단계별 관측 범위도 같은 결론인지 확인한다.

## A.4 공정 단계별 조합 비교

**질문:** 충전·사출·전환·계량·열적 조건에서 불량이 관측 양품 범위를 벗어나는가?

충전시간–최대사출속도, 사출시간–최대사출압력, 전환압력–쿠션위치, 계량위치–배압, 계량시간–위치에 더해 RG3에서 KS가 높은 배럴 온도 5–호퍼 온도를 확인한다. 각 쌍은 공정 해석을 위한 요약이며 인과적 순서나 공정 실패 원인을 증명하지 않는다. `Injection_Time`은 사출 공통 요약값이고, 전환압력과 쿠션 위치만으로 보압 유지 전체를 알 수는 없다. 금형 온도 3·4는 정의의 불확실성을 고려해 A.2의 열적 참고 변수로 남긴다.

In [ ]:
eda3_process_pairs=[
    ('Filling','Filling_Time','Max_Injection_Speed'),
    ('Injection summary','Injection_Time','Max_Injection_Pressure'),
    ('Switch / cushion','Max_Switch_Over_Pressure','Cushion_Position'),
    ('Plasticizing / back pressure','Plasticizing_Position','Max_Back_Pressure'),
    ('Plasticizing time / position','Plasticizing_Time','Plasticizing_Position'),
    ('Thermal context','Barrel_Temperature_5','Hopper_Temperature')]
eda3_process_rows=[]
fig,axes=plt.subplots(3,2,figsize=(16,14))
for ax,(stage,x,y) in zip(axes.flat,eda3_process_pairs):
    eda3_scatter(ax,x,y);ax.set_title(stage)
    freq=eda3_normal.groupby([x,y]).size()
    for _,r in eda3_bad_unique.iterrows():
        eda3_process_rows.append([stage,int(r.source_id),int(r.pattern_id),r.pattern_type,
            int(freq.get((r[x],r[y]),0)),
            bool(r[x]<eda3_normal[x].min() or r[x]>eda3_normal[x].max()),
            bool(r[y]<eda3_normal[y].min() or r[y]>eda3_normal[y].max())])
plt.tight_layout();plt.show()
eda3_process_detail=pd.DataFrame(eda3_process_rows,columns=['stage','representative_id','pattern_id','type','normal_rows_matching_pair','x_outside_observed_normal_range','y_outside_observed_normal_range'])
eda3_process_summary=eda3_process_detail.groupby(['stage','type']).agg(
    defect_patterns=('pattern_id','size'),pair_shared_with_normal=('normal_rows_matching_pair',lambda s:int(s.gt(0).sum())),
    x_outside=('x_outside_observed_normal_range','sum'),y_outside=('y_outside_observed_normal_range','sum'))
display(eda3_process_summary)
eda3_say('**공정별 결과 — 전체 불량 고유 패턴 기준:**')
for stage, _, _ in eda3_process_pairs:
    g = eda3_process_detail[eda3_process_detail.stage.eq(stage)]
    eda3_say(f'- {stage}: {len(g)}개 중 {g.normal_rows_matching_pair.gt(0).sum()}개는 '
             f'같은 조합의 양품이 있다. x 관측 양품 범위 밖 {g.x_outside_observed_normal_range.sum()}개, '
             f'y 범위 밖 {g.y_outside_observed_normal_range.sum()}개.')


**공정 비교의 해석:** 6개 공정 쌍 모두 불량 25개 패턴이 같은 조합의 양품과 겹치며, 각 축에서 관측 양품 범위를 벗어나는 불량은 0개다. RG3에는 현재 값으로 식별되는 비충돌 특이 불량이 없다. 따라서 관측 양품 범위를 이용한 단순 이상값 규칙으로 불량을 찾는 접근에는 근거가 부족하다.

이는 공정에 원인이 없다는 뜻이 아니다. 현재 측정값과 해상도로는 해당 실패를 특정할 정보가 충분하지 않다는 뜻이다. 패턴 빈도에 따른 예측 가능성은 별도 검증할 수 있지만, 로트·시간·검수·측정 과정 등 추가 정보 없이 어느 조건이 실패를 일으켰다고 확정할 수 없다. 원본 라벨을 보존하고 관측된 데이터의 한계를 전처리 명세에 기록한다.

In [ ]:
plt.close("all")


# 공정 기반 비교·탐색적 파생변수

원본 분석: `EDA/molding_process_eda.ipynb`. 모듈화 전 원본은 복구 ZIP에 보관되어 있습니다.


# 사출성형 공정 기반 EDA와 파생변수 탐색
**질문:** 불량은 정상과 비교할 때 충전·사출 공통·보압 전환·보압 관련 관측값에서 어떻게 다른가?

CN7과 RG3를 분리한다. 원본 보존, 정확 입력 패턴 관점, 문서 근거와 추론의 분리를 원칙으로 한다. 예측 모델 및 pseudo-labeling은 수행하지 않는다. 그래프는 폰트 호환성을 위해 영문, 해석은 한국어로 제공한다. 아래 첫 코드 셀의 두 경로를 수정하면 ZIP 또는 CSV 폴더에서 재실행할 수 있다.

필요 패키지: `numpy pandas scipy matplotlib seaborn pymupdf ipython` (Jupyter 실행에는 `ipykernel` 추가). 예: `%pip install numpy pandas scipy matplotlib seaborn pymupdf ipykernel`. 패키지 설치는 분석 셀에서 자동 수행하지 않는다. 원본 4개 CSV는 ZIP에서 읽으며 노트북에 원시 데이터를 재저장하지 않는다.

박사님 의견: 정상과 비정상의 레코드가 동일한 경우 정상 레코드 삭제 -> 보수적으로 설정

In [ ]:
from pathlib import Path
DATA_PATH = PROJECT_ROOT/'data/origin'
GUIDE_PATH = PROJECT_ROOT/'data/origin/04. Guidebook_Molding-압축됨.pdf'
SEED = 20260922
REPEATS = 12  # 계산 예산: 6회와 12회 요약을 비교하며 수렴을 확정하지 않는다.
IQR_COEFFICIENTS = [1.5, 3.0, 4.5]  # 탐색 민감도 범위; 규격/불량 판정값이 아니다.


In [ ]:
import io, zipfile, hashlib, platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fitz
from scipy.stats import ks_2samp, spearmanr, skew
from scipy.spatial.distance import cdist
from IPython.display import display, Markdown, Image


rng = np.random.default_rng(SEED)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 16)
plt.rcParams.update({'font.family':'DejaVu Sans','figure.dpi':110})
COLORS = {0:'#2471A3', 1:'#D35400'}
display(pd.DataFrame({'package':['python','numpy','pandas','scipy','matplotlib','seaborn','PyMuPDF'],
'version':[platform.python_version()]+[importlib.metadata.version(x) for x in ['numpy','pandas','scipy','matplotlib','seaborn','PyMuPDF']]}))
def say(s): display(Markdown(s))
def ecdf(ax, values, label, color):
    a=np.sort(pd.Series(values).dropna().to_numpy()); ax.step(a,np.arange(1,len(a)+1)/max(len(a),1),where='post',label=f'{label} (n={len(a)})',color=color)
def finish(ax,title,x,y):
    ax.set(title=title,xlabel=x,ylabel=y); ax.legend(fontsize=7)

def safe_corr(a,b):
    a=np.asarray(a);b=np.asarray(b)
    return float(spearmanr(a,b).statistic) if np.unique(a).size>1 and np.unique(b).size>1 else np.nan
def finite_median(a):
    a=np.asarray(a,float);a=a[np.isfinite(a)]
    return float(np.median(a)) if len(a) else np.nan


## 1. 가이드북 확인과 관측 범위
**방법·근거:** PDF의 물리 페이지와 인쇄 쪽수는 다르다. 이 파일에서는 물리 페이지=인쇄 쪽수+1이다. 아래에서 해당 페이지의 본문과 코드 이미지를 직접 출력한다. 다른 PDF로 교체할 경우 페이지 대응을 확인해야 한다. 공정 존재가 해당 공정을 완전히 관측한다는 뜻은 아니다.

In [ ]:
if not GUIDE_PATH.is_file(): raise FileNotFoundError(f'가이드북 없음: {GUIDE_PATH}')
guide=fitz.open(GUIDE_PATH)
for printed in [5,6,7,20,21]:
    say(f'**가이드북 인쇄 {printed}쪽 (PDF {printed+1}쪽)**')
    print(guide[printed].get_text()[:14000])
for printed in [43,62,69]:
    say(f'**가이드북 인쇄 {printed}쪽: 코드/표 이미지 원문**')
    display(Image(data=guide[printed].get_pixmap(matrix=fitz.Matrix(1.25,1.25)).tobytes('png')))


In [ ]:
FEATURES = ['Injection_Time','Filling_Time','Plasticizing_Time','Cycle_Time','Clamp_Close_Time','Cushion_Position','Plasticizing_Position','Clamp_Open_Position','Max_Injection_Speed','Max_Screw_RPM','Average_Screw_RPM','Max_Injection_Pressure','Max_Switch_Over_Pressure','Max_Back_Pressure','Average_Back_Pressure']+[f'Barrel_Temperature_{i}' for i in range(1,7)]+['Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4']

GROUPS = {'Filling':['Filling_Time','Max_Injection_Speed'], 'Injection shared':['Injection_Time','Max_Injection_Pressure'], 'Switch':['Max_Switch_Over_Pressure'], 'Cushion':['Cushion_Position'], 'Plasticizing':['Plasticizing_Time','Plasticizing_Position','Max_Back_Pressure','Average_Back_Pressure']}

THERMAL=[f'Barrel_Temperature_{i}' for i in range(1,7)]+['Hopper_Temperature']

defs=['고압+사출시간','충전시간','1회 생산량 수지의 용융·계량시간','제품 생산 사이클 시간','금형 닫힘 시간','보압을 하기 위한 스크류 위치','계량 완료 스크류 위치','금형 열림 위치','금형 내 수지 최대 유입속도','사출을 위한 스크류 최대속도','사출을 위한 스크류 평균속도','금형으로 주입하는 수지 최대압력','사출에서 보압 전환 압력','계량 중 스크류 후퇴 저항 최대압력','계량 중 스크류 후퇴 저항 평균압력']+['계량·사출 중 수지 용융 유지 온도']*6+['재료 투입구 온도','재료 투입구 온도로 기재됨','재료 투입구 온도로 기재됨']
units=['s']*5+['mm']*3+['mm/s']*3+['MPa']*4+['°C']*9
dictionary=[]
for c,d,u in zip(FEATURES,defs,units):
    g=next((g for g,cs in GROUPS.items() if c in cs), '열적 조건' if c in THERMAL else '의미 확인 필요' if 'RPM' in c or 'Mold_Temperature' in c else '전체 생산시간' if c=='Cycle_Time' else '금형 닫힘' if c=='Clamp_Close_Time' else '금형 열림')
    dictionary.append([c,d,u,g,20 if FEATURES.index(c)<15 else 21,'이름·설명·단위 불일치' if 'RPM' in c else '금형 이름과 투입구 설명 불일치' if 'Mold_Temperature' in c else '정의 요약; 원문은 위 페이지 확인','파일별 표준화 척도(원시 단위 아님)'])
variable_dictionary=pd.DataFrame(dictionary,columns=['컬럼','가이드북 정의 요약','원래 단위','분석 구분','인쇄 쪽수','미확인 사항','현재 척도'])
display(variable_dictionary)
say('충전 → 가압 → 보압 순서로 해석한다. Injection_Time과 Max_Injection_Pressure는 사출 공통 지표이며 가압 전용 측정값이 아니다. Cushion_Position은 보압 관련 위치다. 전환압력과 쿠션만으로 보압 시간·압력 유지 전체를 측정할 수 없다. RPM과 금형 온도는 확정적인 단계 증거로 사용하지 않는다. 69쪽 코드는 0=정상, 1=불량 및 파일별·컬럼별 StandardScaler 적용을 보여준다.')


## 2. 입력 검증·원본 보존·데이터 구조
**방법·근거:** 파일명으로 검색하며 중복 파일명은 임의 선택하지 않는다. 공정값은 반올림 없이 정확 비교한다. source_id는 출처일 뿐 시간이나 생산 사이클이 아니다. 결측/무한대가 발견되면 자동 대치하지 않고 진단 후 중단한다.

In [ ]:
NAMES=[f'moldset_{kind}_{p}.csv' for kind in ['labeled','unlabeled'] for p in ['cn7','rg3']]
raw={}; data={}
if not DATA_PATH.exists(): raise FileNotFoundError(f'데이터 없음: {DATA_PATH}')
for name in NAMES:
    if DATA_PATH.is_dir():
        matches=list(DATA_PATH.rglob(name))
        if len(matches)!=1: raise ValueError(f'{name}: 발견 {len(matches)}개; 정확히 하나 필요')
        payload=matches[0].read_bytes()
    else:
        with zipfile.ZipFile(DATA_PATH) as z:
            matches=[x for x in z.namelist() if Path(x).name==name]
            if len(matches)!=1: raise ValueError(f'{name}: ZIP 안 발견 {len(matches)}개')
            payload=z.read(matches[0])
    df=pd.read_csv(io.BytesIO(payload)); raw[name]=df.copy(deep=True)
    required=FEATURES+(['PassOrFail'] if '_labeled_' in name else [])
    missing=set(required)-set(df)
    if missing: raise ValueError(f'{name}: 필수 컬럼 누락 {sorted(missing)}')
    if 'Unnamed: 0' not in df: raise ValueError(f'{name}: 원본 ID 컬럼 없음')
    df=df.rename(columns={'Unnamed: 0':'source_id'}).copy()
    df['product']=name.split('_')[-1].split('.')[0].upper(); df['source_file']=name; df['record_order']=np.arange(len(df))
    if 'PassOrFail' in df and not set(df.PassOrFail.unique())<={0,1}: raise ValueError('0/1 이외 라벨')
    data[name]=df
quality=[]
for name,d in data.items():
    a=d[FEATURES]
    quality.append([name,len(d),len(raw[name].columns),a.isna().sum().sum(),np.isinf(a.to_numpy()).sum(),list(a.columns[a.nunique()==1]),hashlib.sha256(raw[name].to_csv(index=False).encode()).hexdigest()])
display(pd.DataFrame(quality,columns=['파일','행','원본 열','결측','무한대','상수열','내용 SHA256']))
display(pd.concat({name:d[FEATURES].agg(['count','nunique']).T.assign(dtype=d[FEATURES].dtypes.astype(str)) for name,d in data.items()},names=['파일','변수']))
assert all(q[3]==0 and q[4]==0 for q in quality),'비유한값 처리 방침을 먼저 결정해야 함'
L={p:data[f'moldset_labeled_{p.lower()}.csv'].copy() for p in ['CN7','RG3']}
U={p:data[f'moldset_unlabeled_{p.lower()}.csv'].copy() for p in L}
PAT={}; NORMAL={}; BAD={}
for p,d in L.items():
    d['pattern_id']=pd.MultiIndex.from_frame(d[FEATURES]).factorize(sort=False)[0]
    PAT[p]=d.groupby('pattern_id').PassOrFail.agg(rows='size',bad='sum'); PAT[p]['normal']=PAT[p].rows-PAT[p].bad
    PAT[p]['type']=np.select([PAT[p].bad.eq(0),PAT[p].normal.eq(0)],['normal only','defect only'],default='conflicting')
    NORMAL[p]=d[d.PassOrFail.eq(0)].drop_duplicates(FEATURES); BAD[p]=d[d.PassOrFail.eq(1)].drop_duplicates(FEATURES)
    say(f'**{p}: 행 기준 {len(d)}개, 정상 {(d.PassOrFail==0).sum()}개, 불량 {d.PassOrFail.sum()}개 ({d.PassOrFail.mean():.2%}); 공정 패턴 {len(PAT[p])}개**')
    display(PAT[p].groupby('type').agg(patterns=('rows','size'),rows=('rows','sum'),normal=('normal','sum'),bad=('bad','sum')))
    display(pd.DataFrame({'중복 정의':['ID 제외·라벨 포함','ID·라벨 제외'], '중복 초과 행':[d.duplicated(FEATURES+['PassOrFail']).sum(),d.duplicated(FEATURES).sum()]}))
for p,d in U.items():
    say(f'{p} 비라벨: 전체 공정값 고유 패턴 {len(d.drop_duplicates(FEATURES)):,}, 중복 초과 행 {d.duplicated(FEATURES).sum():,}.')


In [ ]:
conflict_pairs=[]
for p,d in L.items():
    for pid,g in d.groupby('pattern_id'):
        for _,b in g[g.PassOrFail.eq(1)].iterrows():
            for _,a in g[g.PassOrFail.eq(0)].iterrows():
                conflict_pairs.append([p,pid,a.source_id,b.source_id,abs(b.source_id-a.source_id),abs(b.record_order-a.record_order)])
conflict_pairs=pd.DataFrame(conflict_pairs,columns=['product','pattern_id','normal_source_id','bad_source_id','ID_gap','row_gap'])
display(conflict_pairs)
fig,axs=plt.subplots(2,3,figsize=(15,8))
for i,(p,d) in enumerate(L.items()):
    counts=d.PassOrFail.value_counts().reindex([0,1],fill_value=0)
    axs[i,0].bar(['Normal','Defect'],counts,color=list(COLORS.values()),label=f'Rows n={len(d)}')
    for j,c in enumerate(counts): axs[i,0].text(j,c,f'{c} ({c/len(d):.1%})',ha='center')
    finish(axs[i,0],p+' labels','Label','Rows')
    PAT[p].groupby(['rows','type']).size().unstack(fill_value=0).plot.bar(ax=axs[i,1]);finish(axs[i,1],f'{p} patterns n={len(PAT[p])}','Pattern size','Patterns')
    b=d[d.PassOrFail.eq(1)]; shared=b.pattern_id.isin(NORMAL[p].pattern_id)
    axs[i,2].bar(['Exact normal match','No exact match'],[shared.sum(),(~shared).sum()],label=f'Defect rows n={len(b)}')
    finish(axs[i,2],p+' conflicting inputs','Full input match','Defect rows')
plt.tight_layout();plt.show()
fig,axs=plt.subplots(1,2,figsize=(16,4))
for ax,p in zip(axs,L):
    pair=conflict_pairs[conflict_pairs['product'].eq(p)].iloc[0]; d=L[p]
    for label,id_ in [(0,pair.normal_source_id),(1,pair.bad_source_id)]:
        row=d[d.source_id.eq(id_)].iloc[0]
        ax.plot(range(len(FEATURES)),row[FEATURES].astype(float),marker='o' if label==0 else 'x',label=f'{label}: ID {id_} (n=1)',color=COLORS[label],alpha=.7)
    ax.set_xticks(range(len(FEATURES)),FEATURES,rotation=90,fontsize=6);finish(ax,p+' identical inputs','Feature','File standardized value')
plt.tight_layout();plt.show()
say('행 기준은 기록 빈도, 고유 패턴 기준은 서로 다른 입력 조합을 나타낸다. 상충 패턴은 정상·불량 고유집합 양쪽에 포함되므로 두 집합 크기의 합은 전체 고유 패턴 수와 다를 수 있다. 라벨 수정·중복 삭제는 원본에 적용하지 않는다. ID 간격 1은 동일 사이클이나 좌우 제품 대응의 증거가 아니다.')


## 3. 기본 분포와 극단값
**방법·설정:** 분위수는 분포 요약이며 판정값이 아니다. 행/라벨별 고유 입력 두 관점을 비교한다. IQR 계수 1.5는 관행적 박스플롯 기준, 3은 이전 분석 재현, 4.5는 더 느슨한 탐색 범위다. 어느 것도 제조 규격이 아니다. 0-IQR은 비상수 이산열에서도 발생하므로 별도 표기한다. 봉우리 수는 히스토그램 분할에 민감하여 자동 확정하지 않는다.

In [ ]:
distribution=[]; extreme=[]
for p,d in L.items():
    for view,t in [('rows',d),('unique within label',d.drop_duplicates(FEATURES+['PassOrFail']))]:
        for label,g in t.groupby('PassOrFail'):
            q=g[FEATURES].describe(percentiles=[.05,.25,.5,.75,.95]).T
            q['product']=p;q['view']=view;q['label']=label;distribution.append(q)
    ref=d[d.PassOrFail.eq(0)][FEATURES]; q1=ref.quantile(.25);q3=ref.quantile(.75);iq=q3-q1
    diag=pd.DataFrame({'nunique':ref.nunique(),'skew':ref.skew(),'IQR':iq,'constant':ref.nunique().eq(1),'IQR_zero_nonconstant':iq.eq(0)&ref.nunique().gt(1)})
    say(f'**{p} 정상 행 분포 진단**');display(diag)
    for coef in IQR_COEFFICIENTS:
        flag=((d[FEATURES]<q1-coef*iq)|(d[FEATURES]>q3+coef*iq)) & iq.gt(0)
        for label in [0,1]:
            mask=d.PassOrFail.eq(label); extreme.append([p,coef,label,int(mask.sum()),int(flag.loc[mask].any(axis=1).sum()),d.loc[mask&flag.any(axis=1),'source_id'].tolist()])
distribution_table=pd.concat(distribution);display(distribution_table)
extreme_table=pd.DataFrame(extreme,columns=['product','IQR coefficient','label','row denominator','flagged rows','source IDs']);display(extreme_table)
for p,d in L.items():
    fig,axs=plt.subplots(6,4,figsize=(18,20))
    for ax,c in zip(axs.flat,FEATURES):
        for label in [0,1]:
            g=d[d.PassOrFail.eq(label)][c];ax.boxplot(g,positions=[label],widths=.5,showfliers=False)
            vals=g if label else g.sample(min(200,len(g)),random_state=SEED)
            ax.scatter(np.full(len(vals),label)+np.linspace(-.15,.15,len(vals)),vals,s=8,color=COLORS[label],alpha=.45,label=f'{label} n={len(g)}; dots={len(vals)}')
        ax.set_xticks([0,1],['Normal','Defect']);finish(ax,p+' '+c,'Label','Standardized value')
    plt.tight_layout();plt.show()
    fig,axs=plt.subplots(2,3,figsize=(16,8))
    for ax,c in zip(axs.flat,['Max_Injection_Speed','Cushion_Position','Filling_Time','Max_Switch_Over_Pressure','Average_Back_Pressure','Cycle_Time']):
        for label in [0,1]:
            g=d[d.PassOrFail.eq(label)];ecdf(ax,g[c],f'{label} rows',COLORS[label])
            a=np.sort(g.drop_duplicates(FEATURES)[c]);ax.step(a,np.arange(1,len(a)+1)/len(a),ls='--',color=COLORS[label],label=f'{label} unique n={len(a)}')
        finish(ax,p+' '+c,c,'ECDF')
    plt.tight_layout();plt.show()
say('불량 표본은 적고 반복 기록이 있다. 박스플롯만으로 꼬리 사례를 숨기지 않도록 불량 점을 모두 표시했다. 정상 점 최대 200개는 렌더링 예산이며 통계 계산은 전체 행이다. IQR=0 열은 이 규칙의 표시에서 제외했지만 실제 값 분포와 정확 조합 분석에서 계속 확인한다. 관측 정상 최솟값·최댓값은 공정 허용 한계가 아니다.')


### 분포 모양에 따른 극단값 규칙의 한계
**질문 → 방법:** 이산 수준과 히스토그램 분할을 바꿔도 보이는 봉우리를 확인한다. FD는 IQR과 표본수, Sturges는 로그 표본수에 근거한 시각화 분할 규칙이며 정상/불량 임계값이 아니다. 자동 봉우리 수를 선택하지 않는다. 아래 수준별 집계의 분모는 행이며, 열별 고유값 수가 적어도 공정 전체 입력은 다를 수 있다.

In [ ]:
for p,d in L.items():
    fig,axs=plt.subplots(1,3,figsize=(16,4))
    for ax,c in zip(axs,['Max_Injection_Speed','Cushion_Position','Average_Back_Pressure']):
        normal=d.loc[d.PassOrFail.eq(0),c]
        for rule,style in [('fd','-'),('sturges','--')]:
            ax.hist(normal,bins=rule,density=True,histtype='step',linestyle=style,color=COLORS[0],label=f'Normal {rule} n={len(normal)}')
        bad=d.loc[d.PassOrFail.eq(1),c];ax.scatter(bad,np.zeros(len(bad)),color=COLORS[1],marker='x',label=f'Defect n={len(bad)}')
        finish(ax,p+' shape sensitivity',c,'Density')
    plt.tight_layout();plt.show()
    display(d.groupby('Cushion_Position').PassOrFail.agg(row_denominator='size',bad_rows='sum',bad_fraction='mean'))
say('쿠션처럼 적은 수준에 관측이 집중된 변수는 히스토그램 모양을 연속 밀도로 과해석하지 않는다. 극단값의 삭제보다 정상·불량이 공유하는 실제 수준과 공정 조합을 확인하는 것이 우선이다.')

## 4. 라벨·비라벨 비교
**방법·근거:** 모집단 표준편차(ddof=0)는 StandardScaler 결과와 대조하기 위함이다. KS는 분포 차이의 크기만 요약한다. 반복·이산 관측 때문에 독립 연속표본 가정의 p값을 유의성 판정으로 사용하지 않는다. 평균 0/표준편차 1은 코드와 일치하는 정황이지 원시 공정의 동등함이 아니다.

In [ ]:
shift=[]
for p in L:
    for c in FEATURES:
        a=L[p][c];b=U[p][c]
        shift.append([p,c,a.mean(),a.std(ddof=0),a.nunique(),b.mean(),b.std(ddof=0),b.nunique(),ks_2samp(a,b).statistic,((b<a.min())|(b>a.max())).mean()])
shift_table=pd.DataFrame(shift,columns=['product','feature','L mean','L std0','L unique','U mean','U std0','U unique','KS','U outside L range']);display(shift_table)
for p in L:
    fig,axs=plt.subplots(2,3,figsize=(16,8))
    for j,c in enumerate(['Clamp_Open_Position','Max_Injection_Speed','Cushion_Position']):
        for name,d,color in [('Labeled',L[p],'#2471A3'),('Unlabeled',U[p],'#7D3C98')]:
            ecdf(axs[0,j],d[c],name,color)
            axs[1,j].hist(d[c],bins='auto',density=True,alpha=.45,color=color,label=f'{name} n={len(d)}')
        finish(axs[0,j],p+' '+c,c,'ECDF');finish(axs[1,j],p+' '+c,c,'Density')
    plt.tight_layout();plt.show()
say('69쪽 코드에서 네 파일을 각각 표준화한다. 따라서 같은 표준화 숫자는 동일 원시 공정값을 뜻하지 않는다. Clamp_Open_Position은 라벨에서는 상수인데 비라벨에서는 변한다. 비라벨을 정상으로 취급하거나 이 차이를 공정 드리프트로 단정할 수 없다. 라벨 정상 참조로 만든 아래 변환을 비라벨에 직접 적용하지 않는다. 원시값과 파일별 평균·표준편차가 필요하다.')


## 5. 공정별 직접 비교
**방법·근거:** 반올림 없는 정확 일치와 관측 정상 범위를 먼저 확인한다. 개별값 일치, 단계 조합 일치, 전체24변수 일치를 구분한다. 조건 추가에 따른 빈도 감소는 기계적으로 발생하며 그 자체는 이상 증거가 아니다.

In [ ]:
direct=[]; individual=[]
for p,d in L.items():
    normals=d[d.PassOrFail.eq(0)]; unique=NORMAL[p]
    for _,b in d[d.PassOrFail.eq(1)].iterrows():
        outside=[c for c in FEATURES if b[c]<normals[c].min() or b[c]>normals[c].max()]
        for c in FEATURES:
            individual.append([p,b.source_id,c,int(normals[c].eq(b[c]).sum()),c in outside])
        for group,cs in {**GROUPS,'Full input':FEATURES,'Filling + cushion':GROUPS['Filling']+['Cushion_Position'],'Speed + cushion + back':['Max_Injection_Speed','Cushion_Position','Average_Back_Pressure']}.items():
            direct.append([p,b.source_id,b.pattern_id,group,int(normals[cs].eq(b[cs]).all(axis=1).sum()),int(unique[cs].eq(b[cs]).all(axis=1).sum()),', '.join(outside)])
direct_table=pd.DataFrame(direct,columns=['product','source_id','pattern_id','group','normal rows matched','normal unique matched','outside observed normal range'])
individual_table=pd.DataFrame(individual,columns=['product','source_id','feature','normal rows equal','outside normal range'])
display(direct_table);display(individual_table[individual_table['normal rows equal'].eq(0)])
for p,d in L.items():
    fig,axs=plt.subplots(2,3,figsize=(17,9))
    for ax,(x,y) in zip(axs.flat,[('Filling_Time','Max_Injection_Speed'),('Injection_Time','Max_Injection_Pressure'),('Max_Injection_Speed','Cushion_Position')]):
        for label in [0,1]:
            g=d[d.PassOrFail.eq(label)];ax.scatter(g[x],g[y],c=COLORS[label],alpha=.35 if label==0 else .9,s=15 if label==0 else 45,marker='o' if label==0 else 'x',label=f'{label} rows n={len(g)}')
        finish(ax,p+' joint process values',x,y)
    for ax,c in zip(axs.flat[3:5],['Max_Switch_Over_Pressure','Cushion_Position']):
        for label in [0,1]:ecdf(ax,d[d.PassOrFail.eq(label)][c],str(label),COLORS[label])
        finish(ax,p+' '+c,c,'ECDF')
    ax=axs.flat[5];g=d[d.PassOrFail.eq(1)];cs=sum([GROUPS[x] for x in ['Filling','Injection shared','Switch','Cushion','Plasticizing']],[])
    sns.heatmap(g.set_index('source_id')[cs],ax=ax,cmap='vlag',center=0,cbar_kws={'label':'File standardized value'})
    ax.set(title=f'{p} defect profiles n={len(g)}',xlabel='Process-ordered features',ylabel='Source ID (not time)')
    plt.tight_layout();plt.show()
for p in L:
    t=direct_table[direct_table['product'].eq(p)].drop_duplicates(['pattern_id','group'])
    display(t.groupby('group')['normal unique matched'].agg(patterns='size',with_match=lambda x:int((x>0).sum()),minimum='min',maximum='max'))
say('CN7에는 같은 전체 정상 입력이 없는 불량 패턴이 존재하지만 RG3 불량은 전체 입력이 같은 정상을 가진다. 보압 전환압력의 정상 일치는 쿠션·충전 조합의 일치와 별개다. 프로파일의 표준화 크기를 서로 다른 물리 단위의 효과 크기로 해석하지 않는다.')


## 6. 거리 보조 분석: 설정의 안정성과 국소성
**목적·가정:** 정확 일치로 설명하기 어려운 2변수 충전·사출 공통 조합에만 적용한다. 단일 전환/쿠션에 독립 다변량 단계 점수를 만들지 않는다.

후보는 표준편차 스케일 Euclidean, IQR 스케일 Euclidean, IQR Manhattan이다. IQR=0은 참조 표준편차로 대체하며 상수는 스케일 1로 둔다. 상관계수가 큰 경우 유클리드 거리의 정보 중복 가능성을 보고한다. 정상 고유 입력 25%를 공통 점검용으로 고정하고 나머지의 80%를 반복 참조한다. 이 비율은 성능 최적화가 아닌 참조 변동 실험의 계산 설계다.

k=1은 가장 가까운 관측 패턴이라는 직접 의미가 있다. 2부터 √참조크기까지 로그 간격 후보는 단일 사례에서 더 넓은 국소범위로 확장하는 탐색 범위다. k 선택은 불량 분리가 아닌 정상 점검점의 점수 변동·순위 안정성·k번째 이웃 거리 확장으로 검토한다. 12회는 계산 예산이며 6회 요약과 비교한다. 최적 k나 정상/이상 임계값은 선언하지 않는다.

In [ ]:
def distance_matrix(ref,query,cols,mode='iqr_euclidean'):
    r=ref[cols].to_numpy(float);q=query[cols].to_numpy(float)
    if mode=='std_euclidean': scale=np.std(r,axis=0)
    else: scale=np.quantile(r,.75,axis=0)-np.quantile(r,.25,axis=0)
    fallback=np.std(r,axis=0);scale=np.where(scale>0,scale,np.where(fallback>0,fallback,1.))
    return cdist(q/scale,r/scale,metric='cityblock' if 'manhattan' in mode else 'euclidean')
stability=[]; distance_runs={}
for p in L:
    normal=NORMAL[p]; perm=np.random.default_rng(SEED).permutation(len(normal)); nq=max(2,len(normal)//4)
    query=normal.iloc[perm[:nq]]; pool=normal.iloc[perm[nq:]];nr=int(.8*len(pool))
    ks=sorted(set([1]+np.geomspace(2,max(2,int(np.sqrt(nr))),5).round().astype(int).tolist()))
    for group in ['Filling','Injection shared']:
        cols=GROUPS[group];say(f'{p} {group}: 정상 고유 {len(normal)}, 점검 {nq}, 반복 참조 {nr}, k={ks}; 상관={normal[cols].corr().iloc[0,1]:.3f}')
        for mode in ['std_euclidean','iqr_euclidean','iqr_manhattan']:
            scores=[];radii=[];ties=[]
            for rep in range(REPEATS):
                ref=pool.sample(nr,random_state=SEED+rep)
                assert not set(query.pattern_id)&set(ref.pattern_id)
                ds=np.sort(distance_matrix(ref,query,cols,mode),axis=1)
                scores.append(np.column_stack([ds[:,:k].mean(axis=1) for k in ks]));radii.append(np.column_stack([ds[:,k-1] for k in ks]))
                ties.append(np.mean(np.isclose(ds[:,0],ds[:,1])))
            a=np.array(scores); rr=np.array(radii);distance_runs[p,group,mode]=(ks,a,rr)
            for j,k in enumerate(ks):
                corr=[safe_corr(a[x,:,j],a[y,:,j]) for x in range(REPEATS) for y in range(x)]
                corr6=[safe_corr(a[x,:,j],a[y,:,j]) for x in range(min(6,REPEATS)) for y in range(x)]
                stability.append([p,group,mode,k,finite_median(corr),finite_median(corr6),np.median(a[:,:,j].std(axis=0)),np.median(rr[:,:,j]),np.mean(ties),int(np.isfinite(corr).sum()),len(corr)])
stability_table=pd.DataFrame(stability,columns=['product','group','geometry','k','rank stability 12','rank stability 6','score SD median','radius median','nearest tie rate','valid rank comparisons','total rank comparisons']);display(stability_table)
fig,axs=plt.subplots(2,2,figsize=(14,9))
for i,p in enumerate(L):
    for group in ['Filling','Injection shared']:
        for mode in ['std_euclidean','iqr_euclidean','iqr_manhattan']:
            t=stability_table.query('product==@p and group==@group and geometry==@mode')
            label=f'{group}/{mode} (normal unique n={len(NORMAL[p])})'
            axs[i,0].plot(t.k,t['rank stability 12'],marker='o',label=label);axs[i,1].plot(t.k,t['radius median'],marker='o',label=label)
    finish(axs[i,0],p+' reference sensitivity','Neighbors k','Median Spearman');finish(axs[i,1],p+' neighborhood expansion','Neighbors k','Median radius (geometry-specific)')
plt.tight_layout();plt.show()
say('순위 안정성이 좋아져도 이웃 반경 확대는 국소성 감소를 뜻한다. 서로 다른 거리 척도의 반경 절댓값을 직접 비교하지 않는다. 위 정상 점검 실험은 설정 민감도 진단이며 결함 탐지 성능 검증이 아니다. 후속 파생변수는 해석이 명확한 최근접 거리(k=1)를 설명용으로 두고 k 변화는 계속 병기한다.')

say('동일한 단계 조합이 반복되어 점수가 상수이면 순위 상관은 정의되지 않는다. valid rank comparisons가 적거나 0인 후보는 높은 유효 상관값만으로 안정적이라고 판단하지 않는다. 전체 입력 패턴은 분리했지만 단계 부분조합은 양쪽에 존재할 수 있다.')


## 7. CN7 사례와 보조 조건
**방법:** 모든 불량을 계산한 뒤 요청한 ID를 표시한다. 온도 비교는 정의가 확인된 Barrel1–6·Hopper를 기준으로 하고 Mold3–4 포함 여부를 비교한다. m 후보는 정상 고유 수의 제곱근 전후(절반·1배·2배)로 국소 비교군 폭을 탐색한다. 정상의 leave-one-pattern-out 온도 m번째 거리 분포와 불량의 거리를 비교한다. 이는 정상 규격이 아니라 비교군 적합성의 상대 지표다. 사출/배압 차이가 커지는 m을 선택하지 않는다.

상충 불량에는 동일한 정상 전체입력이 온도 비교군에 포함될 수 있어 반경이 작아질 수 있다. 이 포함 결과를 우선 제시한 것이며, 정상의 자기제외 반경과 완전히 동일한 평가 조건은 아니다. 해당 상대위치를 독립적인 적합성 검증이나 불량확률로 사용하지 않는다.

In [ ]:
ids=[95,103,107,113,115,116,117,118,119,120]
display(L['CN7'].loc[L['CN7'].source_id.isin(ids),['source_id','pattern_id','PassOrFail']+sum(GROUPS.values(),[])+['Cycle_Time']])
display(direct_table[(direct_table['product']=='CN7')&direct_table.source_id.isin(ids)])
conditional=[]
for p in L:
    ref=NORMAL[p];bad=BAD[p];ms=sorted(set(max(2,int(np.sqrt(len(ref))*f)) for f in [.5,1,2]))
    for include_mold in [False,True]:
        cs=THERMAL+(['Mold_Temperature_3','Mold_Temperature_4'] if include_mold else [])
        dd=distance_matrix(ref,bad,cs);nn=distance_matrix(ref,ref,cs);np.fill_diagonal(nn,np.inf)
        for m in ms:
            nr=np.sort(nn,axis=1)[:,m-1]
            for j,(_,b) in enumerate(bad.iterrows()):
                order=np.argsort(dd[j],kind='stable')[:m];g=ref.iloc[order];radius=dd[j,order[-1]]
                conditional.append([p,b.source_id,include_mold,m,radius,np.mean(nr<=radius),int(np.isclose(dd[j],radius).sum()),b.Max_Injection_Speed-g.Max_Injection_Speed.median(),b.Cushion_Position-g.Cushion_Position.median(),b.Average_Back_Pressure-g.Average_Back_Pressure.median()])
conditional_table=pd.DataFrame(conditional,columns=['product','source_id','include ambiguous mold','m','temperature radius','normal radius percentile','boundary tie count','speed median contrast','cushion median contrast','back median contrast']);display(conditional_table)
fig,axs=plt.subplots(1,3,figsize=(16,4))
t=conditional_table.query("product=='CN7' and `include ambiguous mold`==False")
for ax,c in zip(axs,['temperature radius','normal radius percentile','back median contrast']):
    for sid,g in t.groupby('source_id'): ax.plot(g.m,g[c],marker='o',label=f'ID {sid} (n={len(g)})')
    finish(ax,'CN7 temperature matching', 'Normal comparison size m',c)
plt.tight_layout();plt.show()
say('m 증가에 따른 온도 반경과 차이 변동을 함께 읽어야 한다. 정상 반경 분포 바깥이면 비교 가능한 정상 부족의 정황으로 기록하며, 안쪽이라고 공정 조건 동등성을 확정하지 않는다. 동률 경계에서는 안정 정렬 순서가 비교군 구성을 바꿀 수 있다. 따라서 조건부 지표는 이번 파생변수 입력 후보에서 보류한다. 속도·쿠션·배압의 동반 차이는 독립 영향이나 이상 전파를 증명하지 않는다.')


### 보조 조건 및 온도 비교군의 실제 점검 결과
**방법:** 계량·배압·사이클·금형 동작·불명확 RPM/금형온도를 불량 개별 점과 함께 비교한다. 모호한 측정값은 별도 열에 유지하고 충전·가압·보압 증거로 사용하지 않는다. 온도 비교군 적합성은 정상 자기제외 반경 분포를 기준으로 연속적으로 보고한다.

In [ ]:
aux=['Plasticizing_Time','Plasticizing_Position','Max_Back_Pressure','Average_Back_Pressure','Cycle_Time','Clamp_Close_Time','Clamp_Open_Position','Max_Screw_RPM','Average_Screw_RPM','Mold_Temperature_3','Mold_Temperature_4']
for p,d in L.items():
    fig,axs=plt.subplots(3,4,figsize=(18,12))
    for ax,c in zip(axs.flat,aux):
        for label in [0,1]:ecdf(ax,d.loc[d.PassOrFail.eq(label),c],str(label),COLORS[label])
        finish(ax,p+' auxiliary condition',c,'ECDF')
    axs.flat[-1].axis('off');plt.tight_layout();plt.show()
    t=conditional_table[conditional_table['product'].eq(p)]
    for mold,g in t.groupby('include ambiguous mold'):
        ranges=g.groupby('source_id')['normal radius percentile'].agg(['min','max'])
        say(f'{p}, 모호한 금형온도 포함={mold}: 정상 반경 상대위치의 전체 범위 {g["normal radius percentile"].min():.3f}–{g["normal radius percentile"].max():.3f}. m 변경에 따른 사례별 상대위치 최대 변화 {(ranges["max"]-ranges["min"]).max():.3f}.')
        lacking=g[g['normal radius percentile'].eq(1)].source_id.unique().tolist()
        say(f'정상 점검 반경의 최댓값 이상인 사례(하나 이상의 m): {lacking}. 이 경우 비교 가능한 정상 부족의 정황으로 보고하며 차이를 단정하지 않는다.')
    core=conditional_table.query('product==@p').pivot_table(index=['source_id','m'],columns='include ambiguous mold',values=['temperature radius','speed median contrast','cushion median contrast','back median contrast'])
    display(core)
say('비교군 크기에 따라 요약이 변하거나 모호한 금형온도 포함 여부로 비교군이 달라질 수 있다. 따라서 하나의 m을 최적으로 채택하지 않고 표 전체를 민감도 결과로 유지했다. 열적 거리의 크기는 표준화된 통계적 유사성일 뿐 물리적 온도 차이가 아니다.')

## 8. EDA 근거 파생변수 생성
**설계:** CN7의 속도·쿠션 동반 패턴을 설명하기 위해 정상 고유 입력의 중간순위 ECDF와 방향성 결합을 생성한다. 중간순위는 동률 양쪽 끝 대신 중간을 배정하는 대칭 처리다. `min(speed_pct, cushion_low_pct)`는 두 조건 중 약한 쪽으로 제한하는 통계적 AND이며 물리량·불량확률이 아니다. 곱과 평균도 민감도 비교한다. 분포 밖 값은 순위 0/1에 압축되므로 원본값을 함께 보존한다.

정상 참조를 고정하고 모든 행에 동일 변환을 적용한다. 포함 최근접 거리를 먼저 제시하고, 전체 입력이 같은 참조를 **라벨과 무관하게** 제외한 거리는 별도의 주변패턴 설명 지표다. 정상 자기 자신만 라벨별로 특별 처리하지 않는다. 같은 원본 입력은 같은 파생변수를 가져야 한다. 전체 정상 참조를 사용한 아래 값은 EDA 전용이다.

In [ ]:
def midrank(ref,x):
    a=np.sort(np.asarray(ref));x=np.asarray(x)
    return (np.searchsorted(a,x,side='left')+np.searchsorted(a,x,side='right'))/(2*len(a))
def derive(ref,target):
    out=pd.DataFrame(index=target.index)
    out['speed_pct']=midrank(ref.Max_Injection_Speed,target.Max_Injection_Speed)
    out['cushion_low_pct']=1-midrank(ref.Cushion_Position,target.Cushion_Position)
    out['speed_cushion_min']=out[['speed_pct','cushion_low_pct']].min(axis=1)
    out['speed_cushion_product']=out.speed_pct*out.cushion_low_pct
    out['speed_cushion_mean']=(out.speed_pct+out.cushion_low_pct)/2
    dd=distance_matrix(ref,target,GROUPS['Filling'])
    out['filling_nearest_included']=dd.min(axis=1)
    same=np.ones(dd.shape,dtype=bool)
    for c in FEATURES:same &= target[c].to_numpy()[:,None]==ref[c].to_numpy()[None,:]
    excl=dd.copy();excl[same]=np.inf
    out['filling_nearest_excluded']=excl.min(axis=1)
    cs=['Max_Switch_Over_Pressure','Cushion_Position']
    freq=ref.groupby(cs).size()
    out['switch_cushion_frequency']=[freq.get(tuple(v),0)/len(ref) for v in target[cs].to_numpy()]
    return out
DERIVED={p:derive(NORMAL[p],d) for p,d in L.items()}
feature_specs=pd.DataFrame([
['speed_pct / cushion_low_pct','충전 속도와 쿠션 위치의 정상 대비 방향','정상 고유 입력 중간순위 / 1−중간순위','상대 위치','정상 참조, 동률 중간순위','후속 모델링 검증 후보','원본 크기·꼬리 정보 압축'],
['speed_cushion_min','속도 높음·쿠션 낮음 동반 관찰','min(speed_pct,cushion_low_pct)','통계적 결합','AND 의미; 곱·평균 비교','후속 모델링 검증 후보','서로 다른 입력이 동일 점수로 압축'],
['speed_cushion_product / mean','결합 연산 선택 민감도','곱 / 산술평균','통계적 결합','min과 순위·충돌 비교','설명용','평균은 한쪽 높음으로 상쇄 가능'],
['filling_nearest_included','정확 정상 조합 일치 보완','정상 참조 충전 2변수 최근접 거리','통계적 거리','IQR Euclidean; k=1은 가장 가까운 패턴','설명용','자기/동일 입력 포함 0거리'],
['filling_nearest_excluded','주변의 다른 전체 입력 설명','같은 전체 입력 제외 최근접 거리','통계적 거리','대상 라벨과 무관하게 동일 제외','보류','스케일·참조 희소성 영향'],
['switch_cushion_frequency','전환과 쿠션 이산 조합 지원','정상 고유 입력 중 조합 빈도','관측 빈도','반올림 없음; 행빈도와 구분','설명용','연속형 희소성은 이상이 아님'],
['conditional thermal contrasts','온도 유사 정상과 비교','후보 m별 중앙값 대비','통계적 비교','열적 반경·동률·m 안정성','보류','비교군 충분성 미확정'],
['physical time ratio / pressure or temperature difference','물리적 공정 관계','원시 시간비·압력차·온도차','물리량','원시값 또는 역변환 계수 필요','현재 생성 불가','서로 별도 표준화된 값으로 계산 불가']
],columns=['후보','관찰·가설','계산','의미','설정 근거','판정','한계'])
feature_specs['이용 가능성']='관련 센서값 수집 후·검사 전 제공 여부 확인 필요'
feature_specs['누수']='정상 참조·스케일·설정은 향후 학습 폴드 내부; 대상 라벨 사용 금지'
display(feature_specs)
for p,d in L.items():
    z=DERIVED[p]
    assert np.isfinite(z.to_numpy()).all()
    check=pd.concat([d[['pattern_id']],z],axis=1).groupby('pattern_id')[z.columns].nunique()
    assert check.le(1).all().all(),'같은 원본 입력의 변환 불일치'
    say(f'**{p}: 원본 피처 {len(FEATURES)}개 / 생성 후보 {len(z.columns)}개; 동일 입력 동일 변환 검증 통과**')
    display(z.agg(['count','nunique','min','max']).T.assign(missing=z.isna().sum(),infinite=np.isinf(z).sum(),constant=z.nunique().le(1)))
ORIGINAL_FEATURES=FEATURES.copy()
AUXILIARY_COLUMNS=['source_id','source_file','product','record_order','PassOrFail','pattern_id']
DERIVED_FEATURES=list(next(iter(DERIVED.values())).columns)
say('source_id, pattern_id, 상충 라벨 및 불량 빈도는 추론용 피처에 포함하지 않는다. 행 순서의 시간 의미가 확인되지 않아 이동평균·직전 차이·변화율은 생성하지 않는다.')


## 9. 파생변수의 탐색적 평가
**방법·설정:** 분포·원본 쌍·상관·압축 충돌·참조 민감도를 함께 확인한다. 상관은 Spearman으로 순서 중복을 본다. 80% 정상 참조 반복은 표본 의존성 점검이며 신뢰구간이나 독립 검증이 아니다. 불량 분리도를 최대화하여 설정을 고르지 않는다.

In [ ]:
sensitivity=[];collisions=[]
for p,d in L.items():
    z=DERIVED[p]
    fig,axs=plt.subplots(2,4,figsize=(19,8))
    for ax,c in zip(axs.flat,z.columns):
        for label in [0,1]:
            mask=d.PassOrFail.eq(label);ecdf(ax,z.loc[mask,c],f'{label} rows',COLORS[label])
            unique_mask=mask&~d.duplicated(FEATURES+['PassOrFail'])
            vals=np.sort(z.loc[unique_mask,c]);ax.step(vals,np.arange(1,len(vals)+1)/len(vals),ls='--',color=COLORS[label],label=f'{label} unique n={len(vals)}')
        vals=z.loc[d.PassOrFail.eq(1),c];ax.scatter(vals,np.zeros(len(vals)),marker='x',s=14,color=COLORS[1],label=f'Defect points n={len(vals)}')
        finish(ax,p+' '+c,c,'ECDF')
    plt.tight_layout();plt.show()
    fig,axs=plt.subplots(1,2,figsize=(14,5))
    for ax,c in zip(axs,['speed_cushion_min','filling_nearest_excluded']):
        sc=ax.scatter(d.Max_Injection_Speed,d.Cushion_Position,c=z[c],cmap='viridis',s=22,alpha=.7,label=f'Rows n={len(d)}')
        bad=d.PassOrFail.eq(1);ax.scatter(d.loc[bad,'Max_Injection_Speed'],d.loc[bad,'Cushion_Position'],facecolors='none',edgecolors=COLORS[1],s=65,label=f'Defects n={bad.sum()}')
        fig.colorbar(sc,ax=ax,label=c);finish(ax,p+' '+c,'Max injection speed','Cushion position')
    plt.tight_layout();plt.show()
    source=['Filling_Time','Max_Injection_Speed','Cushion_Position','Max_Switch_Over_Pressure','Average_Back_Pressure']
    combined=pd.concat([d[source],z],axis=1);uniq=~d.duplicated(FEATURES)
    fig,ax=plt.subplots(figsize=(12,9));sns.heatmap(combined.loc[uniq].corr(method='spearman'),vmin=-1,vmax=1,cmap='vlag',ax=ax,cbar_kws={'label':'Spearman correlation'})
    ax.set(title=f'{p} source/derived dependence; unique inputs n={uniq.sum()}',xlabel='Variables',ylabel='Variables');plt.tight_layout();plt.show()
    for c in z:
        pairs=pd.concat([d[['pattern_id']],z[[c]]],axis=1).drop_duplicates('pattern_id').groupby(c).pattern_id.nunique()
        collisions.append([p,c,int(uniq.sum()),len(pairs),int((pairs>1).sum()),int(pairs.max())])
    for rep in range(REPEATS):
        ref=NORMAL[p].sample(frac=.8,random_state=SEED+rep);alt=derive(ref,d)
        for c in z:
            sensitivity.append([p,rep,c,safe_corr(z[c],alt[c]),np.mean(np.abs(z[c]-alt[c]))])
    bad=d.PassOrFail.eq(1)
    display(pd.concat([d.loc[bad,['source_id','pattern_id']+source],z.loc[bad]],axis=1))
collision_table=pd.DataFrame(collisions,columns=['product','feature','unique input denominator','distinct scores','scores with collisions','maximum inputs per score']);display(collision_table)
sensitivity_table=pd.DataFrame(sensitivity,columns=['product','repeat','feature','rank correlation','mean absolute change']);display(sensitivity_table.groupby(['product','feature']).agg({'rank correlation':['min','median','max'],'mean absolute change':['min','median','max']}))
fig,axs=plt.subplots(1,2,figsize=(17,5))
for ax,p in zip(axs,L):
    t=sensitivity_table[sensitivity_table['product'].eq(p)]
    sns.boxplot(data=t,x='feature',y='rank correlation',ax=ax,color='lightblue');sns.stripplot(data=t,x='feature',y='rank correlation',ax=ax,color='black',size=3)
    ax.tick_params(axis='x',rotation=80);ax.set(title=f'{p} reference sensitivity (repeats={REPEATS}, rows={len(L[p])})',xlabel='Derived feature',ylabel='Rank correlation with full reference')
plt.tight_layout();plt.show()
for p,z in DERIVED.items():
    display(z[['speed_cushion_min','speed_cushion_product','speed_cushion_mean']].corr(method='spearman'))
say('점수의 압축 충돌은 결합 변수가 원본을 대체하지 못함을 보여준다. 최근접 포함 거리는 정상 자기 일치로 0이 많아 상관계수가 정의되지 않을 수 있다. NaN 상관은 유용성 증거가 아니다. 순위 안정성과 실제 원본값을 함께 보고, 특정 불량 몇 행과 중복에 의존하는 차이는 탐색 가설로만 유지한다. 현재 후보 판정은 후속 검증 우선순위이며 성능 향상을 평가하지 않았다.')


### 실제 결과에 따른 설정 판단과 파생변수 프로파일
**질문 → 방법:** 정상 참조 실험의 수치를 요약하고, 불량 몇 패턴에 의존하는지 고유 불량과 반복 행을 구분한다. 거리에서는 포함 결과를 먼저, 동일 전체입력 제외 결과를 다음으로 보인다. k 변화는 IQR 유클리드 거리에서 동일 대상을 비교하며 선택은 하지 않는다.

In [ ]:
for p,d in L.items():
    t=stability_table[stability_table['product'].eq(p)]
    for group,g in t.groupby('group'):
        say(f'{p} {group}: 정상 점검 순위 안정성 중앙값의 후보 범위 {g["rank stability 12"].min():.3f}–{g["rank stability 12"].max():.3f}; 6회와12회 요약 최대차 {(g["rank stability 12"]-g["rank stability 6"]).abs().max():.3f}. 안정성만으로 최적 이웃 수를 정하지 않는다.')
    z=DERIVED[p];mask=d.PassOrFail.eq(1)
    for name,idx in [('불량 행',mask),('고유 불량 패턴',mask&~d.duplicated(FEATURES+['PassOrFail']))]:
        say(f'{p} {name}: n={idx.sum()}, 충전 최근접 포함거리 0인 수={(z.loc[idx,"filling_nearest_included"]==0).sum()}, 전체입력 일치 제외 후 0인 수={(z.loc[idx,"filling_nearest_excluded"]==0).sum()}. 다른 전체입력이 같은 충전 조합을 가질 수 있어 제외 후에도 0이 가능하다.')
    profile_cols=['Max_Injection_Speed','Cushion_Position','Max_Switch_Over_Pressure','Average_Back_Pressure']
    # 서로 다른 척도를 한 색상 축에 혼합하지 않고 두 패널로 분리한다.
    fig,axs=plt.subplots(1,2,figsize=(14,8))
    sns.heatmap(d.loc[mask].set_index('source_id')[profile_cols],ax=axs[0],cmap='vlag',center=0,cbar_kws={'label':'File standardized value'})
    zz=z.loc[mask,['speed_pct','cushion_low_pct','speed_cushion_min','switch_cushion_frequency']].copy();zz.index=d.loc[mask,'source_id']
    sns.heatmap(zz,ax=axs[1],cmap='viridis',vmin=0,vmax=1,cbar_kws={'label':'Relative statistic (not probability)'})
    for ax,title in zip(axs,['Original','Derived']):ax.set(title=f'{p} {title} defects n={mask.sum()}',xlabel='Variables',ylabel='Source ID')
    plt.tight_layout();plt.show()
    ks=distance_runs[p,'Filling','iqr_euclidean'][0];ds=np.sort(distance_matrix(NORMAL[p],BAD[p],GROUPS['Filling']),axis=1)
    fig,ax=plt.subplots(figsize=(10,4))
    for j,(_,b) in enumerate(BAD[p].iterrows()):ax.plot(ks,[ds[j,:k].mean() for k in ks],marker='o',label=f'ID {b.source_id} (unique n=1)')
    finish(ax,f'{p} filling k sensitivity; defect unique n={len(BAD[p])}','Neighbors k','Mean neighbor distance, matches included')
    ax.legend(fontsize=6,ncol=2,loc='upper left',bbox_to_anchor=(1.02,1))
    plt.tight_layout();plt.show()
    # 3IQR 규칙의 실제 데이터 결과와 설정 변화 폭을 해설한다.
    for label in [0,1]:
        e=extreme_table.query('product==@p and label==@label');old=e[e['IQR coefficient'].eq(3)].iloc[0]
        say(f'{p}, 라벨 {label}: 재현용 3IQR 표시 {old["flagged rows"]}/{old["row denominator"]}행. 계수 후보에서 표시 행 수 범위 {e["flagged rows"].min()}–{e["flagged rows"].max()}. 극단값 표시는 라벨과 동의어가 아니다.')
say('파생 순위·결합은 원본에서 함께 관찰한 방향을 한 축으로 설명하지만 동일 점수로 압축되는 다른 패턴이 존재한다. RG3의 상충 입력 문제는 그대로 남는다. 참조 민감도 결과는 후보를 검증할 우선순위에 활용하되 이번 데이터에서 분리돼 보이는 정도를 독립 검증이라고 부르지 않는다.')

## 10. RG3 상충 생성 경위와 최종 해석
**증거 구분:**

- CSV 사실: 아래 재계산된 상충 비율·정확 일치·건수를 사용한다.
- 문서 처리: 43쪽에 CN7/RG3 LH·RH 제품명이 있고, 62쪽 코드는 LH/RH 제품코드를 통합한 뒤 PART_NO와 식별 컬럼을 제거한다. 69쪽은 파일별 표준화를 수행한다.
- 가설: 좌우 구분 제거 후 같은 공정 입력에 서로 다른 검사가 남았을 수 있다.
- 미확정: 각 행의 LH/RH, 실제 생산 사이클 대응, 중복이 생긴 시점은 현재 CSV로 복원할 수 없다. 원본 PART_NO, 시간, 설비, 시리얼, 좌우 제품별 검사 매핑이 필요하다.

43쪽 예시는 CN7 RH 3371/LH 3365, RG3 RH/LH 각 628이며 최종 분석 파일과 처리 단계가 다르다. 69쪽 예시 CN7 정상1398/불량27은 현재 CSV와 다르고 RG3 정상1157/불량25는 일치한다. 예시와 현재 버전이 같다고 전제하지 않는다.

In [ ]:
summary=[]
for p,d in L.items():
    b=d[d.PassOrFail.eq(1)];shared=b.pattern_id.isin(NORMAL[p].pattern_id)
    summary.append([p,len(d),len(NORMAL[p]),len(BAD[p]),len(b),int(shared.sum()),shared.mean(),int((PAT[p]['type']=='conflicting').sum())])
display(pd.DataFrame(summary,columns=['product','total rows','normal unique','bad unique','bad rows','bad rows exact normal match','matched fraction / bad rows','conflicting patterns']))
for p in L:
    z=DERIVED[p];d=L[p]
    pid=PAT[p].index[PAT[p]['type'].eq('conflicting')][0]
    display(pd.concat([d.loc[d.pattern_id.eq(pid),['source_id','PassOrFail','pattern_id']],z.loc[d.pattern_id.eq(pid)]],axis=1))
cn=L['CN7'];pure=cn[cn.PassOrFail.eq(1)&~cn.pattern_id.isin(NORMAL['CN7'].pattern_id)]
say(f'**핵심 관찰:** CN7에서 동일 정상 전체 입력이 없는 불량 ID는 {pure.source_id.tolist()}이다. 이들의 높은 최대 사출속도·낮은 쿠션 위치가 함께 나타나는지 원본 표와 산점도로 확인했다. 전환압력은 정상에서도 관측되므로 보압 전체의 이상으로 확대 해석하지 않는다. RG3 불량은 모두 정확한 정상 입력 대응을 가지므로 동일 참조·변환을 적용한 파생변수로 해당 정상/불량을 구분할 수 없다.')
display(feature_specs[['후보','판정','설정 근거','한계']])
say('**설정 근거와 제한:** 정확 일치는 임계값 없는 구조 확인이다. IQR 계수는 민감도 실험이고, k·거리·온도 비교군 크기는 정상 참조 안정성과 국소성으로 비교했으며 최적값을 확정하지 않았다. 정상 상대순위와 방향성 결합은 관계 설명을 추가하지만 새 측정정보를 만들지 않는다. 독립 효과·인과·공정 간 이상 전파·예측 성능 향상은 이번 분석으로 입증하지 않았다. 전체 데이터를 활용하여 가설을 선택한 EDA다. 후속 모델링에서는 전체 입력 패턴 단위로 분할하고 참조·스케일·빈도·파생변수 설정을 학습 폴드 내부에서 다시 계산해야 한다. 원시값, 표준화 계수, 단위 정정, 품질검사 이전 센서 이용 시점, 생산·좌우 제품 대응 정보를 추가 확보해야 한다.')

ref=NORMAL['CN7']
say(f'CN7 동일 정상 입력이 없는 6행은 3개 고유 패턴이다. 최대 사출속도 범위는 {pure.Max_Injection_Speed.min():.3f}–{pure.Max_Injection_Speed.max():.3f}, 관측 정상 최댓값은 {ref.Max_Injection_Speed.max():.3f}이다. 쿠션 위치는 {pure.Cushion_Position.unique().tolist()}, 관측 정상 최솟값은 {ref.Cushion_Position.min():.3f}이다. 모두 파일별 표준화 척도이며 물리적 허용범위를 뜻하지 않는다.')
display(direct_table.query("product=='CN7' and source_id in [115,117,119] and group=='Switch'")[['source_id','normal rows matched','normal unique matched']])
say('전환압력과 같은 정상값이 존재해도 충전·쿠션의 전체 조합까지 같다는 뜻은 아니다. CN7 95·103·107·113 등은 전체 정상 일치 여부를 함께 보아야 하며, 눈에 띄는 일부 값만으로 독립적인 결함 입력으로 분류하지 않는다.')


## 11. 실행 검증
아래 검증은 파일 보존, 주요 표·파생값의 완결성과 동일입력 변환 일관성을 확인한다. 예측 성능 검증은 아니다. 생성된 그래프는 노트북 출력에 포함되며 별도 HTML 보고서를 생성하지 않는다.

**실행 방식:** 이 환경은 커널 소켓 통신을 허용하지 않아 일반 Jupyter Restart & Run All 실행은 불가능했다. 대신 새 Python 프로세스의 IPython 실행 환경에서 위에서 아래로 모든 코드 셀을 실행하여 표·PNG 그래프·한국어 출력과 실행 번호를 저장했다. 외부 Jupyter 환경에서의 커널 실행은 별도 확인이 필요하다.

In [ ]:
for name,original in raw.items():
    assert len(original)==len(data[name])
    assert np.array_equal(original[FEATURES].to_numpy(),data[name][FEATURES].to_numpy())
assert len(variable_dictionary)==24
assert all(len(DERIVED[p])==len(L[p]) for p in L)
assert len(direct_table)>0 and len(stability_table)>0 and len(sensitivity_table)>0
print('전체 분석 셀 실행 완료: 원본 보존·출력 표·파생값·동일 입력 변환 확인. 모델 학습 없음.')


In [ ]:
plt.close("all")


# RG3 추가 군집·근접 정상 비교

원본 분석: `EDA/rg3_defect_factor_analysis.ipynb`. 모듈화 전 원본은 복구 ZIP에 보관되어 있습니다.


# RG3 공정 패턴 군집화

원본의 동일한 24개 입력을 하나의 패턴으로 묶은 뒤 K-means로 군집화합니다. 라벨은 군집 계산에 사용하지 않고, 완성된 군집에서 불량 이력 패턴이 어디에 분포하는지 확인하는 데만 사용합니다. 아래 세 셀은 독립적으로 실행됩니다.

원본 불량 25행은 각각 동일한 입력값의 정상 행과 쌍을 이룹니다. 따라서 여기의 불량 이력 비율은 **제품 불량률이 아니라 고유 공정 패턴 중 불량 이력이 있는 비율**입니다. PCA 2차원 그림은 시각화용이며 K-means는 모든 비상수 입력을 사용합니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

RG3_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'data/origin/moldset_labeled_rg3.csv').is_file())
RG3_CSV = RG3_ROOT / 'data/origin/moldset_labeled_rg3.csv'
CLUSTER_OUT = RG3_ROOT / 'output/modular/eda/rg3_defect_factor_eda/clustering'
CLUSTER_OUT.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RG3_CSV, index_col=0, float_precision='round_trip')
features = raw.columns.drop('PassOrFail').tolist()
assert len(features) == 24 and raw.PassOrFail.isin([0, 1]).all()
assert np.isfinite(raw[features].to_numpy()).all()

groups = raw.groupby(features, sort=False, dropna=False)
patterns = groups.size().reset_index(name='observations')
patterns['defect_count'] = groups.PassOrFail.sum().to_numpy()
patterns['risk'] = (patterns.defect_count > 0).astype(int)
patterns['source_ids'] = [','.join(map(str, g.index)) for _, g in groups]
patterns['pattern_id'] = [f'RG3_L_{i:06d}' for i in range(len(patterns))]
patterns = patterns.set_index('pattern_id')

# 군집화는 입력값만으로 수행한다. 상수 컬럼은 거리 계산에서 제외한다.
active_features = patterns[features].columns[patterns[features].nunique() > 1].tolist()
X = StandardScaler().fit_transform(patterns[active_features])
assert len(patterns) == 591 and patterns.risk.sum() == 25
print(f'고유 패턴 {len(patterns)}개, 불량 이력 패턴 {patterns.risk.sum()}개, '
      f'군집 입력 {len(active_features)}개 컬럼')


In [ ]:
K = 6  # 탐색용 군집 수. 아래에서 4·6·8개 결과도 비교한다.
model = KMeans(n_clusters=K, n_init=20, random_state=42)
patterns['cluster'] = model.fit_predict(X)

cluster_summary = patterns.groupby('cluster').agg(
    patterns=('risk', 'size'),
    risk_patterns=('risk', 'sum'),
)
cluster_summary['normal_only'] = cluster_summary.patterns - cluster_summary.risk_patterns
cluster_summary['risk_rate_pct'] = 100 * cluster_summary.risk_patterns / cluster_summary.patterns
cluster_summary['share_of_all_risk_pct'] = 100 * cluster_summary.risk_patterns / patterns.risk.sum()
cluster_summary['lift_vs_overall'] = (cluster_summary.risk_rate_pct /
                                      (100 * patterns.risk.mean()))
display(cluster_summary.round(2))
print(f'전체 불량 이력 패턴 비율: {patterns.risk.mean():.2%}')

# PCA는 군집화 이후 좌표를 그리기 위해서만 계산한다.
pca = PCA(n_components=2)
xy = pca.fit_transform(X)
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for cluster in sorted(patterns.cluster.unique()):
    mask = patterns.cluster.eq(cluster).to_numpy()
    axes[0].scatter(xy[mask, 0], xy[mask, 1], s=23, alpha=.55,
                    label=f'Cluster {cluster}')
risk_mask = patterns.risk.eq(1).to_numpy()
axes[0].scatter(xy[risk_mask, 0], xy[risk_mask, 1], s=90,
                facecolors='none', edgecolors='black', linewidths=1.5,
                label='Defect history')
axes[0].set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
            ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
            title=f'K-means groups (K={K}); black rings = defect history')
axes[0].legend(fontsize=8, ncol=2)
axes[0].grid(alpha=.2)

bottom = np.zeros(len(cluster_summary))
for values, label, color in [
    (cluster_summary.normal_only, 'Normal-only patterns', '#4C78A8'),
    (cluster_summary.risk_patterns, 'Defect-history patterns', '#E45756')]:
    axes[1].bar(cluster_summary.index.astype(str), values, bottom=bottom,
                color=color, label=label)
    bottom += values.to_numpy()
for i, row in enumerate(cluster_summary.itertuples()):
    axes[1].text(i, row.patterns + 2, f'{row.risk_patterns}/{row.patterns} ({row.risk_rate_pct:.1f}%)',
                 ha='center', fontsize=9)
axes[1].set(xlabel='Cluster', ylabel='Unique process patterns',
            title='Defect-history patterns by cluster')
axes[1].set_ylim(0, cluster_summary.patterns.max() * 1.18)
axes[1].legend(fontsize=8)
fig.savefig(CLUSTER_OUT / 'rg3_kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# 군집 수 선택에 따라 위험 패턴 집중도가 달라지는지 살핀다.
sensitivity = []
for k in (4, 6, 8):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)
    table = pd.DataFrame({'cluster': labels, 'risk': patterns.risk.to_numpy()})
    totals = table.groupby('cluster').risk.agg(['size', 'sum'])
    rates = totals['sum'] / totals['size']
    highest = rates.idxmax()
    sensitivity.append({'K': k, 'highest_rate_cluster': int(highest),
                        'patterns_in_cluster': int(totals.loc[highest, 'size']),
                        'risk_patterns_in_cluster': int(totals.loc[highest, 'sum']),
                        'highest_risk_rate_pct': 100 * rates.max(),
                        'risk_coverage_pct': 100 * totals.loc[highest, 'sum'] / patterns.risk.sum()})
sensitivity = pd.DataFrame(sensitivity)
display(sensitivity.round(2))
cluster_summary.to_csv(CLUSTER_OUT / 'cluster_summary.csv', encoding='utf-8-sig')
patterns[['source_ids', 'observations', 'defect_count', 'risk', 'cluster']].to_csv(
    CLUSTER_OUT / 'pattern_clusters.csv', encoding='utf-8-sig')
sensitivity.to_csv(CLUSTER_OUT / 'k_sensitivity.csv', index=False, encoding='utf-8-sig')
print(f'저장 위치: {CLUSTER_OUT}')


## 세 가지 추가 비지도 군집화 비교

위의 `rg3-clustering-setup` 셀을 먼저 실행합니다. 세 방법은 동일한 591개 고유 패턴의 표준화 입력을 사용하되, **각 방법의 고유한 결과를 별도 그림**으로 보여줍니다. HDBSCAN은 군집·잡음과 소속 강도, GMM은 성분별 소속 확률, Ward는 병합 덴드로그램과 최종 군집의 불량 이력 분포를 표시합니다. 불량 라벨은 군집 적합에 쓰지 않습니다.

HDBSCAN 잡음은 불량 판정이 아니며, GMM 소속 확률도 불량 확률이 아닙니다. 덴드로그램은 마지막 30번의 병합만 표시합니다.


In [ ]:
from sklearn.cluster import HDBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

# 모든 알고리즘은 591개 고유 패턴의 동일한 표준화 입력 X를 사용한다.
H_MIN_SIZE = 10
GMM_K = 6
WARD_K = 6
hdb = HDBSCAN(min_cluster_size=H_MIN_SIZE, min_samples=3, copy=True)
hdb_labels = hdb.fit_predict(X)
gmm = GaussianMixture(n_components=GMM_K, covariance_type='diag',
                      reg_covar=0.01, n_init=5, random_state=42)
gmm_labels = gmm.fit_predict(X)
gmm_max_membership = gmm.predict_proba(X).max(axis=1)
ward_labels = AgglomerativeClustering(n_clusters=WARD_K, linkage='ward').fit_predict(X)

alternative_labels = {
    'HDBSCAN': hdb_labels,
    'GMM': gmm_labels,
    'Ward': ward_labels,
}

def cluster_table(labels):
    table = pd.DataFrame({'cluster': labels, 'risk': patterns.risk.to_numpy()})
    result = table.groupby('cluster').risk.agg(patterns='size', risk_patterns='sum')
    result['normal_only'] = result.patterns - result.risk_patterns
    result['risk_rate_pct'] = 100 * result.risk_patterns / result.patterns
    result['risk_coverage_pct'] = 100 * result.risk_patterns / patterns.risk.sum()
    result['lift_vs_overall'] = result.risk_rate_pct / (100 * patterns.risk.mean())
    return result

alt_tables = {name: cluster_table(labels) for name, labels in alternative_labels.items()}
for name, table in alt_tables.items():
    print(f'\n{name}: cluster -1 means HDBSCAN noise')
    display(table.round(2))
    assert table.patterns.sum() == len(patterns)
    assert table.risk_patterns.sum() == patterns.risk.sum()

print(f'GMM 최대 소속 확률 중앙값: {np.median(gmm_max_membership):.2f}; '
      '이 값은 불량 확률이 아닙니다.')

# HDBSCAN: 군집/잡음 배치와 밀도 기반 소속 강도. PCA는 위치 표시용이다.
alt_xy = PCA(n_components=2).fit_transform(X)
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
risk_mask = patterns.risk.eq(1).to_numpy()
for label in sorted(np.unique(hdb_labels)):
    mask = hdb_labels == label
    axes[0].scatter(alt_xy[mask, 0], alt_xy[mask, 1],
                    color='#aaaaaa' if label == -1 else f'C{int(label) % 10}',
                    s=22, alpha=.55, label='Noise (-1)' if label == -1 else f'Cluster {label}')
axes[0].scatter(alt_xy[risk_mask, 0], alt_xy[risk_mask, 1],
                s=90, facecolors='none', edgecolors='black', linewidths=1.4,
                label='Defect history')
axes[0].set(title='HDBSCAN: clusters and noise', xlabel='PCA 1', ylabel='PCA 2')
axes[0].legend(fontsize=8, ncol=2)
axes[0].grid(alpha=.2)
axes[1].hist(hdb.probabilities_[~risk_mask], bins=np.linspace(0, 1, 11),
             alpha=.65, label='Normal-only')
axes[1].hist(hdb.probabilities_[risk_mask], bins=np.linspace(0, 1, 11),
             alpha=.8, label='Defect history')
axes[1].set(title='HDBSCAN: membership strength (noise = 0)',
            xlabel='Strength of assigned cluster', ylabel='Unique patterns')
axes[1].legend()
fig.savefig(CLUSTER_OUT / 'rg3_hdbscan.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# GMM: 군집별 소속 확률. 이것은 불량 확률이 아니다.
gmm_probabilities = gmm.predict_proba(X)
order = np.lexsort((gmm_max_membership, gmm_labels))
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
im = axes[0].imshow(gmm_probabilities[order].T, aspect='auto',
                    cmap='viridis', vmin=0, vmax=1)
axes[0].scatter(np.flatnonzero(risk_mask[order]), gmm_labels[order][risk_mask[order]],
                marker='|', s=90, c='red', label='Defect history')
axes[0].set(title='GMM: component membership probabilities',
            xlabel='Unique patterns grouped by assigned component',
            ylabel='Mixture component')
axes[0].set_yticks(np.arange(GMM_K))
axes[0].legend(loc='upper right')
fig.colorbar(im, ax=axes[0], label='Membership probability')
axes[1].boxplot([gmm_max_membership[~risk_mask], gmm_max_membership[risk_mask]],
                tick_labels=['Normal-only', 'Defect history'], showfliers=True)
axes[1].set(title='GMM: strongest component membership',
            ylabel='Maximum membership probability')
axes[1].grid(axis='y', alpha=.2)
fig.savefig(CLUSTER_OUT / 'rg3_gmm.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# Ward: 실제 병합 높이의 요약 덴드로그램과 최종 군집별 불량 분포.
from scipy.cluster.hierarchy import linkage, dendrogram
ward_tree = linkage(X, method='ward')
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
dendrogram(ward_tree, truncate_mode='lastp', p=30,
           show_leaf_counts=True, leaf_rotation=75, ax=axes[0])
axes[0].set(title='Ward: last 30 merges (numbers = pattern counts)',
            xlabel='Merged groups', ylabel='Ward distance')
ward_table = alt_tables['Ward']
positions = np.arange(len(ward_table))
axes[1].bar(positions, ward_table.normal_only, color='#4C78A8', label='Normal-only')
axes[1].bar(positions, ward_table.risk_patterns, bottom=ward_table.normal_only,
            color='#E45756', label='Defect history')
for i, record in enumerate(ward_table.itertuples()):
    axes[1].text(i, record.patterns + ward_table.patterns.max()*.02,
                 f'{record.risk_patterns}/{record.patterns}\n{record.risk_rate_pct:.1f}%',
                 ha='center', fontsize=9)
axes[1].set_xticks(positions, [str(v) for v in ward_table.index])
axes[1].set_ylim(0, ward_table.patterns.max()*1.18)
axes[1].set(title='Ward: defect history in final clusters',
            xlabel='Cluster', ylabel='Unique patterns')
axes[1].legend()
fig.savefig(CLUSTER_OUT / 'rg3_ward.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# 설정 변화에 따른 가장 높은 군집 비율과 해당 군집의 불량 포괄률
setting_rows = []
for size in (5, 10, 15, 20):
    setting_rows.append(('HDBSCAN', f'min_cluster_size={size}',
        HDBSCAN(min_cluster_size=size, min_samples=3, copy=True).fit_predict(X)))
for k in (4, 6, 8):
    setting_rows.append(('GMM', f'components={k}',
        GaussianMixture(n_components=k, covariance_type='diag', reg_covar=.01,
                        n_init=5, random_state=42).fit_predict(X)))
    setting_rows.append(('Ward', f'clusters={k}',
        AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X)))

stability = []
for method, setting, labels in setting_rows:
    table = cluster_table(labels)
    # 잡음은 최대 비율 군집을 찾을 때 제외하되 규모를 따로 표시한다.
    candidates = table.drop(index=-1, errors='ignore')
    best = candidates.sort_values(['risk_rate_pct', 'risk_patterns'], ascending=False).iloc[0]
    noise = table.loc[-1] if -1 in table.index else None
    stability.append({'method': method, 'setting': setting,
        'clusters': len(candidates), 'noise_patterns': int(noise.patterns) if noise is not None else 0,
        'risk_in_noise': int(noise.risk_patterns) if noise is not None else 0,
        'best_cluster_size': int(best.patterns), 'risk_in_best': int(best.risk_patterns),
        'best_risk_rate_pct': best.risk_rate_pct,
        'best_risk_coverage_pct': best.risk_coverage_pct})
stability = pd.DataFrame(stability)
print('설정 민감도 — 최고 비율 군집은 각 설정 안에서만 비교')
display(stability.round(2))

for name, table in alt_tables.items():
    table.to_csv(CLUSTER_OUT / f'{name.lower()}_cluster_summary.csv', encoding='utf-8-sig')
assignments = patterns[['source_ids', 'observations', 'defect_count', 'risk']].copy()
for name, labels in alternative_labels.items():
    assignments[name.lower() + '_cluster'] = labels
assignments['gmm_max_membership'] = gmm_max_membership
assignments.to_csv(CLUSTER_OUT / 'alternative_cluster_assignments.csv', encoding='utf-8-sig')
stability.to_csv(CLUSTER_OUT / 'alternative_cluster_sensitivity.csv', index=False,
                 encoding='utf-8-sig')
print(f'저장 위치: {CLUSTER_OUT}')


## 공정 단계별 입력만 사용한 군집화

용어집 `reference/KAMP_moldset_용어해설집_RPG비유.docx`의 변수 설명을 바탕으로 사출·충전, 가소화·스크류·배압, 배럴·호퍼 온도, 금형 온도, 사이클·형개폐의 다섯 묶음으로 나눴습니다. `rg3-clustering-setup` 셀을 먼저 실행하세요. 각 묶음은 서로 다른 입력값으로 **별도의 K-means 모델**을 적합합니다. 라벨은 군집화 이후 집계에만 사용합니다.

산점도 좌표도 해당 묶음의 컬럼만 사용합니다. 컬럼이 두 개인 금형 온도와 사이클·형개폐 묶음은 표준화된 두 값을 그대로 표시합니다. `Clamp_Open_Position`은 상수라 군집 거리에서 제외합니다. 각 군집 위의 `불량 이력 수/군집 크기`를 확인하세요. 전체 기준은 25/591=4.23%입니다.

다섯 묶음과 K=3·4·5에서 크기 20 이상인 군집을 모두 검색한 뒤, 최고 비율에 대해 3,000회 라벨 재배치 비교를 합니다. 이는 탐색 선택 효과를 고려한 참고값이며 독립 검증 데이터의 성능은 아닙니다.


In [ ]:
DOMAIN_GROUPS = {
    'Injection and filling': [
        'Injection_Time', 'Filling_Time', 'Max_Injection_Speed',
        'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Cushion_Position'],
    'Plasticizing and back pressure': [
        'Plasticizing_Time', 'Plasticizing_Position',
        'Max_Screw_RPM', 'Average_Screw_RPM',
        'Max_Back_Pressure', 'Average_Back_Pressure'],
    'Barrel and hopper temperature': [
        *[f'Barrel_Temperature_{i}' for i in range(1, 7)],
        'Hopper_Temperature'],
    'Mold temperature': ['Mold_Temperature_3', 'Mold_Temperature_4'],
    'Cycle and clamp': ['Cycle_Time', 'Clamp_Close_Time', 'Clamp_Open_Position'],
}
assert set(sum(DOMAIN_GROUPS.values(), [])) == set(features)
assert len(sum(DOMAIN_GROUPS.values(), [])) == len(features)

DOMAIN_K = 4
domain_assignments = patterns[['source_ids', 'observations', 'defect_count', 'risk']].copy()
domain_tables = {}
domain_coordinates = {}
domain_settings = []
search_masks = []
for name, columns in DOMAIN_GROUPS.items():
    active = [c for c in columns if patterns[c].nunique() > 1]
    Z = StandardScaler().fit_transform(patterns[active])
    labels = KMeans(n_clusters=DOMAIN_K, n_init=20, random_state=42).fit_predict(Z)
    table = cluster_table(labels)
    domain_tables[name] = table
    domain_assignments[name] = labels
    if Z.shape[1] == 2:
        coords = Z
        axis_labels = active
    else:
        projection = PCA(n_components=2).fit(Z)
        coords = projection.transform(Z)
        axis_labels = ['PC1', 'PC2']
    domain_coordinates[name] = (coords, axis_labels)
    assert table.patterns.sum() == 591 and table.risk_patterns.sum() == 25
    print(f'\n{name} | 군집 입력 {len(active)}개: {", ".join(active)}')
    display(table.round(2))
    for k in (3, 4, 5):
        candidate = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(Z)
        candidate_table = cluster_table(candidate)
        # 군집 크기가 20 이상인 곳에서의 최고 비율도 함께 기록한다.
        eligible = candidate_table[candidate_table.patterns >= 20]
        if eligible.empty:
            continue
        for cluster_id in eligible.index:
            search_masks.append(candidate == cluster_id)
        best = eligible.sort_values(['risk_rate_pct', 'risk_patterns'], ascending=False).iloc[0]
        domain_settings.append({'domain': name, 'K': k,
            'best_cluster_size': int(best.patterns),
            'risk_in_best': int(best.risk_patterns),
            'best_rate_pct': best.risk_rate_pct,
            'coverage_pct': best.risk_coverage_pct})

# 관심이 높은 배럴·호퍼 온도 군집의 변수별 평균을 전체 패턴 대비 표준편차로 표시
temperature_columns = DOMAIN_GROUPS['Barrel and hopper temperature']
temperature_values = patterns[temperature_columns]
temperature_z = (temperature_values - temperature_values.mean()) / temperature_values.std(ddof=0)
temperature_profile = temperature_z.assign(
    cluster=domain_assignments['Barrel and hopper temperature']).groupby('cluster').mean()
print('배럴·호퍼 온도 군집 프로필: 양수는 전체 패턴 평균보다 높음')
display(temperature_profile.round(2))
temperature_profile.to_csv(CLUSTER_OUT / 'domain_barrel_hopper_profile.csv',
                           encoding='utf-8-sig')

fig, axes = plt.subplots(len(DOMAIN_GROUPS), 2,
                         figsize=(17, 4.5*len(DOMAIN_GROUPS)), constrained_layout=True)
risk_mask = patterns.risk.eq(1).to_numpy()
for row, name in enumerate(DOMAIN_GROUPS):
    ax, bars = axes[row]
    coords, axis_labels = domain_coordinates[name]
    labels = domain_assignments[name].to_numpy()
    table = domain_tables[name]
    for label in sorted(np.unique(labels)):
        mask = labels == label
        ax.scatter(coords[mask, 0], coords[mask, 1], s=20, alpha=.55,
                   label=f'Cluster {label}')
    ax.scatter(coords[risk_mask, 0], coords[risk_mask, 1],
               s=90, facecolors='none', edgecolors='black', linewidths=1.3,
               label='Defect history')
    ax.set(title=name, xlabel=axis_labels[0], ylabel=axis_labels[1])
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=.2)

    x_pos = np.arange(len(table))
    bars.bar(x_pos, table.normal_only, color='#4C78A8', label='Normal-only')
    bars.bar(x_pos, table.risk_patterns, bottom=table.normal_only,
             color='#E45756', label='Defect history')
    for i, record in enumerate(table.itertuples()):
        bars.text(i, record.patterns + max(table.patterns)*.02,
                  f'{record.risk_patterns}/{record.patterns}\n{record.risk_rate_pct:.1f}%',
                  ha='center', fontsize=9)
    bars.set_xticks(x_pos, [str(v) for v in table.index])
    bars.set_ylim(0, max(table.patterns)*1.17)
    bars.set(xlabel='Cluster', ylabel='Unique patterns',
             title=f'{name}: defect-history share')
    bars.legend(fontsize=8)
fig.savefig(CLUSTER_OUT / 'rg3_domain_clusters.png', dpi=130, bbox_inches='tight')
plt.show()
plt.close(fig)

domain_sensitivity = pd.DataFrame(domain_settings)
# 다섯 묶음 × K=3,4,5의 군집을 모두 검색한 뒤 최고 비율을 고른 효과를 반영한다.
# 고정된 군집 배정 위에서 불량 이력 라벨 25개를 무작위로 재배치한다.
mask_matrix = np.stack(search_masks).astype(np.int16)
sizes = mask_matrix.sum(axis=1)
observed_max_rate = float((mask_matrix @ patterns.risk.to_numpy() / sizes).max())
rng = np.random.default_rng(42)
N_PERMUTATIONS = 3000
random_max_rates = np.empty(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    shuffled = rng.permutation(patterns.risk.to_numpy())
    random_max_rates[i] = (mask_matrix @ shuffled / sizes).max()
permutation_p = (1 + np.count_nonzero(random_max_rates >= observed_max_rate)) / (N_PERMUTATIONS + 1)
print(f'검색한 군집 {len(sizes)}개(크기 20 이상) 중 최대 비율 {observed_max_rate:.2%}; '
      f'무작위 라벨 재배치 기준 p={permutation_p:.3f} ({N_PERMUTATIONS}회)')
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.hist(random_max_rates * 100, bins=30, color='#4C78A8', alpha=.8)
ax.axvline(observed_max_rate * 100, color='#E45756', lw=2,
           label=f'Observed maximum {observed_max_rate:.1%}')
ax.set(title='Maximum rate after searching all domain groups and K values',
       xlabel='Highest defect-history percentage (%)', ylabel='Random label permutations')
ax.legend()
fig.savefig(CLUSTER_OUT / 'rg3_domain_permutation_check.png', dpi=150)
plt.show()
plt.close(fig)
print('군집 수 민감도: 크기 20 이상인 군집만 최고 비율 후보로 표시')
display(domain_sensitivity.round(2))
domain_assignments.to_csv(CLUSTER_OUT / 'domain_cluster_assignments.csv',
                          encoding='utf-8-sig')
domain_sensitivity.to_csv(CLUSTER_OUT / 'domain_cluster_sensitivity.csv',
                          index=False, encoding='utf-8-sig')
for name, table in domain_tables.items():
    safe = name.lower().replace(' ', '_')
    table.to_csv(CLUSTER_OUT / f'domain_{safe}_summary.csv', encoding='utf-8-sig')
print(f'저장 위치: {CLUSTER_OUT}')


## 고유 공정 패턴 순서 기반 변화·선행 창 군집화

위의 `rg3-clustering-setup`과 공정 묶음 정의 셀을 먼저 실행합니다. 원본의 동일 입력 두 행을 고유 패턴 하나로 사용하고, **원본 첫 등장 순서**로 정렬합니다. 인덱스를 실제 시각으로 가정하지 않습니다. 원본의 동일 입력 쌍 일부는 인접하지 않으므로 이것은 기록 순서에 대한 근사입니다.

첫 그림은 공정 묶음별 연속 패턴 간 변화 크기를 보여줍니다. 이후 각 패턴의 **직전 10개 또는 20개 패턴만**으로 평균·변동·시작과 끝의 차이·평균 단계 변화를 계산해 별도 K-means 군집을 만듭니다. 현재 패턴의 입력값과 불량 라벨은 군집 입력에 포함하지 않습니다. 서로 겹치는 창은 독립 표본이 아니므로 군집별 비율은 탐색 지표입니다. 원본의 라벨 순서나 인접 중복 행을 특징으로 사용하지 않습니다.

마지막 검사는 불량 이력 25개의 **간격을 그대로 유지한 채 위치만 원형 이동**합니다. 다섯 묶음·두 창 길이·군집 수 3~5에서 최고 비율을 고른 효과를 함께 고려합니다. 독립 생산 기간 검증을 대신하지는 않습니다.


In [ ]:
# patterns와 DOMAIN_GROUPS는 위의 설정 셀에서 정의됨
assert len(patterns) == 591 and len(DOMAIN_GROUPS) == 5
sequence = patterns.copy()  # groupby(sort=False): 원본에서 각 패턴의 첫 등장 순서
sequence['source_first_id'] = [int(s.split(',')[0]) for s in sequence.source_ids]
assert sequence.source_first_id.is_monotonic_increasing
seq_risk = sequence.risk.to_numpy()
N = len(sequence)

# 현재 패턴과 바로 직전 패턴 사이의 점프. 탐색용 시각화이며 미래 값을 쓰지 않는다.
fig, axes = plt.subplots(len(DOMAIN_GROUPS), 1, figsize=(16, 12),
                         sharex=True, constrained_layout=True)
change_scores = {}
for ax, (name, columns) in zip(axes, DOMAIN_GROUPS.items()):
    active = [c for c in columns if sequence[c].nunique() > 1]
    z = StandardScaler().fit_transform(sequence[active])
    jump = np.r_[np.nan, np.sqrt((np.diff(z, axis=0)**2).mean(axis=1))]
    change_scores[name] = jump
    ax.plot(np.arange(N), jump, lw=.8, color='#4C78A8')
    ax.scatter(np.flatnonzero(seq_risk), jump[seq_risk == 1],
               color='#E45756', s=30, zorder=3, label='Defect-history pattern')
    ax.set(ylabel='Step change', title=name)
    ax.grid(alpha=.2)
axes[0].legend()
axes[-1].set_xlabel('Unique-pattern order (not elapsed time)')
fig.savefig(CLUSTER_OUT / 'rg3_domain_step_changes.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

def preceding_window_features(values, window):
    # 행 t의 특징은 [t-window, ..., t-1]만 사용한다.
    result = []
    for t in range(window, len(values)):
        past = values[t-window:t]
        result.append(np.r_[past.mean(axis=0), past.std(axis=0),
                            past[-1] - past[0], np.abs(np.diff(past, axis=0)).mean(axis=0)])
    return np.asarray(result)

TEMPORAL_WINDOWS = (10, 20)
TEMPORAL_K = 4
temporal_tables = {}
temporal_assignments = pd.DataFrame(index=sequence.index)
temporal_assignments['source_first_id'] = sequence.source_first_id
temporal_assignments['risk'] = sequence.risk
temporal_candidates = []
temporal_summary = []

for name, columns in DOMAIN_GROUPS.items():
    active = [c for c in columns if sequence[c].nunique() > 1]
    values = StandardScaler().fit_transform(sequence[active])
    for window in TEMPORAL_WINDOWS:
        F = preceding_window_features(values, window)
        # K-means 입력 요약치도 표준화한다. 군집 적합에 라벨은 사용하지 않는다.
        F = StandardScaler().fit_transform(F)
        for k in (3, 4, 5):
            labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(F)
            target_risk = seq_risk[window:]
            table = pd.DataFrame({'cluster': labels, 'risk': target_risk}).groupby('cluster').risk.agg(
                targets='size', risk_targets='sum')
            table['risk_rate_pct'] = 100 * table.risk_targets / table.targets
            table['coverage_pct'] = 100 * table.risk_targets / target_risk.sum()
            if k == TEMPORAL_K:
                temporal_tables[(name, window)] = table
                colname = f'{name} | W={window}'
                temporal_assignments[colname] = pd.Series(labels, index=sequence.index[window:])
            for cluster, record in table.iterrows():
                if record.targets >= 20:
                    mask = np.zeros(N, dtype=bool)
                    mask[window:] = labels == cluster
                    temporal_candidates.append(mask)
            best = table.loc[table.targets >= 20].sort_values(
                ['risk_rate_pct', 'risk_targets'], ascending=False).iloc[0]
            temporal_summary.append({'domain': name, 'window': window, 'K': k,
                'best_group_size': int(best.targets), 'risk_in_best': int(best.risk_targets),
                'best_rate_pct': best.risk_rate_pct, 'coverage_pct': best.coverage_pct})

temporal_summary = pd.DataFrame(temporal_summary)
print('직전 창 군집화: K=4 결과 (각 군집의 라벨은 해당 방법 안에서만 의미가 있음)')
for (name, window), table in temporal_tables.items():
    print(f'\n{name} | preceding W={window}')
    display(table.round(2))

# 각 묶음·창 길이의 최고 군집 비율을 시각적으로 비교
current = temporal_summary.loc[temporal_summary.K.eq(TEMPORAL_K)].copy()
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
positions = np.arange(len(DOMAIN_GROUPS))
for offset, window in [(-.18, 10), (.18, 20)]:
    view = current[current.window.eq(window)].set_index('domain').loc[list(DOMAIN_GROUPS)]
    ax.bar(positions+offset, view.best_rate_pct, width=.35, label=f'Previous {window} patterns')
    for x, record in zip(positions+offset, view.itertuples()):
        ax.text(x, record.best_rate_pct+.15,
                f'{record.risk_in_best}/{record.best_group_size}', ha='center', fontsize=8)
ax.axhline(100*sequence.risk.mean(), color='black', ls='--',
           label=f'All patterns {100*sequence.risk.mean():.1f}%')
ax.set_xticks(positions, list(DOMAIN_GROUPS), rotation=20, ha='right')
ax.set(ylabel='Highest cluster risk-history rate (%)',
       title='Clustering on preceding process behavior only (K=4)')
ax.legend()
fig.savefig(CLUSTER_OUT / 'rg3_preceding_window_clusters.png', dpi=150,
            bbox_inches='tight')
plt.show()
plt.close(fig)

# 불량 간격과 총 개수를 보존하는 원형 이동 검정.
mask_matrix = np.stack(temporal_candidates).astype(np.int16)
sizes = mask_matrix.sum(axis=1)
observed = float((mask_matrix @ seq_risk / sizes).max())
shift_max = np.array([(mask_matrix @ np.roll(seq_risk, shift) / sizes).max()
                      for shift in range(1, N)])
shift_p = (1 + np.count_nonzero(shift_max >= observed)) / N
print(f'전체 5개 묶음 × 창 10·20 × K=3·4·5, 최소 군집 20개: '
      f'최고 비율 {observed:.2%}; 위치 이동 기준 p={shift_p:.3f}')

display(temporal_summary.round(2))
temporal_assignments.to_csv(CLUSTER_OUT / 'temporal_cluster_assignments.csv',
                            encoding='utf-8-sig')
temporal_summary.to_csv(CLUSTER_OUT / 'temporal_cluster_summary.csv',
                        index=False, encoding='utf-8-sig')
pd.DataFrame(change_scores, index=sequence.index).to_csv(
    CLUSTER_OUT / 'domain_step_changes.csv', encoding='utf-8-sig')
print(f'저장 위치: {CLUSTER_OUT}')


---
## 이전 분석


# RG3 — 불량 이력 패턴과 근접 정상 패턴의 차이

이 섹션은 아래의 기존 CN7 기반 탐색 코드와 독립적으로 실행됩니다. **이 섹션만 위에서부터 실행하세요.**

- 원본 RG3의 24개 입력이 정확히 같은 행을 한 패턴으로 묶습니다. 불량 이력이 하나라도 있으면 위험 패턴입니다. 이는 개별 제품의 불량 판정과 다릅니다.
- 정상만 관측된 패턴으로 평균과 표준편차를 구하고, 표준화된 입력의 유클리드 거리로 각 위험 패턴의 근접 정상 K개를 찾습니다. 상수 컬럼은 거리에서 제외합니다.
- 히트맵 값 = **(위험 패턴 값 − 근접 정상 K개 평균) / 전체 정상 패턴의 표준편차**. 빨강은 더 높음, 파랑은 더 낮음입니다. 회색은 정상 표준편차가 0이라 비교할 수 없는 컬럼입니다.
- 모든 변수를 이웃 선정에도 사용하므로 차이가 작아지는 경향이 있으며, 상관이 큰 컬럼들이 거리에 중복 기여할 수 있습니다. 이웃은 동일 생산 시기나 동일 설정 조건이 보장되지 않습니다.
- 이 결과는 탐색용이며 인과관계나 예측 성능의 증거가 아닙니다. 원래 파일의 인덱스를 시간으로 가정하지 않습니다.


In [ ]:
K_NEIGHBORS = 5  # 기존 근접 정상 비교의 기본 이웃 수
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RG3_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data/origin/moldset_labeled_rg3.csv").is_file())
RG3_OUT = RG3_ROOT / "output/modular/eda/rg3_defect_factor_eda/neighbor_heatmap"
RG3_OUT.mkdir(parents=True, exist_ok=True)
K_NEIGHBORS = 5  # 3, 5, 10 등으로 비교. 정상 패턴 수 이하로 지정

rg3_raw = pd.read_csv(RG3_ROOT / "data/origin/moldset_labeled_rg3.csv",
                      index_col=0, float_precision="round_trip")
rg3_features = rg3_raw.columns.drop("PassOrFail").tolist()
assert len(rg3_features) == 24
assert rg3_raw["PassOrFail"].isin([0, 1]).all()
assert np.isfinite(rg3_raw[rg3_features].to_numpy()).all()

# 혼합 패턴의 정상 행은 정상 비교군에서 제외하여 동일 입력과의 비교 방지
rg3_groups = rg3_raw.groupby(rg3_features, sort=False, dropna=False)
rg3_pattern = rg3_groups.size().reset_index(name="observations")
rg3_pattern["risk"] = rg3_groups["PassOrFail"].max().to_numpy()
rg3_pattern["defects"] = rg3_groups["PassOrFail"].sum().to_numpy()
rg3_pattern["source_ids"] = [list(g.index) for _, g in rg3_groups]
rg3_pattern["pattern_id"] = [f"RG3_L_{i:06d}" for i in range(len(rg3_pattern))]
rg3_pattern = rg3_pattern.set_index("pattern_id")
rg3_normal = rg3_pattern.loc[rg3_pattern.risk.eq(0)]
rg3_risk = rg3_pattern.loc[rg3_pattern.risk.eq(1)]
assert len(rg3_normal) and len(rg3_risk)
assert isinstance(K_NEIGHBORS, int) and 1 <= K_NEIGHBORS <= len(rg3_normal)
print(f"Rows: {len(rg3_raw)}; unique patterns: {len(rg3_pattern)}; "
      f"normal-only: {len(rg3_normal)}; defect-history: {len(rg3_risk)}")

rg3_mu = rg3_normal[rg3_features].mean()
rg3_sd = rg3_normal[rg3_features].std(ddof=0).replace(0, np.nan)
rg3_active = rg3_sd.dropna().index
rg3_normal_z = ((rg3_normal[rg3_active] - rg3_mu[rg3_active]) / rg3_sd[rg3_active]).to_numpy()
rg3_risk_z = ((rg3_risk[rg3_active] - rg3_mu[rg3_active]) / rg3_sd[rg3_active]).to_numpy()
rg3_distance = np.sqrt(((rg3_risk_z[:, None, :] - rg3_normal_z[None, :, :]) ** 2).sum(axis=2))
rg3_order = np.argsort(rg3_distance, axis=1, kind="stable")

def rg3_neighbor_difference(k):
    neighbors = rg3_normal[rg3_features].to_numpy()[rg3_order[:, :k]]
    baseline = neighbors.mean(axis=1)
    return pd.DataFrame((rg3_risk[rg3_features].to_numpy() - baseline) / rg3_sd.to_numpy(),
                        index=rg3_risk.index, columns=rg3_features)

rg3_delta = rg3_neighbor_difference(K_NEIGHBORS)
rg3_summary = pd.DataFrame({
    "mean_abs_delta": rg3_delta.abs().mean(),
    "median_delta": rg3_delta.median(),
    "positive_count": (rg3_delta > 0).sum(),
    "negative_count": (rg3_delta < 0).sum(),
    "abs_delta_ge_1_count": (rg3_delta.abs() >= 1).sum(),
}).sort_values("mean_abs_delta", ascending=False, na_position="last")
rg3_summary.index.name = "feature"
print("Exploratory difference ranking (not model feature importance):")
display(rg3_summary.round(3))

# 이웃의 원본 ID와 거리를 함께 기록하여 비교 대상을 추적
rg3_matches = []
for i, pid in enumerate(rg3_risk.index):
    for rank, j in enumerate(rg3_order[i, :K_NEIGHBORS], 1):
        rg3_matches.append({"risk_pattern": pid,
            "risk_source_ids": json.dumps(rg3_risk.loc[pid, "source_ids"]),
            "neighbor_rank": rank, "normal_pattern": rg3_normal.index[j],
            "normal_source_ids": json.dumps(rg3_normal.iloc[j]["source_ids"]),
            "standardized_distance": rg3_distance[i, j]})
rg3_matches = pd.DataFrame(rg3_matches)
display(rg3_matches.head(10))


In [ ]:
# 열은 평균 절대 차이 순서, 행은 원본 패턴 등장 순서
rg3_cols = rg3_summary.index.tolist()
rg3_heat = rg3_delta[rg3_cols]
rg3_vmax = max(1.0, float(np.nanmax(np.abs(rg3_heat.to_numpy()))))
rg3_cmap = plt.get_cmap("RdBu_r").copy()
rg3_cmap.set_bad("#dddddd")
fig, ax = plt.subplots(figsize=(18, 11), constrained_layout=True)
im = ax.imshow(np.ma.masked_invalid(rg3_heat.to_numpy()), aspect="auto",
               cmap=rg3_cmap, vmin=-rg3_vmax, vmax=rg3_vmax)
ax.set_xticks(range(len(rg3_cols)), rg3_cols, rotation=65, ha="right", fontsize=9)
row_labels = [f"{pid} | IDs {','.join(map(str, rg3_risk.loc[pid, 'source_ids']))}"
              for pid in rg3_heat.index]
ax.set_yticks(range(len(row_labels)), row_labels, fontsize=8)
ax.set_title(f"RG3: defect-history patterns vs {K_NEIGHBORS} nearest normal-only patterns\n"
             "Red = higher; blue = lower; gray = undefined (constant normal feature)")
ax.set_xlabel("Features ranked by mean absolute standardized difference")
ax.set_ylabel("Defect-history pattern / original row IDs")
fig.colorbar(im, ax=ax, label="(Risk value - neighbor mean) / normal-pattern SD", shrink=.8)
fig.savefig(RG3_OUT / "neighbor_difference_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

# K를 바꾸었을 때 순위가 유지되는지 함께 확인
rg3_sensitivity = pd.DataFrame({f"K={k}": rg3_neighbor_difference(k).abs().mean()
                              for k in sorted({3, K_NEIGHBORS, 10}) if k <= len(rg3_normal)})
print("Sensitivity to neighbor count: mean absolute difference")
display(rg3_sensitivity.loc[rg3_cols].round(3))
rg3_delta.to_csv(RG3_OUT / "pattern_feature_differences.csv", encoding="utf-8-sig")
rg3_summary.to_csv(RG3_OUT / "feature_difference_summary.csv", encoding="utf-8-sig")
rg3_matches.to_csv(RG3_OUT / "neighbor_matches.csv", index=False, encoding="utf-8-sig")
rg3_sensitivity.to_csv(RG3_OUT / "neighbor_count_sensitivity.csv", encoding="utf-8-sig")
print(f"Saved to: {RG3_OUT}")


In [ ]:
plt.close("all")


# CN7 비라벨 컬럼

원본 분석: `EDA/cn7_unlabeled_columns.ipynb`. 모듈화 전 원본은 복구 ZIP에 보관되어 있습니다.


# CN7 비라벨 데이터: 컬럼별 행 번호 그래프

원본 `data/origin/moldset_unlabeled_cn7.csv`의 24개 수치 컬럼을 표시합니다. **원본 CSV 인덱스를 버리고 0부터 35,238까지의 연속 행 번호**를 x축으로 사용합니다. 행 번호를 생산 시각으로 해석하지 않습니다. 비라벨 파일이므로 정상·불량 구분은 없습니다.

`X_MIN`, `X_MAX`는 새 행 번호의 양 끝값(포함)입니다. 선택한 행만 그래프에 들어가고, 각 y축도 선택한 행의 실제 최솟값~최댓값을 사용합니다. `None`이면 해당 끝을 제한하지 않습니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/origin/moldset_unlabeled_cn7.csv').is_file())
CSV = ROOT / 'data/origin/moldset_unlabeled_cn7.csv'
OUT = ROOT / 'output/modular/eda/cn7_unlabeled_column_plots'
OUT.mkdir(parents=True, exist_ok=True)

# 0부터 시작하는 행 번호 범위. 예: X_MIN=1000, X_MAX=2000
X_MIN = None
X_MAX = None
PLOTS_PER_PAGE = 4

df = pd.read_csv(CSV, index_col=0).reset_index(drop=True)
if not df.index.is_unique or not df.index.is_monotonic_increasing:
    raise ValueError('행 번호가 고유하고 오름차순인지 확인해주세요.')
if X_MIN is not None and X_MAX is not None and X_MIN > X_MAX:
    raise ValueError('X_MIN은 X_MAX 이하여야 합니다.')

# 초기화된 행 번호에서 x 범위를 먼저 자른다. 그림과 y축 모두 이 데이터만 사용한다.
left = df.index.min() if X_MIN is None else X_MIN
right = df.index.max() if X_MAX is None else X_MAX
view = df.loc[(df.index >= left) & (df.index <= right)]
if view.empty:
    raise ValueError(f'인덱스 {left}~{right}에 행이 없습니다.')
columns = view.select_dtypes(include='number').columns.tolist()
if len(columns) != 24:
    raise ValueError(f'예상한 수치 컬럼 24개 대신 {len(columns)}개를 찾았습니다.')

print(f'원본: {len(df):,}행, 수치 컬럼 {len(columns)}개')
print(f'표시: {len(view):,}행, 행 번호 {view.index.min()}~{view.index.max()}')
limits = view[columns].replace([np.inf, -np.inf], np.nan).agg(['min', 'max']).T
limits.columns = ['y_min', 'y_max']
display(limits)

for start in range(0, len(columns), PLOTS_PER_PAGE):
    selected = columns[start:start + PLOTS_PER_PAGE]
    fig, axes = plt.subplots(len(selected), 1,
                             figsize=(16, 2.9 * len(selected)),
                             sharex=True, squeeze=False, constrained_layout=True)
    for ax, col in zip(axes.ravel(), selected):
        values = view[col].to_numpy(dtype=float)
        finite = np.isfinite(values)
        ax.scatter(view.index.to_numpy()[finite], values[finite],
                   s=2, alpha=.45, color='#3478B8', rasterized=True)
        good = values[finite]
        if good.size:
            lo, hi = float(good.min()), float(good.max())
            if lo < hi:
                ax.set_ylim(lo, hi)
            else:
                pad = max(abs(lo) * .01, .01)
                ax.set_ylim(lo-pad, hi+pad)
        ax.set_xlim(left-.5, right+.5) if left == right else ax.set_xlim(left, right)
        ax.set_title(col, loc='left', fontsize=11)
        ax.set_ylabel('Value')
        ax.grid(alpha=.2)
    axes[-1, 0].set_xlabel('Row number after index reset (not elapsed time)')
    fig.suptitle(f'CN7 unlabeled | row {left}–{right} | columns {start+1}–{start+len(selected)}')
    fig.savefig(OUT / f'columns_{start+1:02d}_{start+len(selected):02d}.png',
                dpi=130, bbox_inches='tight')
    plt.show()
    plt.close(fig)


In [ ]:
plt.close("all")


# RG3 비라벨 컬럼

원본 분석: `EDA/rg3_unlabeled_columns.ipynb`. 모듈화 전 원본은 복구 ZIP에 보관되어 있습니다.


# RG3 비라벨 데이터: 컬럼별 행 번호 그래프

원본 `data/origin/moldset_unlabeled_rg3.csv`의 24개 수치 컬럼을 표시합니다. **원본 CSV 인덱스를 버리고 0부터 35,940까지의 연속 행 번호**를 x축으로 사용합니다. 행 번호를 생산 시각으로 해석하지 않습니다. 비라벨 파일이므로 정상·불량 구분은 없습니다.

`X_MIN`, `X_MAX`는 새 행 번호의 양 끝값(포함)입니다. 선택한 행만 그래프에 들어가고, 각 y축도 선택한 행의 실제 최솟값~최댓값을 사용합니다. `None`이면 해당 끝을 제한하지 않습니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/origin/moldset_unlabeled_rg3.csv').is_file())
CSV = ROOT / 'data/origin/moldset_unlabeled_rg3.csv'
OUT = ROOT / 'output/modular/eda/rg3_unlabeled_column_plots'
OUT.mkdir(parents=True, exist_ok=True)

# 0부터 시작하는 행 번호 범위. 예: X_MIN=1000, X_MAX=2000
X_MIN = None
X_MAX = None
PLOTS_PER_PAGE = 4

df = pd.read_csv(CSV, index_col=0).reset_index(drop=True)
if not df.index.is_unique or not df.index.is_monotonic_increasing:
    raise ValueError('행 번호가 고유하고 오름차순인지 확인해주세요.')
if X_MIN is not None and X_MAX is not None and X_MIN > X_MAX:
    raise ValueError('X_MIN은 X_MAX 이하여야 합니다.')

# 초기화된 행 번호에서 x 범위를 먼저 자른다. 그림과 y축 모두 이 데이터만 사용한다.
left = df.index.min() if X_MIN is None else X_MIN
right = df.index.max() if X_MAX is None else X_MAX
view = df.loc[(df.index >= left) & (df.index <= right)]
if view.empty:
    raise ValueError(f'인덱스 {left}~{right}에 행이 없습니다.')
columns = view.select_dtypes(include='number').columns.tolist()
if len(columns) != 24:
    raise ValueError(f'예상한 수치 컬럼 24개 대신 {len(columns)}개를 찾았습니다.')

print(f'원본: {len(df):,}행, 수치 컬럼 {len(columns)}개')
print(f'표시: {len(view):,}행, 행 번호 {view.index.min()}~{view.index.max()}')
limits = view[columns].replace([np.inf, -np.inf], np.nan).agg(['min', 'max']).T
limits.columns = ['y_min', 'y_max']
display(limits)

for start in range(0, len(columns), PLOTS_PER_PAGE):
    selected = columns[start:start + PLOTS_PER_PAGE]
    fig, axes = plt.subplots(len(selected), 1,
                             figsize=(16, 2.9 * len(selected)),
                             sharex=True, squeeze=False, constrained_layout=True)
    for ax, col in zip(axes.ravel(), selected):
        values = view[col].to_numpy(dtype=float)
        finite = np.isfinite(values)
        ax.scatter(view.index.to_numpy()[finite], values[finite],
                   s=2, alpha=.45, color='#3478B8', rasterized=True)
        good = values[finite]
        if good.size:
            lo, hi = float(good.min()), float(good.max())
            if lo < hi:
                ax.set_ylim(lo, hi)
            else:
                pad = max(abs(lo) * .01, .01)
                ax.set_ylim(lo-pad, hi+pad)
        ax.set_xlim(left-.5, right+.5) if left == right else ax.set_xlim(left, right)
        ax.set_title(col, loc='left', fontsize=11)
        ax.set_ylabel('Value')
        ax.grid(alpha=.2)
    axes[-1, 0].set_xlabel('Row number after index reset (not elapsed time)')
    fig.suptitle(f'RG3 unlabeled | row {left}–{right} | columns {start+1}–{start+len(selected)}')
    fig.savefig(OUT / f'columns_{start+1:02d}_{start+len(selected):02d}.png',
                dpi=130, bbox_inches='tight')
    plt.show()
    plt.close(fig)


In [ ]:
plt.close("all")


## 모듈 경계

기존 EDA의 원본 행 유지·3-fold 전처리 저장 셀은 현재 패턴 단위 정책과 충돌하므로 제거했습니다.
`rg3_defect_factor_analysis.ipynb` 뒤에 붙어 있던 CN7 제목/RG3 경로 혼합 실험과 잘못된 figsize를 사용하는 임시 시각화는 복구 ZIP에만 보관합니다.
시안의 학습 입력은 `data/processed/{cn7|rg3}/conservative/`만 사용합니다.
